In [ ]:
# importing libraries

import pandas as pd
import geopandas as gpd
import requests
import folium
import json
import time
import os

In [ ]:
# configuration

ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjE2YzYyMzI0OWZlYzRiNzJiYjI2MDAzZmRlZmY4ZjRiIiwiaCI6Im11cm11cjY0In0="

# airport definitions

AIRPORTS = [
    {'name': 'Heathrow Airport',
     'iata': 'LHR',
     'lat': 51.4700,
     'lon': -0.4543,
     'colour': 'blue'},
    {'name': 'Gatwick Airport',
     'iata': 'LGW',
     'lat': 51.1537,
     'lon': -0.1821,
     'colour': 'green'},
    {'name': 'Luton Airport',
     'iata': 'LTN',
     'lat': 51.8747,
     'lon': -0.3683,
     'colour': 'red'}
]

# isochrone time bands
TIME_BANDS = [1800, 2700, 3600]
TIME_LABELS = ['30 min', '45 min', '60 min']

print(f' Airports: {[a["iata"] for a in AIRPORTS]}')
print(f' Time bands: {TIME_BANDS}')

airport_lookup = {a['iata']: a for a in AIRPORTS}
print(f' Airport lookup ready')

In [ ]:
from prompt_toolkit import key_binding
from IPython.core.shellapp import SYSTEM_CONFIG_DIRS
# testing API connection on Luton airport

def get_isochrones(lon, lat, time_bands, api_key):
    url = 'https://api.openrouteservice.org/v2/isochrones/driving-car'

    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json'
    }

    body = {
        'locations': [[lon, lat]],
        'range': time_bands,
        'range_type': 'time'
    }

    response = requests.post(url, json=body, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        print(f' API error {response.status_code}: {response.text}')
        return None

# Test with Luton

test_airport = AIRPORTS[2]
print(f' Testing connection with {test_airport["name"]}...')

test_result = get_isochrones(
    lon=test_airport['lon'],
    lat=test_airport['lat'],
    time_bands=TIME_BANDS,
    api_key=ORS_API_KEY
)

if test_result:
    print(f' Received {len(test_result["features"])} zones for {test_airport["name"]}')
else:
    print('Error - check API key')

Run test was good, now generating for all three airports

In [ ]:
# generating isocrhones for all airports

all_isochrones = {}

for airport in AIRPORTS:
    print(f'Generating isochrones for {airport["name"]} ({airport["iata"]})...')

    result = get_isochrones(
        lon=airport['lon'],
        lat=airport['lat'],
        time_bands=TIME_BANDS,
        api_key=ORS_API_KEY
    )

    if result:
        all_isochrones[airport['iata']] = result
        print(f' {airport["iata"]} - {len(result["features"])} received')
    else:
        print(f' {airport["iata"]} - failed')

    time.sleep(1)

print(f' {len(all_isochrones)}/{len(AIRPORTS)} all airports processed')

In [ ]:
all_isochrones = {}
for iata in ['LHR', 'LGW', 'LTN']:
    path = f'/content/drive/MyDrive/catchment_data/{iata}_isochrones.geojson'
    with open(path, 'r') as f:
        all_isochrones[iata] = json.load(f)
    print(f' Loaded {iata}')

In [ ]:
# map of all airports

map_centre = [51.5, -0.3]
map = folium.Map(location=map_centre, zoom_start=7, tiles='cartodbpositron')

opacities = [0.25, 0.15, 0.08]

airport_lookup = {a['iata']: a for a in AIRPORTS}

for iata, geojson in all_isochrones.items():
  airport = airport_lookup[iata]
  colour = airport['colour']
  name = airport['name']

  for i, feature in enumerate(geojson['features']):
    label = f"{airport['name']} - {TIME_LABELS[i]} drive"

    folium.GeoJson(
        feature,
        style_function= lambda x, c=colour, o=opacities[i]: {
            'fillColor': c,
            'fillOpacity': o,
            'color': c,
            'weight': 1.5
        },
        tooltip=label
    ).add_to(map)

#adding airport markers
for airport in AIRPORTS:
  if airport['iata'] in all_isochrones:
    folium.Marker(
        location=[airport['lat'], airport['lon']],
        tooltip=f"{airport['name']} ({airport['iata']})",
        icon=folium.Icon(color=airport['colour'], icon='plane', prefix='fa')
    ).add_to(map)

# adding a legend

legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: white; padding: 12px 16px;
            border-radius: 8px; border: 1px solid #ccc;
            font-family: Arial; font-size: 13px; line-height: 1.8;">
  <b>Airport Catchment Areas</b><br>
  <span style="color:blue">■</span> Heathrow (LHR)<br>
  <span style="color:red">■</span> Gatwick (LGW)<br>
  <span style="color:green">■</span> Luton (LTN)<br>
  <hr style="margin:4px 0">
  Dark fill = 60 min drive<br>
  Medium fill = 90 min drive<br>
  Light fill = 120 min drive
</div>
'''
map.get_root().html.add_child(folium.Element(legend_html))

print('Map built')
print('Map displayed below: ')

map

In [ ]:
# installing map library for boundary data

!pip install mapclassify

In [ ]:
# working on the ITL3 boundaries from the ONS geoportal

itl3 = gpd.read_file("/content/International_Territorial_Level_3_(January_2021)_Boundaries_UK_BFC_(V4)_.zip")

print(f" Loaded {len(itl3)} ITL3 regions")
print(f" Columns: {list(itl3.columns)}")
print(f" CRS: {itl3.crs}")

had to use the shapefile as the geojson was corrupted or was missing data

In [ ]:
# need to check the data first

print(itl3.head())
print(f' Coordinate system (CRS): {itl3.crs}')

if itl3.crs.to_epsg() !=4326:
  print('Converting CRS to WGS84 (EPSG:4326) ')
  itl3 = itl3.to_crs(epsg=4326)
  print('CRS converted')

else:
  print('CRS already WSG84, no conversion was needed')

In [ ]:
# spatial join - for each airport isocrhone i need to see under which ITL3 regions it will fall under

from shapely.geometry import shape

isochrone_rows = []

for iata, geojson in all_isochrones.items():
  airport = airport_lookup[iata]
  for i, feature in enumerate(geojson["features"]):
    isochrone_rows.append({
        "airport_name": airport["name"],
        "airport_iata": iata,
        "time_label": TIME_LABELS[i],
        "time_seconds": TIME_BANDS[i],
        "geometry": shape(feature["geometry"])
    })

isochrones_gdf = gpd.GeoDataFrame(isochrone_rows, crs="EPSG:4326")
print(f' Converted{len(isochrones_gdf)} isochrone features to a GeoDataFrame')

joined = gpd.sjoin(
    itl3,
    isochrones_gdf,
    how="inner",
    predicate="intersects"
)

print(f' Spatial join is complete')
print(f' Found {len(joined)} ITL3- isocrhone mathces \n')

# summary -> how many itl3 regions per airport and per time band

summary = (
    joined
    .groupby(["airport_iata", "time_label"])["ITL321NM"]
    .count()
    .reset_index()
    .rename(columns={"ITL321NM": "itl3_region_count"})
)

print('ITL3 regions within each catchment zone: ')
print(summary.to_string(index=False))

In [ ]:
# visualising itl3 regions on the map
# building a layered map

# Centre map over London
m_combined = folium.Map(
    location=[51.5, -0.3],
    zoom_start=7,
    tiles="CartoDB positron"
)

# layer 1:ITL3 regions within catchment

matched_itl3_codes = joined["ITL321CD"].unique()
itl3_in_catchment  = itl3[itl3["ITL321CD"].isin(matched_itl3_codes)]

folium.GeoJson(
    itl3_in_catchment.__geo_interface__,
    name="ITL3 Regions",
    style_function=lambda x: {
        "fillColor":   "#ffff00",
        "color":       "#666666",
        "fillOpacity": 0.15,
        "weight":      1
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ITL321CD", "ITL321NM"],
        aliases=["ITL3 Code:", "ITL3 Region:"]
    )
).add_to(m_combined)

# layer 2:isochrone catchment zones

opacities = [0.25, 0.15, 0.08]

for iata, geojson in all_isochrones.items():
    airport = airport_lookup[iata]
    colour  = airport["colour"]
    for i, feature in enumerate(geojson["features"]):
        folium.GeoJson(
            feature,
            name=f"{airport['iata']} — {TIME_LABELS[i]}",
            style_function=lambda x, c=colour, o=opacities[i]: {
                "fillColor":   c,
                "color":       c,
                "fillOpacity": o,
                "weight":      1.5
            },
            tooltip=f"{airport['name']} — {TIME_LABELS[i]} drive"
        ).add_to(m_combined)

# layer 3:airport markers

for airport in AIRPORTS:
    if airport["iata"] in all_isochrones:
        folium.Marker(
            location=[airport["lat"], airport["lon"]],
            tooltip=f"{airport['name']} ({airport['iata']})",
            icon=folium.Icon(color=airport["colour"], icon="plane", prefix="fa")
        ).add_to(m_combined)

# ;ayer control -> this is bassicaly a toggle to turn layers on and off
folium.LayerControl().add_to(m_combined)

# legend for the map

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: white; padding: 12px 16px;
            border-radius: 8px; border: 1px solid #ccc;
            font-family: Arial; font-size: 13px; line-height: 1.8;">
  <b>Airport Catchment Areas</b><br>
  <span style="color:blue">&#9632;</span> Heathrow (LHR)<br>
  <span style="color:green">&#9632;</span> Gatwick (LGW)<br>
  <span style="color:red">&#9632;</span> Luton (LTN)<br>
  <span style="color:#cccc00">&#9632;</span> ITL3 regions within catchment<br>
  <hr style="margin:4px 0">
  Dark fill = 30 min drive<br>
  Medium fill = 45 min drive<br>
  Light fill = 60 min drive
</div>
"""
m_combined.get_root().html.add_child(folium.Element(legend_html))

print(" Combined map built with 3 layers:")
print(" ITL3 region boundaries")
print(" Isochrone catchment zones")
print(" Airport markers")
print(" Layer toggle added (top right of map)")
m_combined

In [ ]:
# addidng bus data
# only kept what I thought i needed from the whole file

coach_df = pd.read_excel('/content/Transport.updated.xlsx')

print(f' Coach stops loaded: {len(coach_df)} stops')
print(f' Columns: {list(coach_df.columns)}')
print(coach_df.head())

In [ ]:
# going to have to filter it more

# filter to Southeast England only (around our three airports)
coach_df = coach_df[
    (coach_df['Latitude']  >= 51.0) &
    (coach_df['Latitude']  <= 53.5) &
    (coach_df['Longitude'] >= -2.5) &
    (coach_df['Longitude'] <= 0.5)
].copy()

# keep only BCT stops
coach_df = coach_df[coach_df['StopType'] == 'BCT'].copy()

# reset index
coach_df = coach_df.reset_index(drop=True)

print(f' Coach stops after filtering: {len(coach_df)} stops')
print(coach_df.head())

In [ ]:
# converting the stops to geodataframe

coach_gdf = gpd.GeoDataFrame(
    coach_df[['CommonName', 'Longitude', 'Latitude', 'StopType']],
    geometry = gpd.points_from_xy(coach_df['Longitude'], coach_df['Latitude']),
    crs='EPSG:4326'
)

print(f' Coach stops converted to GeoDataFrame: {len(coach_gdf)}')
print(f' CRS: {coach_gdf.crs}')

In [ ]:
# adding rail data - zip file

import zipfile

with zipfile.ZipFile('/content/uk-railway-stations-main.zip', 'r') as z:
    with z.open('uk-railway-stations-main/stations.csv') as f:
        rail_df = pd.read_csv(f)

rail_gdf = gpd.GeoDataFrame(
    rail_df,
    geometry=gpd.points_from_xy(rail_df['long'], rail_df['lat']),
    crs='EPSG:4326'
)

print(f' Rail stations loaded: {len(rail_gdf)} stations')
print(f' Columns: {list(rail_df.columns)}')

In [ ]:
# adding tube data
# British National Grid - EPSG:27700

tube_df = pd.read_csv('/content/Underground_Stations.csv')

tube_gdf = gpd.GeoDataFrame(
    tube_df,
    geometry=gpd.points_from_xy(tube_df['X'], tube_df['Y']),
    crs='EPSG:27700'
)

# convert to WGS84 to match everything else
tube_gdf = tube_gdf.to_crs(epsg=4326)

print(f' Tube stations loaded: {len(tube_gdf)} stations')
print(f' Networks: {tube_df["NETWORK"].unique()}')
print(f' CRS converted to WGS84')

In [ ]:
# spatial join again within the catchment zone

# rail network first

rail_in_catchment = gpd.sjoin(
    rail_gdf,
    isochrones_gdf,
    how='inner',
    predicate='within'
)

rail_summary = (
    rail_in_catchment
    .groupby(['airport_iata', 'time_label'])['stationName']
    .count()
    .reset_index()
    .rename(columns={'stationName': 'rail_count'})
)

print(f' Rail stations within each catchment zone: {len(rail_in_catchment)}')
print(rail_summary.to_string(index=False))

In [ ]:
# tube network

tube_in_catchment = gpd.sjoin(
    tube_gdf,
    isochrones_gdf,
    how='inner',
    predicate='within'
)

tube_summary = (
    tube_in_catchment
    .groupby(['airport_iata', 'time_label'])['NAME']
    .count()
    .reset_index()
    .rename(columns={'NAME': 'tube_count'})
)

print(f' Tube stations within each catchment zone: {len(tube_in_catchment)}')
print(tube_summary.to_string(index=False))

In [ ]:
# coach data

coach_in_catchment = gpd.sjoin(
    coach_gdf,
    isochrones_gdf,
    how='inner',
    predicate='within'
)

coach_summary = (
    coach_in_catchment
    .groupby(['airport_iata', 'time_label'])['CommonName']
    .count()
    .reset_index()
    .rename(columns={'CommonName': 'coach_count'})
)

print(f' Coach stations within each catchment zone: {len(coach_in_catchment)}')
print(coach_summary.to_string(index=False))

For rail ->
LHR has 471 rail stations within 60 mins
LGW has 376, LTN has 246

For tube ->
LHR has 234 tube stations within 60 mins
LGW has 29, LTN has 170

For coach ->
fairly equal across all three airports



In [ ]:
# population data

pop_df = pd.read_excel(
    '/content/mye24tablesew.xlsx',
    sheet_name = 'MYE2 - Persons',
    skiprows=7
)

pop_df = pop_df[['Code', 'Name', 'Geography', 'All ages']].copy()
pop_df.columns = ['area_code', 'area_name', 'geography_type', 'population_2024']

# need to filter to Local Authority level only

pop_la = pop_df[pop_df['geography_type'] == 'Metropolitan District'].copy()
pop_la = pd.concat([
    pop_df[pop_df['geography_type'] == 'Non-metropolitan District'],
    pop_df[pop_df['geography_type'] == 'London Borough'],
    pop_df[pop_df['geography_type'] == 'Unitary Authority'],
    pop_df[pop_df['geography_type'] == 'County'],
]).reset_index(drop=True)

print(f' Population data loaded: {len(pop_la)} local authorities')
print(f' Geography types: {pop_la["geography_type"].unique()}')
print(pop_la.head())


In [ ]:
# GDHI data - Gross Disposable Household Income

gdhi_df = pd.read_excel(
    '/content/regionalgrossdisposablehouseholdincomeallitlregions2023.xlsx',
    sheet_name = 'Table 1',
    skiprows=1
)

#renaming columns

gdhi_df.columns = ['ITL_level', 'ITL_code', 'Region_name'] + \
                  [str(c) for c in gdhi_df.columns[3:]]

gdhi_itl3 = gdhi_df[gdhi_df['ITL_level'] == 'ITL3'][
    ['ITL_code', 'Region_name', '2023']
].copy()

gdhi_itl3 = gdhi_itl3.rename(columns={'2023': 'GDHI_2023_million_gbp'})
gdhi_itl3 = gdhi_itl3.reset_index(drop=True)

print(f' GDHI data loaded: {len(gdhi_itl3)} ITL3 regions')
print(gdhi_itl3.head())

In [ ]:
# joining gdhi to itl3 boundaries

itl3_v2 = itl3.merge(
    gdhi_itl3[['ITL_code', 'GDHI_2023_million_gbp']],
    left_on = 'ITL321CD',
    right_on = 'ITL_code',
    how = 'left'
)

# checking for matches

matched = itl3_v2['GDHI_2023_million_gbp'].notnull().sum()
unmatched = itl3_v2['GDHI_2023_million_gbp'].isnull().sum()

print(f' GDHI joined to ITL3 boundaries')
print(f' Matched: {matched}')
print(f' Unmatched: {unmatched}')
print(itl3_v2[['ITL321CD', 'ITL321NM', 'GDHI_2023_million_gbp']].head())

In [ ]:
# a lot of unmatched
# checking to see what the issue is

print("ITL3 boundary codes (first 10):")
print(itl3['ITL321CD'].head(10).tolist())

print("\nGDHI codes (first 10):")
print(gdhi_itl3['ITL_code'].head(10).tolist())

In [ ]:
# gdhi files uses a different version of ITL3 codes.
# system might've gotten updated
# trying to match on region name instead

itl3_v2 = itl3.merge(
    gdhi_itl3[['Region_name', 'GDHI_2023_million_gbp']],
    left_on = 'ITL321NM',
    right_on = 'Region_name',
    how = 'left'
)

# checking for matches

matched = itl3_v2['GDHI_2023_million_gbp'].notnull().sum()
unmatched = itl3_v2['GDHI_2023_million_gbp'].isnull().sum()

print(f' GDHI joined on region name')
print(f' Matched: {matched}')
print(f' Unmatched: {unmatched}')


# need to see the unmatched ones in case i need to investigate again

if unmatched > 0:
  print(f' \n Unmatched regions: ')
  print(itl3_v2[itl3_v2['GDHI_2023_million_gbp'].isna()]['ITL321NM'].tolist())

update : 157 vs 122 from before.

checked and the issue is in the difference in names between the two files, i.e: north yorkshire cc and north yorshire in the other.

fixing some that i need

In [ ]:
# manually fixing names

name_fixes = {
    'North Yorkshire CC':           'North Yorkshire',
    'Barnsley, Doncaster and Rotherham': 'Barnsley, Doncaster and Rotherham',
    'Suffolk CC':                   'Suffolk',
    'Hertfordshire CC':             'Hertfordshire',
    'Camden and City of London':    'Camden & City of London',
    'Westminster':                  'Westminster',
    'Berkshire':                    'Berkshire',
    'Somerset CC':                  'Somerset',
    'Bath and North East Somerset, North Somerset and South Gloucestershire':
        'Bath and North East Somerset, North Somerset and South Gloucestershire',
}

# apply fixes to ITL3 boundary names
itl3_fixed = itl3.copy()
itl3_fixed['ITL321NM_fixed'] = itl3_fixed['ITL321NM'].replace(name_fixes)

# retry the merge with fixed names
itl3_enriched = itl3_fixed.merge(
    gdhi_itl3[['Region_name', 'GDHI_2023_million_gbp']],
    left_on='ITL321NM_fixed',
    right_on='Region_name',
    how='left'
)

matched   = itl3_enriched['GDHI_2023_million_gbp'].notna().sum()
unmatched = itl3_enriched['GDHI_2023_million_gbp'].isna().sum()

print(f' Matched: {matched} regions')
print(f' Unmatched: {unmatched} regions')

20 unmatched regions - checking to see if any are near our airpots


In [ ]:
unmatched_regions = itl3_v2[
    itl3_v2['GDHI_2023_million_gbp'].isna()
]['ITL321NM'].tolist()

#checking overlap

catchment_regions = joined['ITL321NM'].unique().tolist()

problem_regions = [r for r in unmatched_regions if r in catchment_regions]

if len(problem_regions) == 0:
  print(' None of the unmatched regions are within our catchment zones')
  print(' Good to go. GDHI data is ok for our ares')
else:
  print(f' The unmatched regions are within the catchemnt zones: ')
  print(problem_regions)

these four are important so i need to see how to fix them

In [ ]:
# finding the exat anmes in the gdhi file

search_terms = ['Hertfordshire', 'Camden', 'Westminster', 'Berkshire']

for term in search_terms:
  matches = gdhi_itl3[gdhi_itl3['Region_name'].str.contains(term, case=False)]
  print(f' {term}: ')
  print(matches[['ITL_code', 'Region_name']].to_string(index=False))
  print()

so the zones are split into sub-regions. great

In [ ]:
split_regions = {
    'Hertfordshire CC': ['North and East Hertfordshire', 'South West Hertfordshire'],
    'Camden and City of London': ['Camden', 'Westminster and City of London'],
    'Westminster': ['Westminster and City of London'],
    'Berkshire': ['Berkshire East', 'Berkshire West'],
}

fix_rows = []
for boundary_name, gdhi_names in split_regions.items():
    total_gdhi = gdhi_itl3[
        gdhi_itl3['Region_name'].isin(gdhi_names)
    ]['GDHI_2023_million_gbp'].sum()
    fix_rows.append({
        'Region_name': boundary_name,
        'GDHI_2023_million_gbp': total_gdhi
    })

gdhi_fixes = pd.DataFrame(fix_rows)
print(' GDHI fixes calculated: ')
print(gdhi_fixes.to_string(index=False))

for _, fix_row in gdhi_fixes.iterrows():
    mask = itl3_v2['ITL321NM'] == fix_row['Region_name']
    itl3_v2.loc[mask, 'GDHI_2023_million_gbp'] = fix_row['GDHI_2023_million_gbp']

# checking again for to see the match count
matched   = itl3_v2['GDHI_2023_million_gbp'].notna().sum()
unmatched = itl3_v2['GDHI_2023_million_gbp'].isna().sum()

print(f' \n Final matched: {matched} regions')
print(f' Remaining unmatched: {unmatched} regions')

checking again for the 4 zones, the other 18 are in wales or scotland which we dont need


In [ ]:
unmatched_names = itl3_v2[
    itl3_v2['GDHI_2023_million_gbp'].isna()
]['ITL321NM'].tolist()

catchment_regions = joined['ITL321NM'].unique().tolist()
problem_regions = [r for r in unmatched_names if r in catchment_regions]

if len(problem_regions) == 0:
    print(' All catchment regions have GDHI data')
    print(f'  Remaining 18 unmatched are outside catchment zones')
    print(f'\n  Unmatched regions (for reference): ')
    for r in unmatched_names:
        print(f'    - {r}')
else:
    print(f' Still missing GDHI for these catchment regions: ')
    print(problem_regions)

In [ ]:
# joining the population data
# checking to see what geography types are in the file

print(f' All geography types in the pop data: ')
print(pop_df['geography_type'].value_counts().to_string())

this is at local authority level. need to find another link for the ITL3


In [ ]:
# found the data

lookup_df = pd.read_csv(
    '/content/LAD_(December_2024)_to_LAU1_to_ITL3_to_ITL2_to_ITL1_(January_2025)_Lookup_in_the_UK.csv',
)

print(f' Lookup table loaded: {len(lookup_df)} rows')
print(f' Columns: {list(lookup_df.columns)}')

# joining population to lookup table using Local Authority code
pop_itl3 = lookup_df.merge(
    pop_df[['area_code', 'population_2024']],
    left_on='LAD24CD',
    right_on='area_code',
    how='left'
)

# aggregate population up to ITL3 level
pop_itl3_agg = (
    pop_itl3
    .groupby(['ITL325CD', 'ITL325NM'])['population_2024']
    .sum()
    .reset_index()
    .rename(columns={'ITL325NM': 'Region_name'})
)

print(f'\n Population aggregated to ITL3: {len(pop_itl3_agg)} regions')
print(pop_itl3_agg.head())

same issue here as before. need to check the unmathced regions

In [ ]:
# Clean up itl3_v2 columns completely
# Drop all population columns and start fresh
cols_to_drop = [col for col in itl3_v2.columns if 'population' in col]
print(f'Dropping columns: {cols_to_drop}')
itl3_v2 = itl3_v2.drop(columns=cols_to_drop)

# Re-merge population cleanly
itl3_v2 = itl3_v2.merge(
    pop_itl3_agg[['ITL325CD', 'population_2024']],
    left_on='ITL321CD',
    right_on='ITL325CD',
    how='left'
)

print(f'✓ Columns now: {list(itl3_v2.columns)}')
print(f'  Matched: {itl3_v2["population_2024"].notna().sum()} regions')

In [ ]:
unmatched_names = itl3_v2[
    itl3_v2['population_2024'].isna()
]['ITL321NM'].tolist()

catchment_regions = joined['ITL321NM'].unique().tolist()
problem_regions = [r for r in unmatched_names if r in catchment_regions]

if len(problem_regions) == 0:
    print(' All catchment regions have population data — safe to proceed')
else:
    print(f' Missing population data for these catchment regions:')
    for r in problem_regions:
        print(f'  - {r}')

In [ ]:
pop_fixes = {
    'Hertfordshire CC': ['North Hertfordshire', 'East Hertfordshire',
                         'Welwyn Hatfield', 'Stevenage', 'St Albans',
                         'Watford', 'Three Rivers', 'Hertsmere',
                         'Broxbourne', 'Dacorum'],
    'Camden and City of London': ['Camden', 'City of London'],
    'Westminster': ['Westminster'],
    'Berkshire': ['Reading', 'Slough', 'Windsor and Maidenhead',
                  'Bracknell Forest', 'Wokingham', 'West Berkshire'],
}

for boundary_name, la_names in pop_fixes.items():
    total_pop = pop_df[
        pop_df['area_name'].isin(la_names)
    ]['population_2024'].sum()

    mask = itl3_v2['ITL321NM'] == boundary_name
    itl3_v2.loc[mask, 'population_2024'] = total_pop
    print(f' {boundary_name}: {total_pop:,.0f} people')

# final check
matched   = itl3_v2['population_2024'].notna().sum()
unmatched = itl3_v2['population_2024'].isna().sum()
print(f'\n Final matched: {matched} regions')
print(f' Remaining unmatched: {unmatched} regions')

In [ ]:
unmatched_names = itl3_v2[
    itl3_v2['population_2024'].isna()
]['ITL321NM'].tolist()

catchment_regions = joined['ITL321NM'].unique().tolist()
problem_regions = [r for r in unmatched_names if r in catchment_regions]

if len(problem_regions) == 0:
    print(' All catchment regions have population data')
    print(f'\n  Summary of itl3_v2:')
    print(f' Total regions: {len(itl3_v2)}')
    print(f' Regions with population: {itl3_v2["population_2024"].notna().sum()}')
    print(f' Regions with GDHI: {itl3_v2["GDHI_2023_million_gbp"].notna().sum()}')
    print(f'\n  Columns: {list(itl3_v2.columns)}')
else:
    print(f' Still missing population for these catchment regions:')
    for r in problem_regions:
        print(f'  - {r}')

summary: itl3_v2 has - itl3 boundaries(geometry), region names and codes, GDHI wealth data(2023) and pop data (2024)

the final map will have:
Isochrones - 3 airports, 3 time bands
pop and GDHI
rail stations
tube sations
coach stops



In [ ]:
print(coach_in_catchment.columns.tolist())

In [ ]:
# Reduce coach stops to a manageable number
# Keep 1 in every 10 stops
coach_reduced = coach_in_catchment.groupby(
    ['airport_iata', 'time_label']
).apply(lambda x: x.sample(frac=0.1, random_state=42)).reset_index(drop=True)

print(f' Coach stops reduced from {len(coach_in_catchment)} to {len(coach_reduced)}')
print(coach_reduced.groupby(['airport_iata', 'time_label'])['CommonName'].count())

In [ ]:
# another map

from folium.plugins import MarkerCluster

m_final = folium.Map(
    location=[51.5, -0.3],
    zoom_start=7,
    tiles='CartoDB positron'
)

# LAYER 1: Population
# Only show ITL3 regions within catchment zones
catchment_codes = joined['ITL321CD'].unique()
itl3_catchment_only = itl3_v2[itl3_v2['ITL321CD'].isin(catchment_codes)]

folium.Choropleth(
    geo_data=itl3_catchment_only.__geo_interface__,
    data=itl3_catchment_only,
    columns=['ITL321NM', 'population_2024'],
    key_on='feature.properties.ITL321NM',
    fill_color='YlOrRd',
    fill_opacity=0.5,
    line_opacity=0.2,
    legend_name='Population (2024)',
    name='Population by ITL3 region',
    nan_fill_color='lightgrey'
).add_to(m_final)

# LAYER 2: Isochrones
colours = {'LHR': 'blue', 'LGW': 'red', 'LTN': 'green'}

for iata, geojson in all_isochrones.items():
    colour = colours[iata]
    for i, feature in enumerate(geojson['features']):
        folium.GeoJson(
            feature,
            name=f"{iata} — {TIME_LABELS[i]}",
            style_function=lambda x, c=colour: {
                'fillColor':   c,
                'color':       c,
                'fillOpacity': 0,
                'weight':      2.5
            },
            tooltip=f"{iata} — {TIME_LABELS[i]} drive"
        ).add_to(m_final)

# LAYER 3: Rail stations (dark blue)
rail_cluster = MarkerCluster(name='Rail Stations').add_to(m_final)
for _, row in rail_in_catchment.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='darkblue',
        fill=True,
        fill_color='darkblue',
        fill_opacity=0.8,
        tooltip=f" {row['stationName']}"
    ).add_to(rail_cluster)

# LAYER 4: Tube stations (dark red)
tube_cluster = MarkerCluster(name=' Tube Stations').add_to(m_final)
for _, row in tube_in_catchment.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='darkred',
        fill=True,
        fill_color='darkred',
        fill_opacity=0.8,
        tooltip=f" {row['NAME']}"
    ).add_to(tube_cluster)

# LAYER 5: Coach stops (dark green, reduced)
coach_cluster = MarkerCluster(name=' Coach Stops').add_to(m_final)
for _, row in coach_reduced.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='darkgreen',
        fill=True,
        fill_color='darkgreen',
        fill_opacity=0.6,
        tooltip=f" {row['CommonName']}"
    ).add_to(coach_cluster)

# LAYER 6: Airport markers
for airport in AIRPORTS:
    if airport['iata'] in all_isochrones:
        folium.Marker(
            location=[airport['lat'], airport['lon']],
            tooltip=f"✈ {airport['name']} ({airport['iata']})",
            icon=folium.Icon(
                color=airport['colour'],
                icon='plane',
                prefix='fa'
            )
        ).add_to(m_final)

# LAYER CONTROL
folium.LayerControl(collapsed=False).add_to(m_final)

# LEGEND
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: white; padding: 12px 16px;
            border-radius: 8px; border: 1px solid #ccc;
            font-family: Arial; font-size: 12px; line-height: 1.9;">
  <b>Airport Catchment Analysis</b><br>
  ✈ <span style="color:blue">Heathrow (LHR)</span><br>
  ✈ <span style="color:red">Gatwick (LGW)</span><br>
  ✈ <span style="color:green">Luton (LTN)</span><br>
  <hr style="margin:4px 0">
   Dark blue = Rail stations<br>
   Dark red = Tube stations<br>
   Dark green = Coach stops<br>
  <hr style="margin:4px 0">
  Yellow→Red shading = Population<br>
  Coloured outlines = 30/45/60 min drive<br>
  <hr style="margin:4px 0">
  <i>Use layer toggle (top right)<br>to show/hide layers</i>
</div>
"""
m_final.get_root().html.add_child(folium.Element(legend_html))
m_final

In [ ]:
# Deduplicate all transport layers
rail_dedup  = rail_in_catchment.drop_duplicates(subset=['stationName']).copy()
tube_dedup  = tube_in_catchment.drop_duplicates(subset=['NAME']).copy()
coach_dedup = coach_reduced.drop_duplicates(subset=['CommonName']).copy()

print(f'Rail:  {len(rail_in_catchment):,} → {len(rail_dedup):,}')
print(f'Tube:  {len(tube_in_catchment):,} → {len(tube_dedup):,}')
print(f'Coach: {len(coach_reduced):,} → {len(coach_dedup):,}')

In [ ]:
# FINAL MAP — with deduplicated transport layers
from folium.plugins import MarkerCluster

m_final = folium.Map(
    location=[51.5, -0.3],
    zoom_start=7,
    tiles='CartoDB positron'
)

# LAYER 1: Population
# Only show ITL3 regions within catchment zones
catchment_codes = joined['ITL321CD'].unique()
itl3_catchment_only = itl3_v2[itl3_v2['ITL321CD'].isin(catchment_codes)]

folium.Choropleth(
    geo_data=itl3_catchment_only.__geo_interface__,
    data=itl3_catchment_only,
    columns=['ITL321NM', 'population_2024'],
    key_on='feature.properties.ITL321NM',
    fill_color='YlOrRd',
    fill_opacity=0.5,
    line_opacity=0.2,
    legend_name='Population (2024)',
    name='Population by ITL3 region',
    nan_fill_color='lightgrey'
).add_to(m_final)

# LAYER 2: Isochrones
colours = {'LHR': 'blue', 'LGW': 'red', 'LTN': 'green'}

for iata, geojson in all_isochrones.items():
    colour = colours[iata]
    for i, feature in enumerate(geojson['features']):
        folium.GeoJson(
            feature,
            name=f"{iata} — {TIME_LABELS[i]}",
            style_function=lambda x, c=colour: {
                'fillColor':   c,
                'color':       c,
                'fillOpacity': 0,
                'weight':      2.5
            },
            tooltip=f"{iata} — {TIME_LABELS[i]} drive"
        ).add_to(m_final)

# LAYER 3: Rail stations (blue cluster)
fg_rail = folium.FeatureGroup(name=' Rail Stations', show=False)
cluster_rail = MarkerCluster(
    icon_create_function="""
    function(cluster) {
        return L.divIcon({
            html: '<div style="background-color:#00008B; color:white; border-radius:50%; width:30px; height:30px; display:flex; align-items:center; justify-content:center; font-weight:bold; font-size:11px;">' + cluster.getChildCount() + '</div>',
            className: '',
            iconSize: [30, 30]
        });
    }"""
).add_to(fg_rail)

for _, row in rail_dedup.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='darkblue',
        fill=True,
        fill_color='darkblue',
        fill_opacity=0.8,
        tooltip=f" {row['stationName']}"
    ).add_to(cluster_rail)
fg_rail.add_to(m_final)

# LAYER 4: Tube stations (red cluster)
fg_tube = folium.FeatureGroup(name='Tube Stations', show=False)
cluster_tube = MarkerCluster(
    icon_create_function="""
    function(cluster) {
        return L.divIcon({
            html: '<div style="background-color:#8B0000; color:white; border-radius:50%; width:30px; height:30px; display:flex; align-items:center; justify-content:center; font-weight:bold; font-size:11px;">' + cluster.getChildCount() + '</div>',
            className: '',
            iconSize: [30, 30]
        });
    }"""
).add_to(fg_tube)

for _, row in tube_dedup.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='darkred',
        fill=True,
        fill_color='darkred',
        fill_opacity=0.8,
        tooltip=f" {row['NAME']}"
    ).add_to(cluster_tube)
fg_tube.add_to(m_final)

# LAYER 5: Coach stops (green cluster)
fg_coach = folium.FeatureGroup(name=' Coach Stops', show=False)
cluster_coach = MarkerCluster(
    icon_create_function="""
    function(cluster) {
        return L.divIcon({
            html: '<div style="background-color:#006400; color:white; border-radius:50%; width:30px; height:30px; display:flex; align-items:center; justify-content:center; font-weight:bold; font-size:11px;">' + cluster.getChildCount() + '</div>',
            className: '',
            iconSize: [30, 30]
        });
    }"""
).add_to(fg_coach)

for _, row in coach_dedup.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='darkgreen',
        fill=True,
        fill_color='darkgreen',
        fill_opacity=0.6,
        tooltip=f" {row['CommonName']}"
    ).add_to(cluster_coach)
fg_coach.add_to(m_final)

# LAYER 6: Airport markers
for airport in AIRPORTS:
    if airport['iata'] in all_isochrones:
        folium.Marker(
            location=[airport['lat'], airport['lon']],
            tooltip=f"✈ {airport['name']} ({airport['iata']})",
            icon=folium.Icon(
                color=airport['colour'],
                icon='plane',
                prefix='fa'
            )
        ).add_to(m_final)

# LAYER CONTROL
folium.LayerControl(collapsed=False).add_to(m_final)

# LEGEND
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: white; padding: 12px 16px;
            border-radius: 8px; border: 1px solid #ccc;
            font-family: Arial; font-size: 12px; line-height: 1.9;">
  <b>Airport Catchment Analysis</b><br>
  ✈ <span style="color:blue">Heathrow (LHR)</span><br>
  ✈ <span style="color:red">Gatwick (LGW)</span><br>
  ✈ <span style="color:green">Luton (LTN)</span><br>
  <hr style="margin:4px 0">
   Dark blue = Rail stations<br>
   Dark red = Tube stations<br>
   Dark green = Coach stops<br>
  <hr style="margin:4px 0">
  Yellow→Red shading = Population<br>
  Coloured outlines = 30/45/60 min drive<br>
  <hr style="margin:4px 0">
  <i>Use layer toggle (top right)<br>to show/hide layers</i>
</div>
"""
m_final.get_root().html.add_child(folium.Element(legend_html))
m_final

In [ ]:
ORS_API_KEY = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjE2YzYyMzI0OWZlYzRiNzJiYjI2MDAzZmRlZmY4ZjRiIiwiaCI6Im11cm11cjY0In0='

# Test with a single point
url = 'https://api.openrouteservice.org/v2/directions/driving-car/geojson'

body = {
    'coordinates': [
        [0.1218, 52.2053],   # Cambridge town centre
        [-0.4614, 51.4775]   # Heathrow
    ]
}

response = requests.post(url, json=body, headers={
    'Authorization': ORS_API_KEY,
    'Content-Type': 'application/json'
})

print(f'Status: {response.status_code}')
if response.status_code == 200:
    data = response.json()
    mins = round(data['features'][0]['properties']['summary']['duration'] / 60)
    km   = round(data['features'][0]['properties']['summary']['distance'] / 1000, 1)
    print(f' API working — Cambridge to Heathrow: {mins} min · {km} km')
else:
    print(response.text[:300])

tested one random point within the api to see if i can actually do it. it finally worked.

In [ ]:
from shapely.geometry import Point
ORS_API_KEY = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjE2YzYyMzI0OWZlYzRiNzJiYjI2MDAzZmRlZmY4ZjRiIiwiaCI6Im11cm11cjY0In0='

# Airport road-accessible coordinates (slightly adjusted from terminal)
AIRPORT_COORDS = {
    'LHR': {'name': 'London Heathrow', 'lat': 51.4706, 'lon': -0.4541, 'colour': 'blue'},
    'LGW': {'name': 'London Gatwick',  'lat': 51.1537, 'lon': -0.1821, 'colour': 'red'},
    'LTN': {'name': 'London Luton',    'lat': 51.8796, 'lon': -0.3713, 'colour': 'green'},
}

# ── Set your start point here ────────────────────────────────
# Change these coordinates to any point within the catchment
START_LAT = 51.7520   # Oxford
START_LON = -1.2577

print(f'Start point: {START_LAT}, {START_LON}')
print(f'Airports: {list(AIRPORT_COORDS.keys())}')
print(' Configuration ready')

In [ ]:
# ── Find nearest transport stops to start point

start_point = Point(START_LON, START_LAT)
start_gdf   = gpd.GeoDataFrame(geometry=[start_point], crs='EPSG:4326')

def find_nearest_stop(stops_gdf, start_point, name_col):
    """Find the single nearest stop to a given point."""
    distances = stops_gdf.geometry.distance(start_point)
    nearest_idx = distances.idxmin()
    nearest_row = stops_gdf.loc[nearest_idx]
    dist_km = distances[nearest_idx] * 111  # rough degrees to km
    return nearest_row, round(dist_km, 2)

# Find nearest of each type
nearest_rail,  dist_rail  = find_nearest_stop(rail_dedup,  start_point, 'stationName')
nearest_tube,  dist_tube  = find_nearest_stop(tube_dedup,  start_point, 'NAME')
nearest_coach, dist_coach = find_nearest_stop(coach_dedup, start_point, 'CommonName')

print(f' Nearest rail station : {nearest_rail["stationName"]} ({dist_rail} km)')
print(f' Nearest tube station : {nearest_tube["NAME"]} ({dist_tube} km)')
print(f' Nearest coach stop   : {nearest_coach["CommonName"]} ({dist_coach} km)')

In [ ]:
import requests
import time

# Fix CRS warning by projecting to BNG first

def find_nearest_stop_projected(stops_gdf, start_lat, start_lon, name_col):
    """Find nearest stop using projected CRS for accurate distances."""
    start_point_bng = gpd.GeoDataFrame(
        geometry=[Point(start_lon, start_lat)], crs='EPSG:4326'
    ).to_crs('EPSG:27700').geometry.iloc[0]

    stops_projected = stops_gdf.to_crs('EPSG:27700')
    distances = stops_projected.geometry.distance(start_point_bng)
    nearest_idx = distances.idxmin()
    nearest_row = stops_gdf.loc[nearest_idx]
    dist_km = round(distances[nearest_idx] / 1000, 2)
    return nearest_row, dist_km

# Recalculate with correct projection
nearest_rail,  dist_rail  = find_nearest_stop_projected(rail_dedup,  START_LAT, START_LON, 'stationName')
nearest_tube,  dist_tube  = find_nearest_stop_projected(tube_dedup,  START_LAT, START_LON, 'NAME')
nearest_coach, dist_coach = find_nearest_stop_projected(coach_dedup, START_LAT, START_LON, 'CommonName')

print(f'✓ Nearest rail station : {nearest_rail["stationName"]} ({dist_rail} km)')
print(f'✓ Nearest tube station : {nearest_tube["NAME"]} ({dist_tube} km)')
print(f'✓ Nearest coach stop   : {nearest_coach["CommonName"]} ({dist_coach} km)')

# Fetch driving routes via ORS
def get_driving_route(from_lon, from_lat, to_lon, to_lat, api_key):
    """Get driving route between two points via ORS."""
    url = 'https://api.openrouteservice.org/v2/directions/driving-car/geojson'
    body = {'coordinates': [[from_lon, from_lat], [to_lon, to_lat]]}
    response = requests.post(url, json=body, headers={
        'Authorization': api_key,
        'Content-Type': 'application/json'
    })
    if response.status_code == 200:
        data = response.json()
        summary = data['features'][0]['properties']['summary']
        coords  = data['features'][0]['geometry']['coordinates']
        return {
            'duration_min': round(summary['duration'] / 60),
            'distance_km':  round(summary['distance'] / 1000, 1),
            'coords':       [[c[1], c[0]] for c in coords]  # flip to lat/lon for folium
        }
    else:
        print(f'  ✗ Route error: {response.status_code}')
        return None

# Fetch driving routes to all airports

print('\nFetching driving routes...')
drive_routes = {}
for iata, ap in AIRPORT_COORDS.items():
    print(f'  {iata}...', end=' ')
    route = get_driving_route(START_LON, START_LAT, ap['lon'], ap['lat'], ORS_API_KEY)
    if route:
        drive_routes[iata] = route
        print(f'{route["duration_min"]} min · {route["distance_km"]} km')
    time.sleep(0.5)

# Fetch driving routes from start to nearest stops

print('\nFetching routes to nearest stops...')
stop_routes = {}

for stop_name, stop_row, stop_type in [
    (nearest_rail['stationName'], nearest_rail, 'rail'),
    (nearest_tube['NAME'],        nearest_tube,  'tube'),
    (nearest_coach['CommonName'], nearest_coach, 'coach'),
]:
    print(f'  To {stop_name}...', end=' ')
    route = get_driving_route(
        START_LON, START_LAT,
        stop_row.geometry.x, stop_row.geometry.y,
        ORS_API_KEY
    )
    if route:
        stop_routes[stop_type] = {
            'name':   stop_name,
            'route':  route,
            'coords': [stop_row.geometry.y, stop_row.geometry.x]
        }
        print(f'{route["duration_min"]} min · {route["distance_km"]} km')
    time.sleep(0.5)

print('\n All routes fetched')

gatwick seems to be in a restriced area hence the 404 error. trying to work around it

In [ ]:
# Fix Gatwick coordinates, use the road outside the terminal

AIRPORT_COORDS['LGW']['lat'] = 51.1564
AIRPORT_COORDS['LGW']['lon'] = -0.1622

# Retry LGW only

print('Retrying LGW...')
route = get_driving_route(START_LON, START_LAT,
                          AIRPORT_COORDS['LGW']['lon'],
                          AIRPORT_COORDS['LGW']['lat'],
                          ORS_API_KEY)
if route:
    drive_routes['LGW'] = route
    print(f' LGW: {route["duration_min"]} min · {route["distance_km"]} km')

In [ ]:
os.makedirs('/content/drive/MyDrive/catchment_data', exist_ok=True)

# Save isochrones to Drive
for iata, geojson in all_isochrones.items():
    path = f'/content/drive/MyDrive/catchment_data/{iata}_isochrones.geojson'
    with open(path, 'w') as f:
        json.dump(geojson, f)
    print(f' Saved {iata}')

In [ ]:
all_isochrones = {}
for iata in ['LHR', 'LGW', 'LTN']:
    path = f'/content/drive/MyDrive/catchment_data/{iata}_isochrones.geojson'
    with open(path, 'r') as f:
        all_isochrones[iata] = json.load(f)
    print(f' Loaded {iata}')

In [ ]:
from shapely.ops import unary_union
# Build union of all 60 min isochrones

all_polys = []
for iata in ['LHR', 'LGW', 'LTN']:
    features = all_isochrones[iata]['features']
    # Get the 60 min zone (last feature — largest zone)
    geom = shape(features[-1]['geometry'])
    all_polys.append(geom)

combined = unary_union(all_polys)
combined_gdf = gpd.GeoDataFrame(geometry=[combined], crs='EPSG:4326')

# Filter ITL3 by centroid inside catchment
itl3_projected     = itl3_v2.to_crs('EPSG:27700').copy()
combined_projected = combined_gdf.to_crs('EPSG:27700').geometry.iloc[0]
itl3_projected['inside'] = itl3_projected.geometry.centroid.within(combined_projected)
itl3_catchment_filtered  = itl3_v2[itl3_projected['inside'].values].copy()

print(f' itl3_catchment_filtered: {len(itl3_catchment_filtered)} regions')

In [ ]:
import json

# Export all data for HTML

# Isochrones
iso_geojson = {}
for iata, geojson in all_isochrones.items():
    iso_geojson[iata] = geojson

# Transport stops
rail_json  = json.loads(rail_dedup[['stationName', 'geometry']].to_json())
tube_json  = json.loads(tube_dedup[['NAME', 'LINES', 'geometry']].to_json())
coach_json = json.loads(coach_dedup[['CommonName', 'geometry']].head(2000).to_json())

# ITL3 catchment
itl3_json = json.loads(
    itl3_catchment_filtered[['ITL321NM', 'population_2024', 'GDHI_2023_million_gbp', 'geometry']].to_json()
)

print(f' Rail stops:   {len(rail_dedup)}')
print(f' Tube stops:   {len(tube_dedup)}')
print(f' Coach stops:  {min(2000, len(coach_dedup))}')
print(f' ITL3 regions: {len(itl3_catchment_filtered)}')
print(' All data ready for HTML export')

all ready to see if it is gonna be an interactive map or not

In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Interactive Journey Planner</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family: Arial, sans-serif; height:100vh; display:flex; flex-direction:column; }}
  #header {{ background:#1a1a2e; color:white; padding:10px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:14px; font-weight:bold; letter-spacing:0.05em; }}
  #api-bar {{ background:#0d1525; padding:6px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; }}
  #sidebar {{ width:340px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:12px 16px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:10px; }}
  .airport-block {{ margin-bottom:10px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:8px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; }}
  .dot {{ width:8px; height:8px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:7px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .route-left {{ display:flex; align-items:center; gap:8px; }}
  .route-icon {{ width:20px; height:20px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:11px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; }}
  .spinner {{ width:11px; height:11px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:30px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:5px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:8px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:8px; }}
  .leg {{ display:flex; align-items:center; gap:4px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:16px; height:3px; border-radius:1px; }}
  #loading-bar {{ padding:6px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
  .leaflet-popup-content-wrapper {{ background:#1e2d45; color:#e2e8f0; font-family:Arial; font-size:11px; border-radius:5px; }}
  .leaflet-popup-tip {{ background:#1e2d45; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>ORS API KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your OpenRouteService API key…"/>
</div>

<div id="main">
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">
      Click anywhere on the map to calculate routes to all three airports.<br><br>
      Transport routes show the <b>full journey</b>: start → nearest stop → airport.
      Click any row to highlight that route.
    </div>
    <div id="results">
      <div id="empty">Click anywhere on the map<br>to calculate journey times.</div>
    </div>
    <div id="loading-bar">⏳ Loading routes... please wait (~20s)</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Rail</div>
      <div class="leg"><div class="leg-line" style="background:#ff00ff"></div>Tube</div>
      <div class="leg"><div class="leg-line" style="background:#00ff88"></div>Coach</div>
      <div class="leg"><div class="leg-line" style="background:#ff00ff;opacity:0.4;border-top:2px dashed #ff00ff"></div>→ stop</div>
      <div class="leg"><div class="leg-line" style="background:#ff00ff;opacity:1"></div>stop → ✈</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<script>
const ISO_DATA   = {json.dumps(iso_geojson)};
const RAIL_DATA  = {json.dumps(rail_json)};
const TUBE_DATA  = {json.dumps(tube_json)};
const COACH_DATA = {json.dumps(coach_json)};
const ITL3_DATA  = {json.dumps(itl3_json)};

const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lon:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lon:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lon:-0.3713, colour:'#22c55e' }},
}};

// Distinct bright colours that contrast with the catchment zone outlines
const ROUTE_COLOURS = {{
  drive: '#ff8800',   // bright orange
  rail:  '#00ccff',   // bright cyan
  tube:  '#ff00ff',   // bright magenta
  coach: '#00ff88',   // bright lime
}};

const isoColours = {{ LHR:'#3b82f6', LGW:'#ef4444', LTN:'#22c55e' }};
const TIMES = [30, 45, 60];

// ── Map ───────────────────────────────────────────────────────
const map = L.map('map').setView([51.5, -0.3], 8);
L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  attribution:'© OpenStreetMap © CARTO', maxZoom:18
}}).addTo(map);

// ── ITL3 choropleth ───────────────────────────────────────────
const pops = ITL3_DATA.features.map(f => f.properties.population_2024 || 0);
const minPop = Math.min(...pops), maxPop = Math.max(...pops);
function popColour(pop) {{
  if (!pop) return '#374151';
  const t = (pop - minPop) / (maxPop - minPop);
  const r = Math.round(255 * Math.min(1, t * 2));
  const g = Math.round(255 * Math.max(0, 1 - t * 2));
  return `rgb(${{r}},${{g}},50)`;
}}
L.geoJSON(ITL3_DATA, {{
  style: f => ({{ fillColor: popColour(f.properties.population_2024), color:'#374151', weight:0.5, fillOpacity:0.5 }}),
  onEachFeature: (f, layer) => {{
    const p = f.properties;
    layer.bindTooltip(`<b>${{p.ITL321NM}}</b><br>Population: ${{(p.population_2024||0).toLocaleString()}}<br>GDHI: £${{(p.GDHI_2023_million_gbp||0).toLocaleString()}}M`, {{sticky:true}});
  }}
}}).addTo(map);

// ── Isochrones ────────────────────────────────────────────────
Object.entries(ISO_DATA).forEach(([iata, geojson]) => {{
  geojson.features.forEach((feature, i) => {{
    L.geoJSON(feature, {{ style: {{ color:isoColours[iata], weight:[2.5,1.8,1.2][i], fillOpacity:0, opacity:0.9 }} }})
      .bindTooltip(`${{iata}} — ${{TIMES[i]}} min drive`).addTo(map);
  }});
}});

// ── Airport markers ───────────────────────────────────────────
Object.entries(AIRPORTS).forEach(([code, ap]) => {{
  L.marker([ap.lat, ap.lon], {{
    icon: L.divIcon({{
      html:`<div style="background:${{ap.colour}};color:white;border-radius:50%;width:32px;height:32px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border:2px solid rgba(255,255,255,0.4);box-shadow:0 0 10px ${{ap.colour}}88;">✈${{code}}</div>`,
      className:'', iconSize:[32,32], iconAnchor:[16,16]
    }})
  }}).addTo(map).bindPopup(`<b>✈ ${{ap.name}}</b><br>${{code}}`);
}});

// ── Transport layers ──────────────────────────────────────────
function makeLayer(data, colour, nameField, emoji) {{
  const layer = L.layerGroup();
  data.features.forEach(f => {{
    const [lon, lat] = f.geometry.coordinates;
    L.circleMarker([lat, lon], {{ radius:4, color:colour, fillColor:colour, fillOpacity:0.8, weight:1 }})
      .bindTooltip(`${{emoji}} ${{f.properties[nameField]}}`).addTo(layer);
  }});
  return layer;
}}
L.control.layers({{}}, {{
  '🚂 Rail Stations': makeLayer(RAIL_DATA,  '#00ccff', 'stationName', '🚂'),
  '🚇 Tube Stations': makeLayer(TUBE_DATA,  '#ff00ff', 'NAME',        '🚇'),
  '🚌 Coach Stops':   makeLayer(COACH_DATA, '#00ff88', 'CommonName',  '🚌'),
}}, {{collapsed:false}}).addTo(map);

// ── Helpers ───────────────────────────────────────────────────
const delay = ms => new Promise(r => setTimeout(r, ms));

function haversine(lat1, lon1, lat2, lon2) {{
  const R = 6371;
  const dLat = (lat2-lat1)*Math.PI/180, dLon = (lon2-lon1)*Math.PI/180;
  const a = Math.sin(dLat/2)**2 + Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLon/2)**2;
  return R * 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
}}

function findNearest(data, lat, lon) {{
  let best = null, bestDist = Infinity;
  data.features.forEach(f => {{
    const [flon, flat] = f.geometry.coordinates;
    const d = haversine(lat, lon, flat, flon);
    if (d < bestDist) {{ bestDist = d; best = f; }}
  }});
  return {{ feature: best, dist: bestDist.toFixed(2) }};
}}

async function getRoute(apiKey, fromLon, fromLat, toLon, toLat) {{
  try {{
    const res = await fetch('https://api.openrouteservice.org/v2/directions/driving-car/geojson', {{
      method:'POST',
      headers:{{'Authorization':apiKey,'Content-Type':'application/json'}},
      body: JSON.stringify({{coordinates:[[fromLon,fromLat],[toLon,toLat]]}})
    }});
    if (!res.ok) return null;
    const data = await res.json();
    const s = data.features[0].properties.summary;
    return {{
      mins: Math.round(s.duration/60),
      km:   (s.distance/1000).toFixed(1),
      coords: data.features[0].geometry.coordinates.map(c => [c[1],c[0]])
    }};
  }} catch(e) {{ return null; }}
}}

// ── Route state ───────────────────────────────────────────────
let routeLayers = [];
let startMarker = null;

function clearRoutes() {{
  routeLayers.forEach(l => map.removeLayer(l));
  routeLayers = [];
}}

function updateRow(id, leg1, leg2, colour, totalMins, totalKm) {{
  const el = document.getElementById(id);
  if (!el) return;
  const spinner = el.querySelector('.spinner');

  if (!leg1 && !leg2) {{
    if (spinner) spinner.outerHTML = '<div class="route-val" style="color:#ef4444">unavailable</div>';
    return;
  }}

  // Total time = leg1 + leg2
  const mins = totalMins;
  const km   = totalKm;
  if (spinner) spinner.outerHTML = `<div class="route-val done">${{mins}} min · ${{km}} km</div>`;

  // Draw leg1 (start → stop) as dashed
  const lines = [];
  if (leg1) {{
    const l1 = L.polyline(leg1.coords, {{
      color:colour, weight:3, opacity:0.9, dashArray:'8 5'
    }}).addTo(map);
    routeLayers.push(l1);
    lines.push(l1);
  }}

  // Draw leg2 (stop → airport) as solid
  if (leg2) {{
    const l2 = L.polyline(leg2.coords, {{
      color:colour, weight:4, opacity:0.9
    }}).addTo(map);
    routeLayers.push(l2);
    lines.push(l2);
  }}

  // Click to highlight
  el.onclick = () => {{
    routeLayers.forEach(l => l.setStyle({{opacity:0.15, weight:2}}));
    lines.forEach(l => l.setStyle({{opacity:1, weight:5}}));
    if (lines.length > 0) {{
      const allCoords = lines.flatMap(l => l.getLatLngs());
      map.fitBounds(L.latLngBounds(allCoords), {{padding:[40,40]}});
    }}
  }};
}}

// ── Main click handler ────────────────────────────────────────
map.on('click', async (e) => {{
  const {{ lat, lng }} = e.latlng;
  const apiKey = document.getElementById('apiKey').value.trim();

  document.getElementById('coords').textContent = `Start: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;

  if (startMarker) map.removeLayer(startMarker);
  startMarker = L.circleMarker([lat, lng], {{
    radius:9, color:'#ff8800', fillColor:'#ff8800', fillOpacity:1, weight:2
  }}).addTo(map).bindPopup(`<b> Start point</b><br>${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`);

  clearRoutes();

  if (!apiKey) {{
    document.getElementById('results').innerHTML = '<div id="empty">Please enter your ORS API key above.</div>';
    return;
  }}

  const nearest = {{
    rail:  findNearest(RAIL_DATA,  lat, lng),
    tube:  findNearest(TUBE_DATA,  lat, lng),
    coach: findNearest(COACH_DATA, lat, lng),
  }};

  // Render loading skeleton
  const html = Object.entries(AIRPORTS).map(([code, ap]) => `
    <div class="airport-block">
      <div class="airport-header">
        <div class="dot" style="background:${{ap.colour}}"></div>
        ${{ap.name}} (${{code}})
      </div>
      <div class="route-row" id="r-${{code}}-drive">
        <div class="route-left">
          <div class="route-icon" style="background:#ff880033">🚗</div>
          <div class="route-label">Drive direct</div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-rail">
        <div class="route-left">
          <div class="route-icon" style="background:#00ccff33">🚂</div>
          <div class="route-label">Rail<div class="route-sub">via ${{nearest.rail.feature.properties.stationName}} (${{nearest.rail.dist}} km)</div></div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-tube">
        <div class="route-left">
          <div class="route-icon" style="background:#ff00ff33">🚇</div>
          <div class="route-label">Tube<div class="route-sub">via ${{nearest.tube.feature.properties.NAME}} (${{nearest.tube.dist}} km)</div></div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-coach">
        <div class="route-left">
          <div class="route-icon" style="background:#00ff8833">🚌</div>
          <div class="route-label">Coach<div class="route-sub">via ${{nearest.coach.feature.properties.CommonName}} (${{nearest.coach.dist}} km)</div></div>
        </div><div class="spinner"></div>
      </div>
    </div>
  `).join('');
  document.getElementById('results').innerHTML = html;

  const lb = document.getElementById('loading-bar');
  lb.style.display = 'block';
  lb.style.color = '#f59e0b';
  lb.textContent = ' Loading routes... please wait (~20s)';

  // Fetch routes sequentially
  for (const [code, ap] of Object.entries(AIRPORTS)) {{
    const rn = nearest.rail.feature.geometry.coordinates;
    const tn = nearest.tube.feature.geometry.coordinates;
    const cn = nearest.coach.feature.geometry.coordinates;

    // Drive direct (single leg)
    const driveR = await getRoute(apiKey, lng, lat, ap.lon, ap.lat);
    if (driveR) updateRow(`r-${{code}}-drive`, driveR, null, ROUTE_COLOURS.drive, driveR.mins, driveR.km);
    else updateRow(`r-${{code}}-drive`, null, null, ROUTE_COLOURS.drive, 0, 0);
    await delay(1600);

    // Rail: leg1 = start→station, leg2 = station→airport
    const railL1 = await getRoute(apiKey, lng, lat, rn[0], rn[1]);
    await delay(1600);
    const railL2 = await getRoute(apiKey, rn[0], rn[1], ap.lon, ap.lat);
    const railMins = (railL1?.mins||0) + (railL2?.mins||0);
    const railKm   = ((parseFloat(railL1?.km)||0) + (parseFloat(railL2?.km)||0)).toFixed(1);
    updateRow(`r-${{code}}-rail`, railL1, railL2, ROUTE_COLOURS.rail, railMins, railKm);
    await delay(1600);

    // Tube: leg1 = start→tube station, leg2 = tube station→airport
    const tubeL1 = await getRoute(apiKey, lng, lat, tn[0], tn[1]);
    await delay(1600);
    const tubeL2 = await getRoute(apiKey, tn[0], tn[1], ap.lon, ap.lat);
    const tubeMins = (tubeL1?.mins||0) + (tubeL2?.mins||0);
    const tubeKm   = ((parseFloat(tubeL1?.km)||0) + (parseFloat(tubeL2?.km)||0)).toFixed(1);
    updateRow(`r-${{code}}-tube`, tubeL1, tubeL2, ROUTE_COLOURS.tube, tubeMins, tubeKm);
    await delay(1600);

    // Coach: leg1 = start→coach stop, leg2 = coach stop→airport
    const coachL1 = await getRoute(apiKey, lng, lat, cn[0], cn[1]);
    await delay(1600);
    const coachL2 = await getRoute(apiKey, cn[0], cn[1], ap.lon, ap.lat);
    const coachMins = (coachL1?.mins||0) + (coachL2?.mins||0);
    const coachKm   = ((parseFloat(coachL1?.km)||0) + (parseFloat(coachL2?.km)||0)).toFixed(1);
    updateRow(`r-${{code}}-coach`, coachL1, coachL2, ROUTE_COLOURS.coach, coachMins, coachKm);
    await delay(1600);
  }}

  lb.textContent = '✓ All routes loaded — click a row to highlight';
  lb.style.color = '#22c55e';
}});
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_v4.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print(' Saved to Google Drive as airport_journey_planner_v4.html')
print('  Download and open in Chrome/Edge')
print('  Routes now show FULL journey: start → stop → airport')
print('  Dashed line = to nearest stop, Solid line = stop → airport')

In [ ]:
coach_df_wide = pd.read_excel('/content/Transport.updated.xlsx')

# Wider filter — extend north to 54.0 to cover full catchment
coach_df_wide = coach_df_wide[
    (coach_df_wide['Latitude']  >= 51.0) &
    (coach_df_wide['Latitude']  <= 54.0) &  # extended from 53.5
    (coach_df_wide['Longitude'] >= -3.0) &  # extended west slightly
    (coach_df_wide['Longitude'] <= 0.5)  &
    (coach_df_wide['StopType']  == 'BCT')
].copy()

# Convert to GeoDataFrame
coach_gdf_wide = gpd.GeoDataFrame(
    coach_df_wide,
    geometry=gpd.points_from_xy(coach_df_wide['Longitude'], coach_df_wide['Latitude']),
    crs='EPSG:4326'
)

# Spatial join to catchment only
from shapely.ops import unary_union
from shapely.geometry import shape

all_polys = []
for iata in ['LHR', 'LGW', 'LTN']:
    features = all_isochrones[iata]['features']
    geom = shape(features[-1]['geometry'])
    all_polys.append(geom)

combined = unary_union(all_polys)
combined_gdf = gpd.GeoDataFrame(geometry=[combined], crs='EPSG:4326')

coach_in_catchment_wide = gpd.sjoin(
    coach_gdf_wide,
    combined_gdf,
    how='inner',
    predicate='within'
).drop(columns=['index_right'], errors='ignore')

# Deduplicate
coach_dedup_wide = coach_in_catchment_wide.drop_duplicates(subset=['CommonName']).copy()
coach_dedup_wide = coach_dedup_wide.reset_index(drop=True)

# Check by band
coach_dedup_wide['lat_band'] = pd.cut(
    coach_dedup_wide.geometry.y,
    bins=[51.0, 51.5, 52.0, 52.5, 53.0, 53.5, 54.0],
    labels=['51-51.5', '51.5-52', '52-52.5', '52.5-53', '53-53.5', '53.5-54']
)

print(f' Total coach stops: {len(coach_dedup_wide)}')
print('\nCoach stops by latitude band:')
print(coach_dedup_wide['lat_band'].value_counts().sort_index())

In [ ]:
# Sample to keep map performant but with better coverage
# Keep all stops north of 52.0 (sparse area) + 10% sample south of 52.0
north = coach_dedup_wide[coach_dedup_wide.geometry.y >= 52.0].copy()
south = coach_dedup_wide[coach_dedup_wide.geometry.y <  52.0].sample(frac=0.15, random_state=42).copy()

coach_final = pd.concat([north, south]).reset_index(drop=True)

print(f'North of 52.0: {len(north)} stops (all kept)')
print(f'South of 52.0: {len(south)} stops (15% sample)')
print(f'Total: {len(coach_final)} stops')

In [ ]:
# Update coach_json with the better coverage dataset
coach_json_wide = json.loads(
    coach_final[['CommonName', 'geometry']].to_json()
)

print(f' Coach JSON ready: {len(coach_final)} stops')
print('  Regenerating HTML')

had to add more coach stops, because before there was none further up St. Albans

In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Interactive Journey Planner</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family: Arial, sans-serif; height:100vh; display:flex; flex-direction:column; }}
  #header {{ background:#1a1a2e; color:white; padding:10px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:14px; font-weight:bold; letter-spacing:0.05em; }}
  #api-bar {{ background:#0d1525; padding:6px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; }}
  #sidebar {{ width:340px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:12px 16px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:10px; }}
  .airport-block {{ margin-bottom:10px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:8px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; }}
  .dot {{ width:8px; height:8px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:7px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .route-left {{ display:flex; align-items:center; gap:8px; }}
  .route-icon {{ width:20px; height:20px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:11px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; }}
  .spinner {{ width:11px; height:11px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:30px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:5px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:8px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:8px; }}
  .leg {{ display:flex; align-items:center; gap:4px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:16px; height:3px; border-radius:1px; }}
  #loading-bar {{ padding:6px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
  .leaflet-popup-content-wrapper {{ background:#1e2d45; color:#e2e8f0; font-family:Arial; font-size:11px; border-radius:5px; }}
  .leaflet-popup-tip {{ background:#1e2d45; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>ORS API KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your OpenRouteService API key…"/>
</div>

<div id="main">
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">
      Click anywhere on the map to calculate routes to all three airports.<br><br>
      Transport routes show the <b>full journey</b>: start → nearest stop → airport.
      Click any row to highlight that route.
    </div>
    <div id="results">
      <div id="empty">Click anywhere on the map<br>to calculate journey times.</div>
    </div>
    <div id="loading-bar">⏳ Loading routes... please wait (~20s)</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Rail</div>
      <div class="leg"><div class="leg-line" style="background:#ff00ff"></div>Tube</div>
      <div class="leg"><div class="leg-line" style="background:#00ff88"></div>Coach</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<script>
const ISO_DATA   = {json.dumps(iso_geojson)};
const RAIL_DATA  = {json.dumps(rail_json)};
const TUBE_DATA  = {json.dumps(tube_json)};
const COACH_DATA = {json.dumps(coach_json_wide)};
const ITL3_DATA  = {json.dumps(itl3_json)};

const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lon:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lon:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lon:-0.3713, colour:'#22c55e' }},
}};

const ROUTE_COLOURS = {{
  drive: '#ff8800',
  rail:  '#00ccff',
  tube:  '#ff00ff',
  coach: '#00ff88',
}};

const isoColours = {{ LHR:'#3b82f6', LGW:'#ef4444', LTN:'#22c55e' }};
const TIMES = [30, 45, 60];

// ── Map ───────────────────────────────────────────────────────
const map = L.map('map').setView([51.5, -0.3], 8);
L.tileLayer('https://{{s}}.basemaps.cartocdn.com/dark_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  attribution:'© OpenStreetMap © CARTO', maxZoom:18
}}).addTo(map);

// ── ITL3 choropleth ───────────────────────────────────────────
const pops = ITL3_DATA.features.map(f => f.properties.population_2024 || 0);
const minPop = Math.min(...pops), maxPop = Math.max(...pops);
function popColour(pop) {{
  if (!pop) return '#374151';
  const t = (pop - minPop) / (maxPop - minPop);
  const r = Math.round(255 * Math.min(1, t * 2));
  const g = Math.round(255 * Math.max(0, 1 - t * 2));
  return `rgb(${{r}},${{g}},50)`;
}}
L.geoJSON(ITL3_DATA, {{
  style: f => ({{ fillColor: popColour(f.properties.population_2024), color:'#374151', weight:0.5, fillOpacity:0.5 }}),
  onEachFeature: (f, layer) => {{
    const p = f.properties;
    layer.bindTooltip(
      `<b>${{p.ITL321NM}}</b><br>Population: ${{(p.population_2024||0).toLocaleString()}}<br>GDHI: £${{(p.GDHI_2023_million_gbp||0).toLocaleString()}}M`,
      {{sticky:true}}
    );
  }}
}}).addTo(map);

// ── Isochrones ────────────────────────────────────────────────
Object.entries(ISO_DATA).forEach(([iata, geojson]) => {{
  geojson.features.forEach((feature, i) => {{
    L.geoJSON(feature, {{ style: {{ color:isoColours[iata], weight:[2.5,1.8,1.2][i], fillOpacity:0, opacity:0.9 }} }})
      .bindTooltip(`${{iata}} — ${{TIMES[i]}} min drive`).addTo(map);
  }});
}});

// ── Airport markers ───────────────────────────────────────────
Object.entries(AIRPORTS).forEach(([code, ap]) => {{
  L.marker([ap.lat, ap.lon], {{
    icon: L.divIcon({{
      html:`<div style="background:${{ap.colour}};color:white;border-radius:50%;width:32px;height:32px;
            display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;
            border:2px solid rgba(255,255,255,0.4);box-shadow:0 0 10px ${{ap.colour}}88;">✈${{code}}</div>`,
      className:'', iconSize:[32,32], iconAnchor:[16,16]
    }})
  }}).addTo(map).bindPopup(`<b>✈ ${{ap.name}}</b><br>${{code}}`);
}});

// ── Transport layers ──────────────────────────────────────────
function makeLayer(data, colour, nameField, emoji) {{
  const layer = L.layerGroup();
  data.features.forEach(f => {{
    const [lon, lat] = f.geometry.coordinates;
    L.circleMarker([lat, lon], {{
      radius:4, color:colour, fillColor:colour, fillOpacity:0.8, weight:1
    }}).bindTooltip(`${{emoji}} ${{f.properties[nameField]}}`).addTo(layer);
  }});
  return layer;
}}
L.control.layers({{}}, {{
  '🚂 Rail Stations': makeLayer(RAIL_DATA,  '#00ccff', 'stationName', '🚂'),
  '🚇 Tube Stations': makeLayer(TUBE_DATA,  '#ff00ff', 'NAME',        '🚇'),
  '🚌 Coach Stops':   makeLayer(COACH_DATA, '#00ff88', 'CommonName',  '🚌'),
}}, {{collapsed:false}}).addTo(map);

// ── Helpers ───────────────────────────────────────────────────
const delay = ms => new Promise(r => setTimeout(r, ms));

function haversine(lat1, lon1, lat2, lon2) {{
  const R = 6371;
  const dLat = (lat2-lat1)*Math.PI/180, dLon = (lon2-lon1)*Math.PI/180;
  const a = Math.sin(dLat/2)**2 +
    Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLon/2)**2;
  return R * 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
}}

function findNearest(data, lat, lon) {{
  let best = null, bestDist = Infinity;
  data.features.forEach(f => {{
    const [flon, flat] = f.geometry.coordinates;
    const d = haversine(lat, lon, flat, flon);
    if (d < bestDist) {{ bestDist = d; best = f; }}
  }});
  return {{ feature: best, dist: bestDist.toFixed(2) }};
}}

async function getRoute(apiKey, fromLon, fromLat, toLon, toLat) {{
  try {{
    const res = await fetch(
      'https://api.openrouteservice.org/v2/directions/driving-car/geojson',
      {{
        method:'POST',
        headers:{{'Authorization':apiKey,'Content-Type':'application/json'}},
        body: JSON.stringify({{coordinates:[[fromLon,fromLat],[toLon,toLat]]}})
      }}
    );
    if (!res.ok) return null;
    const data = await res.json();
    const s = data.features[0].properties.summary;
    return {{
      mins: Math.round(s.duration/60),
      km:   (s.distance/1000).toFixed(1),
      coords: data.features[0].geometry.coordinates.map(c => [c[1],c[0]])
    }};
  }} catch(e) {{ return null; }}
}}

// ── Route state ───────────────────────────────────────────────
let routeLayers = [];
let startMarker = null;

function clearRoutes() {{
  routeLayers.forEach(l => map.removeLayer(l));
  routeLayers = [];
}}

function updateRow(id, leg1, leg2, colour, totalMins, totalKm) {{
  const el = document.getElementById(id);
  if (!el) return;
  const spinner = el.querySelector('.spinner');
  if (!leg1 && !leg2) {{
    if (spinner) spinner.outerHTML = '<div class="route-val" style="color:#ef4444">unavailable</div>';
    return;
  }}
  if (spinner) spinner.outerHTML = `<div class="route-val done">${{totalMins}} min · ${{totalKm}} km</div>`;
  const lines = [];
  if (leg1) {{
    const l1 = L.polyline(leg1.coords, {{ color:colour, weight:3, opacity:0.9, dashArray:'8 5' }}).addTo(map);
    routeLayers.push(l1);
    lines.push(l1);
  }}
  if (leg2) {{
    const l2 = L.polyline(leg2.coords, {{ color:colour, weight:4, opacity:0.9 }}).addTo(map);
    routeLayers.push(l2);
    lines.push(l2);
  }}
  el.onclick = () => {{
    routeLayers.forEach(l => l.setStyle({{opacity:0.15, weight:2}}));
    lines.forEach(l => l.setStyle({{opacity:1, weight:5}}));
    if (lines.length > 0) {{
      const allCoords = lines.flatMap(l => l.getLatLngs());
      map.fitBounds(L.latLngBounds(allCoords), {{padding:[40,40]}});
    }}
  }};
}}

// ── Click handler ─────────────────────────────────────────────
map.on('click', async (e) => {{
  const {{ lat, lng }} = e.latlng;
  const apiKey = document.getElementById('apiKey').value.trim();

  document.getElementById('coords').textContent =
    `Start: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;

  if (startMarker) map.removeLayer(startMarker);
  startMarker = L.circleMarker([lat, lng], {{
    radius:9, color:'#ff8800', fillColor:'#ff8800', fillOpacity:1, weight:2
  }}).addTo(map).bindPopup(`<b>📍 Start point</b><br>${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`);

  clearRoutes();

  if (!apiKey) {{
    document.getElementById('results').innerHTML =
      '<div id="empty">Please enter your ORS API key above.</div>';
    return;
  }}

  const nearest = {{
    rail:  findNearest(RAIL_DATA,  lat, lng),
    tube:  findNearest(TUBE_DATA,  lat, lng),
    coach: findNearest(COACH_DATA, lat, lng),
  }};

  const html = Object.entries(AIRPORTS).map(([code, ap]) => `
    <div class="airport-block">
      <div class="airport-header">
        <div class="dot" style="background:${{ap.colour}}"></div>
        ${{ap.name}} (${{code}})
      </div>
      <div class="route-row" id="r-${{code}}-drive">
        <div class="route-left">
          <div class="route-icon" style="background:#ff880033">🚗</div>
          <div class="route-label">Drive direct</div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-rail">
        <div class="route-left">
          <div class="route-icon" style="background:#00ccff33">🚂</div>
          <div class="route-label">Rail
            <div class="route-sub">via ${{nearest.rail.feature.properties.stationName}} (${{nearest.rail.dist}} km)</div>
          </div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-tube">
        <div class="route-left">
          <div class="route-icon" style="background:#ff00ff33">🚇</div>
          <div class="route-label">Tube
            <div class="route-sub">via ${{nearest.tube.feature.properties.NAME}} (${{nearest.tube.dist}} km)</div>
          </div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-coach">
        <div class="route-left">
          <div class="route-icon" style="background:#00ff8833">🚌</div>
          <div class="route-label">Coach
            <div class="route-sub">via ${{nearest.coach.feature.properties.CommonName}} (${{nearest.coach.dist}} km)</div>
          </div>
        </div><div class="spinner"></div>
      </div>
    </div>
  `).join('');
  document.getElementById('results').innerHTML = html;

  const lb = document.getElementById('loading-bar');
  lb.style.display = 'block';
  lb.style.color = '#f59e0b';
  lb.textContent = ' Loading routes... please wait (~20s)';

  for (const [code, ap] of Object.entries(AIRPORTS)) {{
    const rn = nearest.rail.feature.geometry.coordinates;
    const tn = nearest.tube.feature.geometry.coordinates;
    const cn = nearest.coach.feature.geometry.coordinates;

    const driveR = await getRoute(apiKey, lng, lat, ap.lon, ap.lat);
    updateRow(`r-${{code}}-drive`, driveR, null, ROUTE_COLOURS.drive, driveR?.mins||0, driveR?.km||0);
    await delay(1600);

    const railL1 = await getRoute(apiKey, lng, lat, rn[0], rn[1]);
    await delay(1600);
    const railL2 = await getRoute(apiKey, rn[0], rn[1], ap.lon, ap.lat);
    updateRow(`r-${{code}}-rail`, railL1, railL2, ROUTE_COLOURS.rail,
      (railL1?.mins||0)+(railL2?.mins||0),
      ((parseFloat(railL1?.km)||0)+(parseFloat(railL2?.km)||0)).toFixed(1));
    await delay(1600);

    const tubeL1 = await getRoute(apiKey, lng, lat, tn[0], tn[1]);
    await delay(1600);
    const tubeL2 = await getRoute(apiKey, tn[0], tn[1], ap.lon, ap.lat);
    updateRow(`r-${{code}}-tube`, tubeL1, tubeL2, ROUTE_COLOURS.tube,
      (tubeL1?.mins||0)+(tubeL2?.mins||0),
      ((parseFloat(tubeL1?.km)||0)+(parseFloat(tubeL2?.km)||0)).toFixed(1));
    await delay(1600);

    const coachL1 = await getRoute(apiKey, lng, lat, cn[0], cn[1]);
    await delay(1600);
    const coachL2 = await getRoute(apiKey, cn[0], cn[1], ap.lon, ap.lat);
    updateRow(`r-${{code}}-coach`, coachL1, coachL2, ROUTE_COLOURS.coach,
      (coachL1?.mins||0)+(coachL2?.mins||0),
      ((parseFloat(coachL1?.km)||0)+(parseFloat(coachL2?.km)||0)).toFixed(1));
    await delay(1600);
  }}

  lb.textContent = '✓ All routes loaded — click a row to highlight';
  lb.style.color = '#22c55e';
}});
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_v5.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print(' Saved: airport_journey_planner_v5.html')
print('  Download from Drive and open in Chrome/Edge')

working on the google street interactive map

In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; }}
  #header {{ background:#1a1a2e; color:white; padding:10px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:14px; font-weight:bold; }}
  #api-bar {{ background:#0d1525; padding:6px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; white-space:nowrap; }}
  #api-bar button:hover {{ background:#2563eb; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; background:#0a0e1a; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}
  #sidebar {{ width:340px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:12px 16px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:10px; }}
  .airport-block {{ margin-bottom:10px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:8px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; }}
  .dot {{ width:8px; height:8px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:7px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .route-left {{ display:flex; align-items:center; gap:8px; }}
  .route-icon {{ width:20px; height:20px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:11px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:11px; height:11px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:30px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:5px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:8px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:8px; }}
  .leg {{ display:flex; align-items:center; gap:4px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:16px; height:3px; border-radius:1px; }}
  #loading-bar {{ padding:6px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key here…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">
      Enter your API key and click Load Map.<br><br>
      Then click anywhere on the map to calculate real journey times to all three airports via Drive and Transit (real train/tube/bus routes).
    </div>
    <div id="results">
      <div id="empty">Load the map first then<br>click anywhere to begin.</div>
    </div>
    <div id="loading-bar">⏳ Calculating routes...</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e' }},
}};

const ROUTE_COLOURS = {{
  DRIVING: '#ff8800',
  TRANSIT: '#00ccff',
}};

const ISO_DATA  = {json.dumps(iso_geojson)};
const ITL3_DATA = {json.dumps(itl3_json)};

let map, directionsService;
let startMarker = null;
let renderers   = [];
let mapLoaded   = false;

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your Google Maps API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display = 'none';
  document.getElementById('map').style.display = 'block';
  const script = document.createElement('script');
  script.src = `https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async = true;
  script.defer = true;
  document.head.appendChild(script);
  mapLoaded = true;
}}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center: {{ lat:51.5, lng:-0.3 }},
    zoom: 8,
    mapTypeId: 'roadmap',
    styles: [
      {{ elementType:'geometry', stylers:[{{color:'#1a1a2e'}}] }},
      {{ elementType:'labels.text.fill', stylers:[{{color:'#94a3b8'}}] }},
      {{ elementType:'labels.text.stroke', stylers:[{{color:'#1a1a2e'}}] }},
      {{ featureType:'road', elementType:'geometry', stylers:[{{color:'#2d3748'}}] }},
      {{ featureType:'road.highway', elementType:'geometry', stylers:[{{color:'#2d4a7a'}}] }},
      {{ featureType:'water', elementType:'geometry', stylers:[{{color:'#0d1525'}}] }},
      {{ featureType:'transit.line', elementType:'geometry', stylers:[{{color:'#4a5568'}}] }},
      {{ featureType:'transit.station', elementType:'geometry', stylers:[{{color:'#374151'}}] }},
      {{ featureType:'poi', stylers:[{{visibility:'off'}}] }},
    ]
  }});

  directionsService = new google.maps.DirectionsService();

  // ── ITL3 choropleth — clickable:false so clicks pass through ─
  const pops   = ITL3_DATA.features.map(f => f.properties.population_2024 || 0);
  const minPop = Math.min(...pops), maxPop = Math.max(...pops);

  function popColour(pop) {{
    if (!pop) return '#374151';
    const t = (pop - minPop) / (maxPop - minPop);
    const r = Math.round(255 * Math.min(1, t * 2));
    const g = Math.round(255 * Math.max(0, 1 - t * 2));
    return `rgb(${{r}},${{g}},50)`;
  }}

  const infoWindow = new google.maps.InfoWindow();

  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const coords = f.geometry.coordinates[0].map(c => ({{lat:c[1], lng:c[0]}}));
    const poly = new google.maps.Polygon({{
      paths:        coords,
      strokeColor:  '#374151',
      strokeWeight: 0.5,
      fillColor:    popColour(f.properties.population_2024),
      fillOpacity:  0.5,
      clickable:    false,
      map:          map
    }});
  }});

  // Isochrone outlines — also non-clickable
  const isoColours = {{ LHR:'#3b82f6', LGW:'#ef4444', LTN:'#22c55e' }};
  const isoWeights = [3, 2, 1.5];

  Object.entries(ISO_DATA).forEach(([iata, geojson]) => {{
    geojson.features.forEach((feature, i) => {{
      const coords = feature.geometry.coordinates[0].map(c => ({{lat:c[1], lng:c[0]}}));
      new google.maps.Polygon({{
        paths:         coords,
        strokeColor:   isoColours[iata],
        strokeWeight:  isoWeights[i],
        strokeOpacity: 0.9,
        fillOpacity:   0,
        clickable:     false,
        map:           map
      }});
    }});
  }});

  // Airport markers
  Object.entries(AIRPORTS).forEach(([code, ap]) => {{
    const marker = new google.maps.Marker({{
      position: {{ lat:ap.lat, lng:ap.lng }},
      map:      map,
      title:    ap.name,
      label:    {{ text:'✈', color:'white', fontSize:'12px' }},
      icon: {{
        path:        google.maps.SymbolPath.CIRCLE,
        scale:       14,
        fillColor:   ap.colour,
        fillOpacity: 1,
        strokeColor: 'white',
        strokeWeight:2,
      }}
    }});
    const iw = new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click', () => iw.open(map, marker));
  }});

  // Click handler
  map.addListener('click', e => {{
    const lat = e.latLng.lat();
    const lng = e.latLng.lng();
    document.getElementById('coords').textContent =
      `Start: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
    if (startMarker) startMarker.setMap(null);
    startMarker = new google.maps.Marker({{
      position: {{ lat, lng }},
      map:      map,
      title:    'Start point',
      icon: {{
        path:        google.maps.SymbolPath.CIRCLE,
        scale:       10,
        fillColor:   '#ff8800',
        fillOpacity: 1,
        strokeColor: 'white',
        strokeWeight:2,
      }}
    }});
    clearRoutes();
    calculateAllRoutes(lat, lng);
  }});

  document.getElementById('empty').textContent =
    'Click anywhere on the map to calculate journey times.';
}}

function clearRoutes() {{
  renderers.forEach(r => r.setMap(null));
  renderers = [];
}}

function calculateAllRoutes(lat, lng) {{
  const lb = document.getElementById('loading-bar');
  lb.style.display = 'block';
  lb.style.color   = '#f59e0b';
  lb.textContent   = '⏳ Calculating routes...';

  const html = Object.entries(AIRPORTS).map(([code, ap]) => `
    <div class="airport-block">
      <div class="airport-header">
        <div class="dot" style="background:${{ap.colour}}"></div>
        ${{ap.name}} (${{code}})
      </div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left">
          <div class="route-icon" style="background:#ff880033">🚗</div>
          <div class="route-label">Drive direct</div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left">
          <div class="route-icon" style="background:#00ccff33">🚆</div>
          <div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div>
        </div><div class="spinner"></div>
      </div>
    </div>
  `).join('');
  document.getElementById('results').innerHTML = html;

  let completed = 0;
  const total   = Object.keys(AIRPORTS).length * 2;

  Object.entries(AIRPORTS).forEach(([code, ap]) => {{
    ['DRIVING', 'TRANSIT'].forEach(mode => {{
      const renderer = new google.maps.DirectionsRenderer({{
        suppressMarkers: true,
        polylineOptions: {{
          strokeColor:   ROUTE_COLOURS[mode],
          strokeWeight:  mode === 'DRIVING' ? 5 : 4,
          strokeOpacity: 0.85,
        }}
      }});
      renderers.push(renderer);

      directionsService.route({{
        origin:      {{ lat, lng }},
        destination: {{ lat:ap.lat, lng:ap.lng }},
        travelMode:  google.maps.TravelMode[mode],
        ...(mode === 'TRANSIT' ? {{
          transitOptions: {{
            departureTime: new Date(),
            modes: [
              google.maps.TransitMode.RAIL,
              google.maps.TransitMode.SUBWAY,
              google.maps.TransitMode.BUS
            ],
            routingPreference: google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }}
        }} : {{}})
      }}, (result, status) => {{
        completed++;
        const el = document.getElementById(`r-${{code}}-${{mode}}`);

        if (status === 'OK') {{
          renderer.setMap(map);
          renderer.setDirections(result);
          const leg  = result.routes[0].legs[0];
          const mins = Math.round(leg.duration.value / 60);
          const km   = (leg.distance.value / 1000).toFixed(1);

          if (el) {{
            const spinner = el.querySelector('.spinner');
            if (spinner) spinner.outerHTML =
              `<div class="route-val done">${{mins}} min · ${{km}} km</div>`;

            if (mode === 'TRANSIT') {{
              const steps = result.routes[0].legs[0].steps
                .filter(s => s.travel_mode === 'TRANSIT')
                .map(s => s.transit?.line?.short_name || s.transit?.line?.name || '')
                .filter(Boolean).join(' → ');
              if (steps) {{
                const sub = el.querySelector('.route-sub');
                if (sub) sub.textContent = steps;
              }}
            }}

            el.onclick = () => {{
              renderers.forEach(r => r.setOptions({{
                polylineOptions:{{ strokeOpacity:0.15, strokeWeight:2 }}
              }}));
              renderer.setOptions({{polylineOptions:{{
                strokeColor:   ROUTE_COLOURS[mode],
                strokeOpacity: 1,
                strokeWeight:  6
              }}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) {{
            const spinner = el.querySelector('.spinner');
            if (spinner) spinner.outerHTML =
              `<div class="route-val" style="color:#64748b">not available</div>`;
          }}
        }}

        if (completed === total) {{
          lb.textContent = '✓ All routes loaded — click a row to highlight';
          lb.style.color = '#22c55e';
        }}
      }});
    }});
  }});
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_v7.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print('  Saved: airport_journey_planner_v7.html')

comparison between different times at different locations or same -

check how many queries - how much do we have to pay -

find some information about buses or coaches -

straight line distance to the airports -

the other way around as well - from the airport to wherever -

if you do have a point within --ITL3 -- maybe have some center points from where you can get there

In [ ]:
# Check geometry types in catchment
for _, row in itl3_catchment_filtered.iterrows():
    geom_type = row.geometry.geom_type
    if geom_type == 'MultiPolygon':
        print(f'MultiPolygon: {row["ITL321NM"]}')

These areas are split up into more than one polygon. ie . hounslow has island within thames and that may be causing the issue of why these areas are not showing on the map.  trying to fix it


In [ ]:
# Re-export ITL3 JSON with MultiPolygon support
# Convert all geometries to GeoJSON properly
itl3_export = itl3_catchment_filtered[['ITL321NM', 'population_2024', 'GDHI_2023_million_gbp', 'geometry']].copy()

# Use geopandas to_json which handles MultiPolygons correctly
itl3_json = json.loads(itl3_export.to_json())

# Verify MultiPolygons are included
multi_count = sum(1 for f in itl3_json['features']
                  if f['geometry']['type'] == 'MultiPolygon')
print(f' Total regions: {len(itl3_json["features"])}')
print(f' MultiPolygon regions: {multi_count}')
print(f' Polygon regions: {len(itl3_json["features"]) - multi_count}')

MultiPolygon regions: 6

MultiPolygon: Thurrock
MultiPolygon: Bexley and Greenwich
MultiPolygon: Hounslow and Richmond upon Thames
MultiPolygon: East Sussex CC
MultiPolygon: West Sussex (North East)
MultiPolygon: West Kent .

so these are the regions

In [ ]:
# Check which Inner London regions are missing
inner_london = ['Hackney', 'Tower Hamlets', 'Newham', 'Southwark',
                'Lewisham', 'Lambeth', 'Westminster', 'Camden',
                'Islington', 'Greenwich', 'Peckham']

for keyword in inner_london:
    in_filtered = itl3_catchment_filtered[
        itl3_catchment_filtered['ITL321NM'].str.contains(keyword, case=False)
    ]['ITL321NM'].tolist()
    in_all = itl3[
        itl3['ITL321NM'].str.contains(keyword, case=False)
    ]['ITL321NM'].tolist()
    print(f'{keyword}: filtered={in_filtered}, all={in_all}')

These zones are missing. trying to fix them


In [ ]:
from shapely.ops import unary_union
from shapely.geometry import shape
import geopandas as gpd

# Build union of all 60 min isochrones
all_polys = []
for iata in ['LHR', 'LGW', 'LTN']:
    features = all_isochrones[iata]['features']
    geom = shape(features[-1]['geometry'])
    all_polys.append(geom)

combined = unary_union(all_polys)
combined_gdf = gpd.GeoDataFrame(geometry=[combined], crs='EPSG:4326')

# Use INTERSECTS instead of centroid within
itl3_projected     = itl3_v2.to_crs('EPSG:27700').copy()
combined_projected = combined_gdf.to_crs('EPSG:27700').geometry.iloc[0]

# Any region that intersects the catchment gets included
itl3_catchment_filtered = itl3_v2[
    itl3_projected.geometry.intersects(combined_projected).values
].copy()

print(f' Regions with centroid method: 34')
print(f' Regions with intersection method: {len(itl3_catchment_filtered)}')

# Check if missing regions are now included
for keyword in ['Hackney', 'Tower Hamlets', 'Lewisham']:
    found = itl3_catchment_filtered[
        itl3_catchment_filtered['ITL321NM'].str.contains(keyword, case=False)
    ]['ITL321NM'].tolist()
    print(f'  {keyword}: {found}')

In [ ]:
# Re-export ITL3 JSON with all 50 regions
itl3_export = itl3_catchment_filtered[['ITL321NM', 'population_2024', 'GDHI_2023_million_gbp', 'geometry']].copy()
itl3_json = json.loads(itl3_export.to_json())

multi_count = sum(1 for f in itl3_json['features'] if f['geometry']['type'] == 'MultiPolygon')
print(f' Total regions: {len(itl3_json["features"])}')
print(f' MultiPolygon: {multi_count}')
print(f' Polygon: {len(itl3_json["features"]) - multi_count}')

In [ ]:
centroids = []
for _, row in itl3_catchment_filtered.iterrows():
    centroid = row.geometry.centroid
    pop  = int(row['population_2024']) if pd.notna(row['population_2024']) else 0
    gdhi = round(float(row['GDHI_2023_million_gbp']), 1) if pd.notna(row['GDHI_2023_million_gbp']) else 0
    centroids.append({
        'name': row['ITL321NM'],
        'lat':  round(centroid.y, 5),
        'lng':  round(centroid.x, 5),
        'pop':  pop,
        'gdhi': gdhi
    })

centroids_json = json.dumps(centroids)
print(f' {len(centroids)} centroids calculated')

In [ ]:
output_path = '/content/drive/MyDrive/airport_journey_planner_v11.html'
with open(output_path, 'w') as f:
    f.write(html_content)

In [ ]:
# Check if missing regions have geometry and data
missing = ['Hackney and Newham', 'Tower Hamlets', 'Lewisham and Southwark']

for name in missing:
    row = itl3_catchment_filtered[itl3_catchment_filtered['ITL321NM'] == name]
    if len(row) > 0:
        r = row.iloc[0]
        print(f'{name}:')
        print(f'  geometry type: {r.geometry.geom_type}')
        print(f'  population: {r["population_2024"]}')
        print(f'  GDHI: {r["GDHI_2023_million_gbp"]}')
        print(f'  geometry valid: {r.geometry.is_valid}')
    else:
        print(f'{name}: NOT FOUND')

In [ ]:
# Check what the GeoJSON looks like for Tower Hamlets

tower_hamlets = itl3_catchment_filtered[
    itl3_catchment_filtered['ITL321NM'] == 'Tower Hamlets'
]

th_json = json.loads(tower_hamlets[['ITL321NM', 'population_2024', 'geometry']].to_json())
feature = th_json['features'][0]

print(f'Geometry type: {feature["geometry"]["type"]}')
print(f'Population: {feature["properties"]["population_2024"]}')
print(f'Num coordinate rings: {len(feature["geometry"]["coordinates"])}')
print(f'First ring first 3 coords: {feature["geometry"]["coordinates"][0][:3]}')

In [ ]:
# Verify the current itl3_json has 50 regions
import json
features = itl3_json['features']
print(f'Regions in itl3_json: {len(features)}')

# Check if Tower Hamlets is in it
names = [f['properties']['ITL321NM'] for f in features]
print(f'Tower Hamlets in itl3_json: {"Tower Hamlets" in names}')
print(f'Hackney and Newham in itl3_json: {"Hackney and Newham" in names}')

In [ ]:
# Check the generated HTML has 50 regions embedded
with open('/content/drive/MyDrive/airport_journey_planner_v11.html', 'r') as f:
    content = f.read()

print(f'Tower Hamlets in HTML: {"Tower Hamlets" in content}')
print(f'Hackney and Newham in HTML: {"Hackney and Newham" in content}')
print(f'Lewisham and Southwark in HTML: {"Lewisham and Southwark" in content}')


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_v11.html', 'r') as f:
    content = f.read()
print(f'Tower Hamlets in HTML: {"Tower Hamlets" in content}')
print(f'Regions count check: {"Hackney and Newham" in content}')

In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; }}
  #header {{ background:#1a1a2e; color:white; padding:10px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:14px; font-weight:bold; }}
  #api-bar {{ background:#0d1525; padding:6px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; white-space:nowrap; }}
  #api-bar button:hover {{ background:#2563eb; }}
  #control-bar {{ background:#0d1525; padding:6px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #control-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .mode-btn {{ padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; border:1px solid #2a3a50; background:#1e2d45; color:#94a3b8; }}
  .mode-btn.active {{ background:#3b82f6; color:white; border-color:#3b82f6; }}
  #itl3-select {{ background:#1e2d45; border:1px solid #2a3a50; color:#e2e8f0; padding:4px 8px; font-size:11px; border-radius:3px; max-width:220px; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; background:#0a0e1a; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}
  #sidebar {{ width:340px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:12px 16px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:10px; }}
  .airport-block {{ margin-bottom:10px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:8px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:12px; font-weight:bold; }}
  .dot {{ width:8px; height:8px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:7px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .straight-row {{ padding:5px 12px; background:#0d1525; border-bottom:1px solid #1e2d45; font-size:10px; color:#64748b; display:flex; justify-content:space-between; }}
  .route-left {{ display:flex; align-items:center; gap:8px; }}
  .route-icon {{ width:20px; height:20px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:11px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:11px; height:11px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:30px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:5px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:8px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:8px; }}
  .leg {{ display:flex; align-items:center; gap:4px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:16px; height:3px; border-radius:1px; }}
  #loading-bar {{ padding:6px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key here…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="control-bar">
  <label>Direction:</label>
  <button class="mode-btn active" id="btn-to"   onclick="setMode('to')">📍→✈ To Airport</button>
  <button class="mode-btn"        id="btn-from" onclick="setMode('from')">✈→📍 From Airport</button>
  <label style="margin-left:10px">ITL3 Region:</label>
  <select id="itl3-select" onchange="selectITL3()">
    <option value="">— or pick an ITL3 region —</option>
  </select>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">
      Enter your API key and click Load Map.<br><br>
      Click anywhere on the map or pick an ITL3 region from the dropdown.
      Straight-line distances shown for each airport.
    </div>
    <div id="results">
      <div id="empty">Load the map first then<br>click anywhere to begin.</div>
    </div>
    <div id="loading-bar"> Calculating routes</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e' }},
}};

const ROUTE_COLOURS = {{ DRIVING:'#ff8800', TRANSIT:'#00ccff' }};
const ISO_DATA    = {json.dumps(iso_geojson)};
const ITL3_DATA   = {json.dumps(itl3_json)};
const CENTROIDS   = {centroids_json};

let map, directionsService;
let startMarker = null, renderers = [], straightLines = [];
let mapLoaded = false, currentMode = 'to';

window.addEventListener('load', () => {{
  const sel = document.getElementById('itl3-select');
  CENTROIDS.sort((a,b) => a.name.localeCompare(b.name)).forEach(c => {{
    const opt = document.createElement('option');
    opt.value = JSON.stringify({{lat:c.lat, lng:c.lng}});
    opt.textContent = `${{c.name}} (pop: ${{c.pop.toLocaleString()}})`;
    sel.appendChild(opt);
  }});
}});

function selectITL3() {{
  const val = document.getElementById('itl3-select').value;
  if (!val || !map) return;
  const {{ lat, lng }} = JSON.parse(val);
  handlePoint(lat, lng);
}}

function setMode(mode) {{
  currentMode = mode;
  document.getElementById('btn-to').className   = 'mode-btn' + (mode === 'to'   ? ' active' : '');
  document.getElementById('btn-from').className = 'mode-btn' + (mode === 'from' ? ' active' : '');
  clearRoutes();
  if (startMarker) {{ startMarker.setMap(null); startMarker = null; }}
  document.getElementById('results').innerHTML  = '<div id="empty">Click anywhere or pick an ITL3 region.</div>';
  document.getElementById('coords').textContent = 'Coordinates: —';
}}

function haversine(lat1, lng1, lat2, lng2) {{
  const R = 6371;
  const dLat = (lat2-lat1)*Math.PI/180, dLng = (lng2-lng1)*Math.PI/180;
  const a = Math.sin(dLat/2)**2 +
    Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLng/2)**2;
  return (R * 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a))).toFixed(1);
}}

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your Google Maps API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display = 'none';
  document.getElementById('map').style.display = 'block';
  const script = document.createElement('script');
  script.src = `https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async = true; script.defer = true;
  document.head.appendChild(script);
  mapLoaded = true;
}}

// ── Helper: convert GeoJSON coords to Google Maps paths ──────
function coordsToPath(coords) {{
  return coords.map(c => ({{lat:c[1], lng:c[0]}}));
}}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center: {{lat:51.5, lng:-0.3}}, zoom:8, mapTypeId:'roadmap',
    styles:[
      {{elementType:'geometry',stylers:[{{color:'#1a1a2e'}}]}},
      {{elementType:'labels.text.fill',stylers:[{{color:'#94a3b8'}}]}},
      {{elementType:'labels.text.stroke',stylers:[{{color:'#1a1a2e'}}]}},
      {{featureType:'road',elementType:'geometry',stylers:[{{color:'#2d3748'}}]}},
      {{featureType:'road.highway',elementType:'geometry',stylers:[{{color:'#2d4a7a'}}]}},
      {{featureType:'water',elementType:'geometry',stylers:[{{color:'#0d1525'}}]}},
      {{featureType:'transit.line',elementType:'geometry',stylers:[{{color:'#4a5568'}}]}},
      {{featureType:'transit.station',elementType:'geometry',stylers:[{{color:'#374151'}}]}},
      {{featureType:'poi',stylers:[{{visibility:'off'}}]}},
    ]
  }});

  directionsService = new google.maps.DirectionsService();

  // ── ITL3 choropleth — handles Polygon AND MultiPolygon ────
  const pops   = ITL3_DATA.features.map(f => f.properties.population_2024 || 0);
  const minPop = Math.min(...pops), maxPop = Math.max(...pops);

  function popColour(pop) {{
    if (!pop) return '#374151';
    const t = (pop - minPop) / (maxPop - minPop);
    const r = Math.round(255 * Math.min(1, t * 2));
    const g = Math.round(255 * Math.max(0, 1 - t * 2));
    return `rgb(${{r}},${{g}},50)`;
  }}

  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const p        = f.properties;
    const colour   = popColour(p.population_2024);
    const geomType = f.geometry.type;

    // Get list of polygon rings depending on geometry type
    const rings = geomType === 'MultiPolygon'
      ? f.geometry.coordinates.map(poly => poly[0])  // outer ring of each polygon
      : [f.geometry.coordinates[0]];                  // single outer ring

    rings.forEach(ring => {{
      new google.maps.Polygon({{
        paths:        coordsToPath(ring),
        strokeColor:  '#374151',
        strokeWeight: 0.5,
        fillColor:    colour,
        fillOpacity:  0.5,
        clickable:    false,
        map:          map
      }});
    }});
  }});

  // ── ITL3 centroid markers ─────────────────────────────────
  CENTROIDS.forEach(c => {{
    const marker = new google.maps.Marker({{
      position: {{lat:c.lat, lng:c.lng}}, map,
      title: c.name,
      icon: {{
        path:google.maps.SymbolPath.CIRCLE, scale:5,
        fillColor:'#ffffff', fillOpacity:0.6,
        strokeColor:'#94a3b8', strokeWeight:1
      }}
    }});
    const iw = new google.maps.InfoWindow({{
      content:`<div style="font-size:11px"><b>${{c.name}}</b><br>Pop: ${{c.pop.toLocaleString()}}<br>GDHI: £${{c.gdhi.toLocaleString()}}M<br><small>Click to calculate routes</small></div>`
    }});
    marker.addListener('click', () => {{ iw.open(map, marker); handlePoint(c.lat, c.lng); }});
  }});

  // ── Isochrones ────────────────────────────────────────────
  const isoColours = {{LHR:'#3b82f6', LGW:'#ef4444', LTN:'#22c55e'}};
  Object.entries(ISO_DATA).forEach(([iata, geojson]) => {{
    geojson.features.forEach((feature, i) => {{
      new google.maps.Polygon({{
        paths:         coordsToPath(feature.geometry.coordinates[0]),
        strokeColor:   isoColours[iata],
        strokeWeight:  [3,2,1.5][i],
        strokeOpacity: 0.9,
        fillOpacity:   0,
        clickable:     false,
        map
      }});
    }});
  }});

  // ── Airport markers ───────────────────────────────────────
  Object.entries(AIRPORTS).forEach(([code, ap]) => {{
    const marker = new google.maps.Marker({{
      position:{{lat:ap.lat, lng:ap.lng}}, map, title:ap.name,
      label:{{text:'✈', color:'white', fontSize:'12px'}},
      icon:{{
        path:google.maps.SymbolPath.CIRCLE, scale:14,
        fillColor:ap.colour, fillOpacity:1,
        strokeColor:'white', strokeWeight:2
      }}
    }});
    const iw = new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click', () => iw.open(map, marker));
  }});

  map.addListener('click', e => handlePoint(e.latLng.lat(), e.latLng.lng()));
  document.getElementById('empty').textContent = 'Click anywhere or pick an ITL3 region.';
}}

function clearRoutes() {{
  renderers.forEach(r => r.setMap(null)); renderers = [];
  straightLines.forEach(l => l.setMap(null)); straightLines = [];
}}

function handlePoint(lat, lng) {{
  document.getElementById('coords').textContent =
    `${{currentMode==='to'?'From':'To'}}: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
  if (startMarker) startMarker.setMap(null);
  startMarker = new google.maps.Marker({{
    position:{{lat,lng}}, map,
    title: currentMode==='to' ? 'Start point' : 'Destination',
    icon:{{
      path:google.maps.SymbolPath.CIRCLE, scale:10,
      fillColor: currentMode==='to' ? '#ff8800' : '#ff00ff',
      fillOpacity:1, strokeColor:'white', strokeWeight:2
    }}
  }});
  clearRoutes();
  calculateAllRoutes(lat, lng);
}}

function calculateAllRoutes(lat, lng) {{
  const lb = document.getElementById('loading-bar');
  lb.style.display='block'; lb.style.color='#f59e0b';
  lb.textContent=' Calculating routes';

  const straight = {{}};
  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    straight[code] = haversine(lat, lng, ap.lat, ap.lng);
  }});

  const modeLabel = currentMode==='to' ? '→ ✈' : '✈ →';
  const html = Object.entries(AIRPORTS).map(([code,ap]) => `
    <div class="airport-block">
      <div class="airport-header">
        <div class="dot" style="background:${{ap.colour}}"></div>
        ${{modeLabel}} ${{ap.name}} (${{code}})
      </div>
      <div class="straight-row">
        <span> Straight line</span>
        <span style="color:#e2e8f0">${{straight[code]}} km</span>
      </div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left">
          <div class="route-icon" style="background:#ff880033">🚗</div>
          <div class="route-label">Drive</div>
        </div><div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left">
          <div class="route-icon" style="background:#00ccff33">🚆</div>
          <div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div>
        </div><div class="spinner"></div>
      </div>
    </div>
  `).join('');
  document.getElementById('results').innerHTML = html;

  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    const line = new google.maps.Polyline({{
      path:[{{lat,lng}},{{lat:ap.lat,lng:ap.lng}}],
      strokeColor:ap.colour, strokeWeight:1, strokeOpacity:0.5,
      icons:[{{icon:{{path:'M 0,-1 0,1',strokeOpacity:1,scale:3}},offset:'0',repeat:'12px'}}],
      map
    }});
    straightLines.push(line);
  }});

  let completed = 0;
  const total   = Object.keys(AIRPORTS).length * 2;

  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    ['DRIVING','TRANSIT'].forEach(mode => {{
      const renderer = new google.maps.DirectionsRenderer({{
        suppressMarkers:true,
        polylineOptions:{{
          strokeColor:ROUTE_COLOURS[mode],
          strokeWeight:mode==='DRIVING'?5:4,
          strokeOpacity:0.85
        }}
      }});
      renderers.push(renderer);

      const origin      = currentMode==='to' ? {{lat,lng}}              : {{lat:ap.lat,lng:ap.lng}};
      const destination = currentMode==='to' ? {{lat:ap.lat,lng:ap.lng}} : {{lat,lng}};

      directionsService.route({{
        origin, destination,
        travelMode: google.maps.TravelMode[mode],
        ...(mode==='TRANSIT' ? {{
          transitOptions:{{
            departureTime:new Date(),
            modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
            routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }}
        }} : {{}})
      }}, (result, status) => {{
        completed++;
        const el = document.getElementById(`r-${{code}}-${{mode}}`);
        if (status==='OK') {{
          renderer.setMap(map);
          renderer.setDirections(result);
          const leg  = result.routes[0].legs[0];
          const mins = Math.round(leg.duration.value/60);
          const km   = (leg.distance.value/1000).toFixed(1);
          if (el) {{
            el.querySelector('.spinner').outerHTML =
              `<div class="route-val done">${{mins}} min · ${{km}} km</div>`;
            if (mode==='TRANSIT') {{
              const steps = result.routes[0].legs[0].steps
                .filter(s=>s.travel_mode==='TRANSIT')
                .map(s=>s.transit?.line?.short_name||s.transit?.line?.name||'')
                .filter(Boolean).join(' → ');
              if (steps) {{ const sub=el.querySelector('.route-sub'); if(sub) sub.textContent=steps; }}
            }}
            el.onclick = () => {{
              renderers.forEach(r=>r.setOptions({{polylineOptions:{{strokeOpacity:0.15,strokeWeight:2}}}}));
              renderer.setOptions({{polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeOpacity:1,strokeWeight:6}}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) el.querySelector('.spinner').outerHTML =
            `<div class="route-val" style="color:#64748b">not available</div>`;
        }}
        if (completed===total) {{
          lb.textContent='✓ All routes loaded — click a row to highlight';
          lb.style.color='#22c55e';
        }}
      }});
    }});
  }});
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_v11.html'
with open(output_path, 'w') as f:
    f.write(html_content)


In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; overflow:hidden; }}

  #header {{ background:#1a1a2e; color:white; padding:8px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:13px; font-weight:bold; }}

  #api-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; }}
  #api-bar button:hover {{ background:#2563eb; }}

  #control-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:8px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #control-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .mode-btn {{ padding:4px 10px; font-size:11px; border-radius:3px; cursor:pointer; border:1px solid #2a3a50; background:#1e2d45; color:#94a3b8; }}
  .mode-btn.active {{ background:#3b82f6; color:white; border-color:#3b82f6; }}
  #itl3-select {{ background:#1e2d45; border:1px solid #2a3a50; color:#e2e8f0; padding:4px 8px; font-size:11px; border-radius:3px; max-width:190px; }}

  #toggle-bar {{ background:#080d1a; padding:5px 20px; display:flex; align-items:center; gap:16px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #toggle-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .airport-group {{ display:flex; align-items:center; gap:5px; }}
  .airport-group-label {{ font-size:11px; font-weight:bold; margin-right:2px; }}
  .tog {{ padding:2px 8px; font-size:10px; border-radius:3px; cursor:pointer; border:1px solid; opacity:0.35; transition:opacity 0.15s,background 0.15s; }}
  .tog.on {{ opacity:1; }}
  button.compare-btn {{ background:#7c3aed; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; margin-left:auto; }}
  button.compare-btn:hover {{ background:#6d28d9; }}
  button.aircraft-btn {{ background:#0891b2; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; }}
  button.aircraft-btn:hover {{ background:#0e7490; }}

  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}

  #sidebar {{ width:320px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:8px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:8px; }}
  .airport-block {{ margin-bottom:8px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:7px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; }}
  .dot {{ width:7px; height:7px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:6px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .straight-row {{ padding:4px 12px; background:#0d1525; border-bottom:1px solid #1e2d45; font-size:10px; color:#64748b; display:flex; justify-content:space-between; }}
  .route-left {{ display:flex; align-items:center; gap:7px; }}
  .route-icon {{ width:18px; height:18px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:10px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:10px; height:10px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:24px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:4px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:6px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:6px; }}
  .leg {{ display:flex; align-items:center; gap:3px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:14px; height:2px; border-radius:1px; }}
  #loading-bar {{ padding:5px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}

  /* ── Comparison popup ─────────────────────────────────── */
  #compare-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #compare-overlay.open {{ display:flex; }}
  #compare-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:90vw; max-width:1100px; max-height:85vh; display:flex; flex-direction:column; overflow:hidden; }}
  #compare-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #compare-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #compare-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #compare-controls {{ padding:10px 20px; border-bottom:1px solid #1e2d45; display:flex; align-items:center; gap:10px; flex-wrap:wrap; }}
  #compare-controls span {{ font-size:11px; color:#64748b; }}
  #compare-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  #compare-table {{ width:100%; border-collapse:collapse; font-size:11px; }}
  #compare-table th {{ background:#1e2d45; color:#94a3b8; padding:8px 10px; text-align:left; border:1px solid #2a3a50; white-space:nowrap; position:sticky; top:0; }}
  #compare-table td {{ padding:6px 10px; border:1px solid #1e2d45; color:#e2e8f0; white-space:nowrap; }}
  #compare-table tr:nth-child(even) td {{ background:#0d1525; }}
  #compare-table td.drive {{ color:#ff8800; }}
  #compare-table td.transit {{ color:#00ccff; }}
  #compare-table td.loading {{ color:#64748b; font-style:italic; }}
  #compare-progress {{ padding:8px 20px; border-top:1px solid #1e2d45; font-size:10px; color:#f59e0b; display:none; }}
  .loc-pin {{ display:inline-block; width:10px; height:10px; border-radius:50%; margin-right:4px; }}

  /* ── Aircraft popup ───────────────────────────────────── */
  #aircraft-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #aircraft-overlay.open {{ display:flex; }}
  #aircraft-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:600px; max-height:80vh; display:flex; flex-direction:column; overflow:hidden; }}
  #aircraft-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #aircraft-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #aircraft-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #aircraft-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  .aircraft-airport {{ margin-bottom:16px; }}
  .aircraft-airport h3 {{ font-size:12px; font-weight:bold; margin-bottom:8px; padding-bottom:4px; border-bottom:1px solid #1e2d45; }}
  .aircraft-table {{ width:100%; border-collapse:collapse; font-size:11px; }}
  .aircraft-table th {{ background:#1e2d45; color:#94a3b8; padding:6px 8px; text-align:left; border:1px solid #2a3a50; }}
  .aircraft-table td {{ padding:5px 8px; border:1px solid #1e2d45; color:#e2e8f0; }}
  .aircraft-table tr:nth-child(even) td {{ background:#0d1525; }}
  .status-ground {{ color:#22c55e; }}
  .status-air {{ color:#f59e0b; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="control-bar">
  <label>Direction:</label>
  <button class="mode-btn active" id="btn-to"   onclick="setMode('to')">📍→✈ To Airport</button>
  <button class="mode-btn"        id="btn-from" onclick="setMode('from')">✈→📍 From Airport</button>
  <label style="margin-left:8px">ITL3:</label>
  <select id="itl3-select" onchange="selectITL3()">
    <option value="">— pick a region —</option>
  </select>
</div>

<div id="toggle-bar">
  <label>Catchment zones:</label>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#3b82f6">LHR</span>
    <button class="tog on" id="tog-LHR-30" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',30)">30</button>
    <button class="tog on" id="tog-LHR-45" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',45)">45</button>
    <button class="tog on" id="tog-LHR-60" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#ef4444">LGW</span>
    <button class="tog on" id="tog-LGW-30" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',30)">30</button>
    <button class="tog on" id="tog-LGW-45" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',45)">45</button>
    <button class="tog on" id="tog-LGW-60" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#22c55e">LTN</span>
    <button class="tog on" id="tog-LTN-30" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',30)">30</button>
    <button class="tog on" id="tog-LTN-45" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',45)">45</button>
    <button class="tog on" id="tog-LTN-60" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',60)">60</button>
  </div>
  <button class="aircraft-btn" onclick="openAircraft()">✈ Live Aircraft</button>
  <button class="compare-btn"  onclick="openCompare()"> Compare</button>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">
      Click anywhere on the map or pick an ITL3 region.<br>
      Toggle catchment zones using the buttons above.<br>
      Use Compare for 24h time analysis.
    </div>
    <div id="results"><div id="empty">Load the map then click anywhere.</div></div>
    <div id="loading-bar"> Calculating routes...</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<!-- ── Comparison popup ──────────────────────────────────── -->
<div id="compare-overlay">
  <div id="compare-panel">
    <div id="compare-header">
      <h2> Journey Time Comparison — 24 Hour Analysis</h2>
      <button onclick="closeCompare()">✕</button>
    </div>
    <div id="compare-controls">
      <span>Click on the map to add locations (max 3). Each location calculates routes for all 24 hours.</span>
      <button class="mode-btn" onclick="clearCompare()" style="margin-left:auto">Clear All</button>
    </div>
    <div id="compare-progress"></div>
    <div id="compare-body">
      <div style="color:#64748b;font-size:11px;text-align:center;padding:40px">
        Close this panel and click locations on the map to add them to the comparison.
      </div>
    </div>
  </div>
</div>

<!-- ── Aircraft popup ────────────────────────────────────── -->
<div id="aircraft-overlay">
  <div id="aircraft-panel">
    <div id="aircraft-header">
      <h2>✈ Live Aircraft Data</h2>
      <button onclick="closeAircraft()">✕</button>
    </div>
    <div id="aircraft-body">
      <div style="color:#64748b;font-size:11px;text-align:center;padding:40px">
        Loading live aircraft data from OpenSky Network...
      </div>
    </div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6',
          bbox:{{ lamin:51.44, lomin:-0.50, lamax:51.51, lomax:-0.42 }} }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444',
          bbox:{{ lamin:51.13, lomin:-0.22, lamax:51.18, lomax:-0.14 }} }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e',
          bbox:{{ lamin:51.86, lomin:-0.42, lamax:51.91, lomax:-0.32 }} }},
}};

const ROUTE_COLOURS = {{ DRIVING:'#ff8800', TRANSIT:'#00ccff' }};
const ISO_DATA    = {json.dumps(iso_geojson)};
const ITL3_DATA   = {json.dumps(itl3_json)};
const CENTROIDS   = {centroids_json};
const TIMES       = [30, 45, 60];
const ISO_WEIGHTS = [3, 2, 1.5];

let map, directionsService;
let startMarker   = null;
let renderers     = [];
let straightLines = [];
let isoPolygons   = {{}};  // isoPolygons['LHR'][30] = [polygon, ...]
let mapLoaded     = false;
let currentMode   = 'to';

// Compare state
let compareMode   = false;
let comparePoints = [];
const PIN_COLOURS = ['#f59e0b', '#a855f7', '#ec4899'];

// ── Populate ITL3 dropdown ────────────────────────────────
window.addEventListener('load', () => {{
  const sel = document.getElementById('itl3-select');
  CENTROIDS.sort((a,b) => a.name.localeCompare(b.name)).forEach(c => {{
    const opt = document.createElement('option');
    opt.value = JSON.stringify({{lat:c.lat, lng:c.lng, name:c.name}});
    opt.textContent = c.name;
    sel.appendChild(opt);
  }});
}});

function selectITL3() {{
  const val = document.getElementById('itl3-select').value;
  if (!val || !map) return;
  const {{ lat, lng }} = JSON.parse(val);
  handlePoint(lat, lng);
}}

function setMode(mode) {{
  currentMode = mode;
  document.getElementById('btn-to').className   = 'mode-btn' + (mode==='to'   ? ' active' : '');
  document.getElementById('btn-from').className = 'mode-btn' + (mode==='from' ? ' active' : '');
  clearRoutes();
  if (startMarker) {{ startMarker.setMap(null); startMarker=null; }}
  document.getElementById('results').innerHTML  = '<div id="empty">Click anywhere to begin.</div>';
  document.getElementById('coords').textContent = 'Coordinates: —';
}}

// ── Catchment toggle ──────────────────────────────────────
function toggleIso(iata, mins) {{
  const key = `${{iata}}-${{mins}}`;
  const btn = document.getElementById(`tog-${{iata}}-${{mins}}`);
  const polys = isoPolygons[iata]?.[mins] || [];
  const isOn = btn.classList.contains('on');
  polys.forEach(p => p.setMap(isOn ? null : map));
  btn.classList.toggle('on', !isOn);
  btn.style.opacity = isOn ? '0.35' : '1';
}}

function haversine(lat1,lng1,lat2,lng2) {{
  const R=6371, dLat=(lat2-lat1)*Math.PI/180, dLng=(lng2-lng1)*Math.PI/180;
  const a=Math.sin(dLat/2)**2+Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLng/2)**2;
  return (R*2*Math.atan2(Math.sqrt(a),Math.sqrt(1-a))).toFixed(1);
}}

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display = 'none';
  document.getElementById('map').style.display = 'block';
  const script = document.createElement('script');
  script.src = `https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async=true; script.defer=true;
  document.head.appendChild(script);
  mapLoaded = true;
}}

function coordsToPath(coords) {{
  return coords.map(c => ({{lat:c[1], lng:c[0]}}));
}}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center:{{lat:51.5,lng:-0.3}}, zoom:8, mapTypeId:'roadmap',
    styles:[
      {{elementType:'geometry',stylers:[{{color:'#1a1a2e'}}]}},
      {{elementType:'labels.text.fill',stylers:[{{color:'#94a3b8'}}]}},
      {{elementType:'labels.text.stroke',stylers:[{{color:'#1a1a2e'}}]}},
      {{featureType:'road',elementType:'geometry',stylers:[{{color:'#2d3748'}}]}},
      {{featureType:'road.highway',elementType:'geometry',stylers:[{{color:'#2d4a7a'}}]}},
      {{featureType:'water',elementType:'geometry',stylers:[{{color:'#0d1525'}}]}},
      {{featureType:'transit.line',elementType:'geometry',stylers:[{{color:'#4a5568'}}]}},
      {{featureType:'transit.station',elementType:'geometry',stylers:[{{color:'#374151'}}]}},
      {{featureType:'poi',stylers:[{{visibility:'off'}}]}},
    ]
  }});

  directionsService = new google.maps.DirectionsService();

  // ── ITL3 choropleth ───────────────────────────────────
  const pops   = ITL3_DATA.features.map(f=>f.properties.population_2024||0);
  const minPop = Math.min(...pops), maxPop = Math.max(...pops);
  function popColour(pop) {{
    if (!pop) return '#374151';
    const t=(pop-minPop)/(maxPop-minPop);
    return `rgb(${{Math.round(255*Math.min(1,t*2))}},${{Math.round(255*Math.max(0,1-t*2))}},50)`;
  }}

  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const colour  = popColour(f.properties.population_2024);
    const rings   = f.geometry.type==='MultiPolygon'
      ? f.geometry.coordinates.map(p=>p[0])
      : [f.geometry.coordinates[0]];
    rings.forEach(ring => {{
      new google.maps.Polygon({{
        paths:coordsToPath(ring),
        strokeColor:'#374151', strokeWeight:0.5,
        fillColor:colour, fillOpacity:0.5,
        clickable:false, map
      }});
    }});
  }});

  // ── ITL3 centroid markers ─────────────────────────────
  CENTROIDS.forEach(c => {{
    const marker = new google.maps.Marker({{
      position:{{lat:c.lat,lng:c.lng}}, map, title:c.name,
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:5,fillColor:'#fff',fillOpacity:0.6,strokeColor:'#94a3b8',strokeWeight:1}}
    }});
    const iw = new google.maps.InfoWindow({{
      content:`<div style="font-size:11px"><b>${{c.name}}</b><br>🚉 ${{c.station}}<br>Pop: ${{c.pop?.toLocaleString()}}<br>GDHI: £${{c.gdhi?.toLocaleString()}}M</div>`
    }});
    marker.addListener('click', ()=>{{ iw.open(map,marker); handlePoint(c.lat,c.lng); }});
  }});

  // ── Isochrones with toggle support ───────────────────
  const isoColours = {{LHR:'#3b82f6',LGW:'#ef4444',LTN:'#22c55e'}};
  Object.entries(ISO_DATA).forEach(([iata, geojson]) => {{
    isoPolygons[iata] = {{}};
    geojson.features.forEach((feature, i) => {{
      const mins = TIMES[i];
      const poly = new google.maps.Polygon({{
        paths:coordsToPath(feature.geometry.coordinates[0]),
        strokeColor:isoColours[iata], strokeWeight:ISO_WEIGHTS[i],
        strokeOpacity:0.9, fillOpacity:0, clickable:false, map
      }});
      if (!isoPolygons[iata][mins]) isoPolygons[iata][mins] = [];
      isoPolygons[iata][mins].push(poly);
    }});
  }});

  // ── Airport markers ───────────────────────────────────
  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    const marker = new google.maps.Marker({{
      position:{{lat:ap.lat,lng:ap.lng}}, map, title:ap.name,
      label:{{text:'✈',color:'white',fontSize:'11px'}},
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:13,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}}
    }});
    const iw = new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click',()=>iw.open(map,marker));
  }});

  // ── Click handler ─────────────────────────────────────
  map.addListener('click', e => {{
    const lat=e.latLng.lat(), lng=e.latLng.lng();
    if (compareMode) {{
      addComparePoint(lat, lng);
    }} else {{
      handlePoint(lat, lng);
    }}
  }});

  document.getElementById('empty').textContent='Click anywhere or pick an ITL3 region.';
}}

// ── Route clearing ────────────────────────────────────────
function clearRoutes() {{
  renderers.forEach(r=>r.setMap(null)); renderers=[];
  straightLines.forEach(l=>l.setMap(null)); straightLines=[];
}}

// ── Main routing ──────────────────────────────────────────
function handlePoint(lat, lng) {{
  document.getElementById('coords').textContent =
    `${{currentMode==='to'?'From':'To'}}: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
  if (startMarker) startMarker.setMap(null);
  startMarker = new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:9,
      fillColor:currentMode==='to'?'#ff8800':'#ff00ff',
      fillOpacity:1,strokeColor:'white',strokeWeight:2}}
  }});
  clearRoutes();
  calculateAllRoutes(lat, lng);
}}

function calculateAllRoutes(lat, lng) {{
  const lb=document.getElementById('loading-bar');
  lb.style.display='block'; lb.style.color='#f59e0b'; lb.textContent='⏳ Calculating routes...';

  const straight={{}};
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{ straight[code]=haversine(lat,lng,ap.lat,ap.lng); }});

  const modeLabel=currentMode==='to'?'→ ✈':'✈ →';
  document.getElementById('results').innerHTML = Object.entries(AIRPORTS).map(([code,ap])=>`
    <div class="airport-block">
      <div class="airport-header"><div class="dot" style="background:${{ap.colour}}"></div>${{modeLabel}} ${{ap.name}} (${{code}})</div>
      <div class="straight-row"><span> Straight line</span><span style="color:#e2e8f0">${{straight[code]}} km</span></div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left"><div class="route-icon" style="background:#ff880033">🚗</div><div class="route-label">Drive</div></div>
        <div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left"><div class="route-icon" style="background:#00ccff33">🚆</div><div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div></div>
        <div class="spinner"></div>
      </div>
    </div>`).join('');

  // Draw straight lines
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    const line=new google.maps.Polyline({{
      path:[{{lat,lng}},{{lat:ap.lat,lng:ap.lng}}],
      strokeColor:ap.colour,strokeWeight:1,strokeOpacity:0.5,
      icons:[{{icon:{{path:'M 0,-1 0,1',strokeOpacity:1,scale:3}},offset:'0',repeat:'12px'}}],map
    }});
    straightLines.push(line);
  }});

  let completed=0;
  const total=Object.keys(AIRPORTS).length*2;

  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    ['DRIVING','TRANSIT'].forEach(mode=>{{
      const renderer=new google.maps.DirectionsRenderer({{
        suppressMarkers:true,
        polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeWeight:mode==='DRIVING'?5:4,strokeOpacity:0.85}}
      }});
      renderers.push(renderer);

      const origin     =currentMode==='to'?{{lat,lng}}:{{lat:ap.lat,lng:ap.lng}};
      const destination=currentMode==='to'?{{lat:ap.lat,lng:ap.lng}}:{{lat,lng}};

      directionsService.route({{
        origin,destination,travelMode:google.maps.TravelMode[mode],
        ...(mode==='TRANSIT'?{{transitOptions:{{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}}}:{{}})
      }},(result,status)=>{{
        completed++;
        const el=document.getElementById(`r-${{code}}-${{mode}}`);
        if (status==='OK') {{
          renderer.setMap(map); renderer.setDirections(result);
          const leg=result.routes[0].legs[0];
          const mins=Math.round(leg.duration.value/60);
          const km=(leg.distance.value/1000).toFixed(1);
          if (el) {{
            el.querySelector('.spinner').outerHTML=`<div class="route-val done">${{mins}} min · ${{km}} km</div>`;
            if (mode==='TRANSIT') {{
              const steps=result.routes[0].legs[0].steps
                .filter(s=>s.travel_mode==='TRANSIT')
                .map(s=>s.transit?.line?.short_name||s.transit?.line?.name||'')
                .filter(Boolean).join(' → ');
              if (steps) {{ const sub=el.querySelector('.route-sub'); if(sub) sub.textContent=steps; }}
            }}
            el.onclick=()=>{{
              renderers.forEach(r=>r.setOptions({{polylineOptions:{{strokeOpacity:0.15,strokeWeight:2}}}}));
              renderer.setOptions({{polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeOpacity:1,strokeWeight:6}}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) el.querySelector('.spinner').outerHTML=`<div class="route-val" style="color:#64748b">not available</div>`;
        }}
        if (completed===total) {{
          lb.textContent='✓ Routes loaded — click a row to highlight'; lb.style.color='#22c55e';
        }}
      }});
    }});
  }});
}}

// ── COMPARISON POPUP ──────────────────────────────────────
let compareMarkers  = [];
let compareResults  = []; // array of {{lat,lng,name,data}}

function openCompare() {{
  compareMode = true;
  document.getElementById('compare-overlay').classList.add('open');
  document.getElementById('instruction').innerHTML =
    '<b style="color:#a855f7">Compare mode ON</b> — click up to 3 locations on map';
}}

function closeCompare() {{
  compareMode = false;
  document.getElementById('compare-overlay').classList.remove('open');
  document.getElementById('instruction').innerHTML =
    'Click anywhere on the map or pick an ITL3 region.';
}}

function clearCompare() {{
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Click locations on the map to add them to the comparison.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function addComparePoint(lat, lng) {{
  if (compareResults.length >= 3) {{
    alert('Maximum 3 locations. Click Clear All to reset.');
    return;
  }}
  const idx    = compareResults.length;
  const colour = PIN_COLOURS[idx];
  const name   = `Location ${{idx+1}} (${{lat.toFixed(3)}}, ${{lng.toFixed(3)}})`;

  const marker = new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},
    label:{{text:`${{idx+1}}`,color:'white',fontSize:'10px'}}
  }});
  compareMarkers.push(marker);

  const entry = {{lat, lng, name, colour, data:{{}}}};
  compareResults.push(entry);

  fetchCompareData(entry, idx);
}}

async function fetchCompareData(entry, idx) {{
  const prog = document.getElementById('compare-progress');
  prog.style.display='block';
  prog.textContent=` Calculating 24h routes for ${{entry.name}}... (this takes ~2 min)`;

  const hours = Array.from({{length:24}}, (_,i)=>i);
  const apiKey = document.getElementById('apiKey').value.trim();

  // We need to re-inject key for DirectionsService — already loaded
  entry.data = {{}};

  for (const hour of hours) {{
    const depTime = new Date();
    depTime.setHours(hour, 0, 0, 0);
    if (depTime < new Date()) depTime.setDate(depTime.getDate()+1);

    entry.data[hour] = {{}};

    for (const [code, ap] of Object.entries(AIRPORTS)) {{
      entry.data[hour][code] = {{}};

      for (const mode of ['DRIVING','TRANSIT']) {{
        await new Promise(resolve => {{
          directionsService.route({{
            origin:{{lat:entry.lat, lng:entry.lng}},
            destination:{{lat:ap.lat, lng:ap.lng}},
            travelMode:google.maps.TravelMode[mode],
            ...(mode==='TRANSIT'?{{transitOptions:{{
              departureTime:depTime,
              modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
              routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
            }}}}:{{}})
          }},(result,status)=>{{
            if (status==='OK') {{
              const leg=result.routes[0].legs[0];
              entry.data[hour][code][mode] = Math.round(leg.duration.value/60);
            }} else {{
              entry.data[hour][code][mode] = null;
            }}
            resolve();
          }});
        }});
        await new Promise(r=>setTimeout(r,200)); // rate limit
      }}
    }}

    prog.textContent=` ${{entry.name}} — hour ${{hour+1}}/24 complete...`;
  }}

  prog.textContent=`✓ ${{entry.name}} complete!`;
  renderCompareTable();
}}

function renderCompareTable() {{
  if (compareResults.length === 0) return;

  const hours = Array.from({{length:24}}, (_,i)=>i);

  // Build header
  let thead = '<tr><th>Time</th>';
  compareResults.forEach(entry => {{
    Object.keys(AIRPORTS).forEach(code => {{
      thead += `<th><span class="loc-pin" style="background:${{entry.colour}}"></span>${{entry.name.split('(')[0].trim()}} → ${{code}} 🚗</th>`;
      thead += `<th><span class="loc-pin" style="background:${{entry.colour}}"></span>${{entry.name.split('(')[0].trim()}} → ${{code}} 🚆</th>`;
    }});
  }});
  thead += '</tr>';

  // Build rows
  let tbody = '';
  hours.forEach(h => {{
    const timeLabel = `${{String(h).padStart(2,'0')}}:00`;
    tbody += `<tr><td style="font-weight:bold;color:#94a3b8">${{timeLabel}}</td>`;
    compareResults.forEach(entry => {{
      Object.keys(AIRPORTS).forEach(code => {{
        const d = entry.data[h]?.[code];
        const drive   = d?.DRIVING != null ? `${{d.DRIVING}} min` : '—';
        const transit = d?.TRANSIT != null ? `${{d.TRANSIT}} min` : '—';
        tbody += `<td class="drive">${{drive}}</td>`;
        tbody += `<td class="transit">${{transit}}</td>`;
      }});
    }});
    tbody += '</tr>';
  }});

  document.getElementById('compare-body').innerHTML =
    `<table id="compare-table"><thead>${{thead}}</thead><tbody>${{tbody}}</tbody></table>`;
}}

// ── AIRCRAFT POPUP ────────────────────────────────────────
function openAircraft() {{
  document.getElementById('aircraft-overlay').classList.add('open');
  fetchAircraftData();
}}

function closeAircraft() {{
  document.getElementById('aircraft-overlay').classList.remove('open');
}}

async function fetchAircraftData() {{
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px"> Loading live aircraft data...</div>';

  let html = '';

  for (const [code, ap] of Object.entries(AIRPORTS)) {{
    try {{
      const bb = ap.bbox;
      const url = `https://opensky-network.org/api/states/all?lamin=${{bb.lamin}}&lomin=${{bb.lomin}}&lamax=${{bb.lamax}}&lomax=${{bb.lomax}}`;
      const res = await fetch(url);
      const data = await res.json();
      const states = data.states || [];

      html += `<div class="aircraft-airport">`;
      html += `<h3 style="color:${{ap.colour}}">✈ ${{ap.name}} (${{code}}) — ${{states.length}} aircraft detected</h3>`;

      if (states.length === 0) {{
        html += '<p style="color:#64748b;font-size:11px">No aircraft detected in this area right now.</p>';
      }} else {{
        html += `<table class="aircraft-table">
          <tr><th>Callsign</th><th>Origin</th><th>Altitude (m)</th><th>Speed (m/s)</th><th>Status</th></tr>`;
        states.slice(0, 15).forEach(s => {{
          const callsign = s[1]?.trim() || 'Unknown';
          const origin   = s[2] || 'Unknown';
          const altitude = s[7] != null ? Math.round(s[7]) : '—';
          const speed    = s[9] != null ? Math.round(s[9]) : '—';
          const onGround = s[8];
          const status   = onGround
            ? '<span class="status-ground">🟢 On Ground</span>'
            : '<span class="status-air">🟡 Airborne</span>';
          html += `<tr><td>${{callsign}}</td><td>${{origin}}</td><td>${{altitude}}</td><td>${{speed}}</td><td>${{status}}</td></tr>`;
        }});
        html += '</table>';
        if (states.length > 15) html += `<p style="color:#64748b;font-size:10px;margin-top:6px">Showing 15 of ${{states.length}} aircraft</p>`;
      }}
      html += '</div>';
    }} catch(e) {{
      html += `<div class="aircraft-airport"><h3 style="color:${{ap.colour}}">✈ ${{ap.name}} (${{code}})</h3><p style="color:#ef4444;font-size:11px">Could not load data — OpenSky may be rate limiting. Try again in a moment.</p></div>`;
    }}
  }}

  body.innerHTML = html || '<p style="color:#64748b">No data available.</p>';
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_FINAL.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print('  Saved: airport_journey_planner_FINAL.html')
print('  Features:')
print('  → Catchment zone toggles (30/45/60 per airport)')
print('  → 24h comparison popup (click up to 3 locations)')
print('  → Live aircraft data (OpenSky Network)')
print('  → To/From routing')
print('  → ITL3 dropdown + centroids')
print('  → Straight line distances')
print('  → MultiPolygon ITL3 regions fixed')

Map works well, but the aviation data does not. switching to aviation stack for the neccesary data and API key


In [ ]:
import requests

key = 'affba01de2df68e89abe5d15b2b83a3c'

# Test AviationStack for Heathrow flights
url = f'http://api.aviationstack.com/v1/flights'
params = {
    'access_key': key,
    'arr_iata': 'LHR',
    'limit': 5
}

response = requests.get(url, params=params)
print(f'Status: {response.status_code}')
if response.status_code == 200:
    data = response.json()
    print(f' Flights found: {len(data.get("data", []))}')
    if data.get("data"):
        f = data["data"][0]
        print(f'  Example: {f.get("flight", {}).get("iata")} — {f.get("flight_status")}')
else:
    print(f' Error: {response.text[:200]}')

testing worked - adding it to the map

In [ ]:
# Read the existing FINAL HTML
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

new_aircraft_func = """async function fetchAircraftData() {
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px"> Loading live aircraft data...</div>';

  let html = '';

  for (const [code, ap] of Object.entries(AIRPORTS)) {
    try {
      const bb  = ap.bbox;
      const url = `https://opensky-network.org/api/states/all?lamin=${bb.lamin}&lomin=${bb.lomin}&lamax=${bb.lamax}&lomax=${bb.lomax}`;
      const res  = await fetch(url, {mode:'cors'});
      const data = await res.json();
      const states = data.states || [];

      html += `<div class="aircraft-airport">`;
      html += `<h3 style="color:${ap.colour}">✈ ${ap.name} (${code}) — ${states.length} aircraft detected</h3>`;

      if (states.length === 0) {
        html += '<p style="color:#64748b;font-size:11px">No aircraft detected in this area right now.</p>';
      } else {
        html += `<table class="aircraft-table">
          <tr><th>Callsign</th><th>Country</th><th>Altitude (m)</th><th>Speed (m/s)</th><th>Status</th></tr>`;
        states.slice(0,15).forEach(s => {
          const callsign = s[1]?.trim() || 'Unknown';
          const country  = s[2] || 'Unknown';
          const altitude = s[7] != null ? Math.round(s[7]) : '—';
          const speed    = s[9] != null ? Math.round(s[9]) : '—';
          const onGround = s[8];
          const status   = onGround
            ? '<span class="status-ground">🟢 On Ground</span>'
            : '<span class="status-air">🟡 Airborne</span>';
          html += `<tr><td>${callsign}</td><td>${country}</td><td>${altitude}</td><td>${speed}</td><td>${status}</td></tr>`;
        });
        html += '</table>';
        if (states.length > 15) html += `<p style="color:#64748b;font-size:10px;margin-top:6px">Showing 15 of ${states.length} aircraft</p>`;
      }
      html += '</div>';
    } catch(e) {
      html += `<div class="aircraft-airport">
        <h3 style="color:${ap.colour}">✈ ${ap.name} (${code})</h3>
        <p style="color:#ef4444;font-size:11px">Could not load — OpenSky may be temporarily unavailable. Try again in a moment.</p>
      </div>`;
    }
  }

  body.innerHTML = html || '<p style="color:#64748b">No data available.</p>';
}"""

import re
pattern = r'async function fetchAircraftData\(\).*?(?=\n// |\n</script>)'
new_content = re.sub(pattern, new_aircraft_func, content, flags=re.DOTALL)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Updated with OpenSky direct browser call')
print(f'opensky in file: {"opensky" in new_content}')

In [ ]:
# Test FlightRadar24 public API for Heathrow area
url = 'https://data-live.flightradar24.com/zones/fcgi/feed.js'
params = {
    'bounds': '51.51,-0.42,51.44,-0.50',  # Heathrow bounding box
    'faa': '1',
    'satellite': '1',
    'mlat': '1',
    'flarm': '1',
    'adsb': '1',
    'gnd': '1',
    'air': '1',
    'vehicles': '0',
    'estimated': '0',
    'gliders': '0',
    'stats': '0',
}
headers = {
    'User-Agent': 'Mozilla/5.0',
    'Origin': 'https://www.flightradar24.com',
}

response = requests.get(url, params=params, headers=headers)
print(f'Status: {response.status_code}')
if response.status_code == 200:
    data = response.json()
    # Remove metadata keys
    flights = {k:v for k,v in data.items() if k not in ['full_count','version','stats']}
    print(f'✓ Aircraft near Heathrow: {len(flights)}')
    if flights:
        first = list(flights.values())[0]
        print(f'  Example: {first}')

changed the airport data with aviation stack

In [ ]:
import requests
import json
from datetime import datetime

KEY = 'affba01de2df68e89abe5d15b2b83a3c'

flight_data = {}

for code in ['LHR', 'LGW', 'LTN']:
    print(f'Fetching flights for {code}...')
    url = 'https://api.aviationstack.com/v1/flights'
    params = {
        'access_key': KEY,
        'arr_iata': code,
        'limit': 15
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        flights = data.get('data', [])
        flight_data[code] = [{
            'flight':   f.get('flight', {}).get('iata', '—'),
            'from':     f.get('departure', {}).get('iata', '—'),
            'from_name': f.get('departure', {}).get('airport', '—'),
            'aircraft': f.get('aircraft', {}).get('iata', '—') if f.get('aircraft') else '—',
            'status':   f.get('flight_status', '—'),
            'arrival':  f.get('arrival', {}).get('estimated', '—'),
        } for f in flights]
        print(f'  ✓ {len(flights)} flights fetched')
    else:
        flight_data[code] = []
        print(f'Error: {response.status_code}')

# Add timestamp
fetch_time = datetime.now().strftime('%d %b %Y %H:%M')
print(f'\n All flights fetched at {fetch_time}')
print(json.dumps(flight_data, indent=2)[:500])

In [ ]:
import requests
import json
from datetime import datetime

KEY = 'affba01de2df68e89abe5d15b2b83a3c'

flight_data = {}

for code in ['LHR', 'LGW', 'LTN']:
    print(f'Fetching flights for {code}...')
    flight_data[code] = {'arrivals': [], 'departures': []}

    # Arrivals
    arr_response = requests.get('https://api.aviationstack.com/v1/flights', params={
        'access_key': KEY, 'arr_iata': code, 'limit': 10
    })
    if arr_response.status_code == 200:
        for f in arr_response.json().get('data', []):
            flight_data[code]['arrivals'].append({
                'flight':    f.get('flight', {}).get('iata', 'N/A') or 'N/A',
                'from':      f.get('departure', {}).get('iata', 'N/A') or 'N/A',
                'from_name': f.get('departure', {}).get('airport', 'N/A') or 'N/A',
                'status':    f.get('flight_status', 'N/A') or 'N/A',
            })
        print(f'Arrivals: {len(flight_data[code]["arrivals"])}')

    # Departures
    dep_response = requests.get('https://api.aviationstack.com/v1/flights', params={
        'access_key': KEY, 'dep_iata': code, 'limit': 10
    })
    if dep_response.status_code == 200:
        for f in dep_response.json().get('data', []):
            flight_data[code]['departures'].append({
                'flight':  f.get('flight', {}).get('iata', 'N/A') or 'N/A',
                'to':      f.get('arrival', {}).get('iata', 'N/A') or 'N/A',
                'to_name': f.get('arrival', {}).get('airport', 'N/A') or 'N/A',
                'status':  f.get('flight_status', 'N/A') or 'N/A',
            })
        print(f'Departures: {len(flight_data[code]["departures"])}')

fetch_time = datetime.now().strftime('%d %b %Y %H:%M')
print(f'\n Done at {fetch_time}')

In [ ]:
import json

# Read existing HTML
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

flight_data_str = json.dumps(flight_data, ensure_ascii=True)

new_aircraft_func = (
    "async function fetchAircraftData() {\n"
    "  const body = document.getElementById('aircraft-body');\n"
    f"  const FLIGHT_DATA = {flight_data_str};\n"
    f"  const FETCH_TIME  = '{fetch_time}';\n"
    "  let html = `<p style='color:#64748b;font-size:10px;margin-bottom:12px'> Data fetched: ${{FETCH_TIME}}</p>`;\n"
    "  for (const [code, ap] of Object.entries(AIRPORTS)) {\n"
    "    const arr = FLIGHT_DATA[code]?.arrivals || [];\n"
    "    const dep = FLIGHT_DATA[code]?.departures || [];\n"
    "    html += `<div class='aircraft-airport'>`;\n"
    "    html += `<h3 style='color:${{ap.colour}}'>✈ ${{ap.name}} (${{code}})</h3>`;\n"
    "    html += `<p style='font-size:10px;color:#94a3b8;margin-bottom:6px'>🛬 ${{arr.length}} arrivals · 🛫 ${{dep.length}} departures</p>`;\n"
    "    html += '<b style=\"font-size:11px;color:#94a3b8\">🛬 Arrivals</b>';\n"
    "    html += '<table class=\"aircraft-table\"><tr><th>Flight</th><th>From</th><th>Airport</th><th>Status</th></tr>';\n"
    "    arr.forEach(f => {\n"
    "      const sc = f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';\n"
    "      html += `<tr><td style='font-weight:bold'>${{f.flight}}</td><td>${{f.from}}</td><td style='font-size:10px;color:#94a3b8'>${{f.from_name}}</td><td style='color:${{sc}}'>${{f.status}}</td></tr>`;\n"
    "    });\n"
    "    html += '</table>';\n"
    "    html += '<b style=\"font-size:11px;color:#94a3b8;display:block;margin-top:10px\">🛫 Departures</b>';\n"
    "    html += '<table class=\"aircraft-table\"><tr><th>Flight</th><th>To</th><th>Airport</th><th>Status</th></tr>';\n"
    "    dep.forEach(f => {\n"
    "      const sc = f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';\n"
    "      html += `<tr><td style='font-weight:bold'>${{f.flight}}</td><td>${{f.to}}</td><td style='font-size:10px;color:#94a3b8'>${{f.to_name}}</td><td style='color:${{sc}}'>${{f.status}}</td></tr>`;\n"
    "    });\n"
    "    html += '</table></div>';\n"
    "  }\n"
    "  body.innerHTML = html;\n"
    "}"
)

# Replace using string find
start = content.find('async function fetchAircraftData()')
end   = content.find('\n// ', start)
if end == -1:
    end = content.find('\n</script>', start)

new_content = content[:start] + new_aircraft_func + content[end:]

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Updated with arrivals and departures')
print(f'  LHR: {len(flight_data["LHR"]["arrivals"])} arrivals, {len(flight_data["LHR"]["departures"])} departures')
print(f'  LGW: {len(flight_data["LGW"]["arrivals"])} arrivals, {len(flight_data["LGW"]["departures"])} departures')
print(f'  LTN: {len(flight_data["LTN"]["arrivals"])} arrivals, {len(flight_data["LTN"]["departures"])} departures')

In [ ]:
# Check if the HTML file is valid
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Check key markers
print(f'Has initMap: {"function initMap()" in content}')
print(f'Has loadMap: {"function loadMap()" in content}')
print(f'Has fetchAircraftData: {"function fetchAircraftData()" in content}')
print(f'Has closing script tag: {"</script>" in content}')
print(f'File size: {len(content)} chars')

# Show the area around fetchAircraftData to spot issues
idx = content.find('async function fetchAircraftData()')
print(f'\nAround fetchAircraftData (first 200 chars):')
print(content[idx:idx+200])

file is too big. trying to limit it


In [ ]:
import json

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Slim down flight data — keep only 5 flights per direction per airport
slim_data = {}
for code in ['LHR', 'LGW', 'LTN']:
    slim_data[code] = {
        'arrivals': flight_data[code]['arrivals'][:5],
        'departures': flight_data[code]['departures'][:5]
    }

flight_data_str = json.dumps(slim_data, ensure_ascii=True)
print(f'Slim data size: {len(flight_data_str)} chars')

new_aircraft_func = (
    "async function fetchAircraftData() {\n"
    "  const body = document.getElementById('aircraft-body');\n"
    f"  const FLIGHT_DATA = {flight_data_str};\n"
    f"  const FETCH_TIME  = '{fetch_time}';\n"
    "  let html = `<p style='color:#64748b;font-size:10px;margin-bottom:12px'>Data fetched: ${FETCH_TIME}</p>`;\n"
    "  for (const [code, ap] of Object.entries(AIRPORTS)) {\n"
    "    const arr = FLIGHT_DATA[code]?.arrivals || [];\n"
    "    const dep = FLIGHT_DATA[code]?.departures || [];\n"
    "    html += `<div class='aircraft-airport'><h3 style='color:${ap.colour}'>✈ ${ap.name} (${code})</h3>`;\n"
    "    html += '<b style=\"font-size:11px;color:#94a3b8\">🛬 Arrivals</b>';\n"
    "    html += '<table class=\"aircraft-table\"><tr><th>Flight</th><th>From</th><th>Status</th></tr>';\n"
    "    arr.forEach(f => {\n"
    "      const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';\n"
    "      html+=`<tr><td style='font-weight:bold'>${f.flight}</td><td>${f.from}</td><td style='color:${sc}'>${f.status}</td></tr>`;\n"
    "    });\n"
    "    html += '</table><b style=\"font-size:11px;color:#94a3b8;display:block;margin-top:8px\">🛫 Departures</b>';\n"
    "    html += '<table class=\"aircraft-table\"><tr><th>Flight</th><th>To</th><th>Status</th></tr>';\n"
    "    dep.forEach(f => {\n"
    "      const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';\n"
    "      html+=`<tr><td style='font-weight:bold'>${f.flight}</td><td>${f.to}</td><td style='color:${sc}'>${f.status}</td></tr>`;\n"
    "    });\n"
    "    html += '</table></div>';\n"
    "  }\n"
    "  body.innerHTML = html;\n"
    "}"
)

# Replace
start = content.find('async function fetchAircraftData()')
end   = content.find('\n// ', start)
if end == -1:
    end = content.find('\n</script>', start)

new_content = content[:start] + new_aircraft_func + content[end:]

print(f'New file size: {len(new_content)} chars')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Fixed —> slim flight data embedded')

In [ ]:
# Find all occurrences of fetchAircraftData
count = content.count('async function fetchAircraftData()')
print(f'Found {count} occurrences of fetchAircraftData')

# Find where the FIRST one starts and the LAST one ends
first_start = content.find('async function fetchAircraftData()')
last_start  = content.rfind('async function fetchAircraftData()')

# Find end of last occurrence
end = content.find('\n// ', last_start)
if end == -1:
    end = content.find('\n</script>', last_start)

print(f'First at position: {first_start}')
print(f'Last at position:  {last_start}')
print(f'End at position:   {end}')
print(f'Removing {end - first_start} chars of old functions')

# Replace everything from first to end with new slim function
new_content = content[:first_start] + new_aircraft_func + content[end:]

print(f'New file size: {len(new_content)} chars')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Fixed')

In [ ]:
# Check size of each data component
print(f'iso_geojson size:   {len(json.dumps(iso_geojson)):,} chars')
print(f'itl3_json size:     {len(json.dumps(itl3_json)):,} chars')
print(f'centroids_json size:{len(centroids_json):,} chars')
print(f'flight_data size:   {len(json.dumps(slim_data)):,} chars')

In [ ]:
import geopandas as gpd
import json

# Simplify ITL3 geometries — reduce coordinate precision
itl3_simplified = itl3_catchment_filtered.copy()
itl3_simplified = itl3_simplified.to_crs('EPSG:27700')  # project to metres
itl3_simplified['geometry'] = itl3_simplified.geometry.simplify(
    tolerance=200,  # 200 metres tolerance
    preserve_topology=True
)
itl3_simplified = itl3_simplified.to_crs('EPSG:4326')  # back to WGS84

# Export
itl3_export = itl3_simplified[['ITL321NM', 'population_2024', 'GDHI_2023_million_gbp', 'geometry']].copy()
itl3_json   = json.loads(itl3_export.to_json())

print(f'Original size: 16,606,327 chars')
print(f'New size:      {len(json.dumps(itl3_json)):,} chars')
print(f'Regions:       {len(itl3_json["features"])}')

In [ ]:
# Recalculate centroids from simplified geometry
centroids = []
for _, row in itl3_simplified.iterrows():
    centroid = row.geometry.centroid
    pop  = int(row['population_2024']) if pd.notna(row['population_2024']) else 0
    gdhi = round(float(row['GDHI_2023_million_gbp']), 1) if pd.notna(row['GDHI_2023_million_gbp']) else 0
    centroids.append({
        'name': row['ITL321NM'],
        'lat':  round(centroid.y, 5),
        'lng':  round(centroid.x, 5),
        'pop':  pop,
        'gdhi': gdhi
    })

centroids_json = json.dumps(centroids)
print(f'{len(centroids)} centroids recalculated')

# Verify all data sizes
print(f'iso_geojson:    {len(json.dumps(iso_geojson)):,} chars')
print(f'itl3_json:      {len(json.dumps(itl3_json)):,} chars')
print(f'centroids_json: {len(centroids_json):,} chars')
print(f'flight_data:    {len(json.dumps(slim_data)):,} chars')

In [ ]:
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airprt Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; overflow:hidden; }}
  #header {{ background:#1a1a2e; color:white; padding:8px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:13px; font-weight:bold; }}
  #api-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; }}
  #api-bar button:hover {{ background:#2563eb; }}
  #control-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:8px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #control-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .mode-btn {{ padding:4px 10px; font-size:11px; border-radius:3px; cursor:pointer; border:1px solid #2a3a50; background:#1e2d45; color:#94a3b8; }}
  .mode-btn.active {{ background:#3b82f6; color:white; border-color:#3b82f6; }}
  #itl3-select {{ background:#1e2d45; border:1px solid #2a3a50; color:#e2e8f0; padding:4px 8px; font-size:11px; border-radius:3px; max-width:190px; }}
  #toggle-bar {{ background:#080d1a; padding:5px 20px; display:flex; align-items:center; gap:16px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #toggle-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .airport-group {{ display:flex; align-items:center; gap:5px; }}
  .airport-group-label {{ font-size:11px; font-weight:bold; margin-right:2px; }}
  .tog {{ padding:2px 8px; font-size:10px; border-radius:3px; cursor:pointer; border:1px solid; opacity:0.35; transition:opacity 0.15s; }}
  .tog.on {{ opacity:1; }}
  button.compare-btn {{ background:#7c3aed; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; margin-left:auto; }}
  button.compare-btn:hover {{ background:#6d28d9; }}
  button.aircraft-btn {{ background:#0891b2; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; }}
  button.aircraft-btn:hover {{ background:#0e7490; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}
  #sidebar {{ width:320px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:8px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:8px; }}
  .airport-block {{ margin-bottom:8px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:7px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; }}
  .dot {{ width:7px; height:7px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:6px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .straight-row {{ padding:4px 12px; background:#0d1525; border-bottom:1px solid #1e2d45; font-size:10px; color:#64748b; display:flex; justify-content:space-between; }}
  .route-left {{ display:flex; align-items:center; gap:7px; }}
  .route-icon {{ width:18px; height:18px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:10px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:10px; height:10px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:24px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:4px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:6px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:6px; }}
  .leg {{ display:flex; align-items:center; gap:3px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:14px; height:2px; border-radius:1px; }}
  #loading-bar {{ padding:5px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
  #compare-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #compare-overlay.open {{ display:flex; }}
  #compare-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:90vw; max-width:1100px; max-height:85vh; display:flex; flex-direction:column; overflow:hidden; }}
  #compare-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #compare-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #compare-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #compare-controls {{ padding:10px 20px; border-bottom:1px solid #1e2d45; display:flex; align-items:center; gap:10px; flex-wrap:wrap; }}
  #compare-controls span {{ font-size:11px; color:#64748b; }}
  #compare-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  #compare-table {{ width:100%; border-collapse:collapse; font-size:11px; }}
  #compare-table th {{ background:#1e2d45; color:#94a3b8; padding:8px 10px; text-align:left; border:1px solid #2a3a50; white-space:nowrap; position:sticky; top:0; }}
  #compare-table td {{ padding:6px 10px; border:1px solid #1e2d45; color:#e2e8f0; white-space:nowrap; }}
  #compare-table tr:nth-child(even) td {{ background:#0d1525; }}
  #compare-table td.drive {{ color:#ff8800; }}
  #compare-table td.transit {{ color:#00ccff; }}
  #compare-progress {{ padding:8px 20px; border-top:1px solid #1e2d45; font-size:10px; color:#f59e0b; display:none; }}
  #aircraft-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #aircraft-overlay.open {{ display:flex; }}
  #aircraft-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:700px; max-height:80vh; display:flex; flex-direction:column; overflow:hidden; }}
  #aircraft-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #aircraft-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #aircraft-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #aircraft-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  .aircraft-airport {{ margin-bottom:16px; }}
  .aircraft-airport h3 {{ font-size:12px; font-weight:bold; margin-bottom:6px; padding-bottom:4px; border-bottom:1px solid #1e2d45; }}
  .aircraft-table {{ width:100%; border-collapse:collapse; font-size:11px; margin-bottom:8px; }}
  .aircraft-table th {{ background:#1e2d45; color:#94a3b8; padding:6px 8px; text-align:left; border:1px solid #2a3a50; }}
  .aircraft-table td {{ padding:5px 8px; border:1px solid #1e2d45; color:#e2e8f0; }}
  .aircraft-table tr:nth-child(even) td {{ background:#0d1525; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="control-bar">
  <label>Direction:</label>
  <button class="mode-btn active" id="btn-to"   onclick="setMode('to')">📍→✈ To Airport</button>
  <button class="mode-btn"        id="btn-from" onclick="setMode('from')">✈→📍 From Airport</button>
  <label style="margin-left:8px">ITL3:</label>
  <select id="itl3-select" onchange="selectITL3()">
    <option value="">— pick a region —</option>
  </select>
</div>

<div id="toggle-bar">
  <label>Catchment:</label>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#3b82f6">LHR</span>
    <button class="tog on" id="tog-LHR-30" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',30)">30</button>
    <button class="tog on" id="tog-LHR-45" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',45)">45</button>
    <button class="tog on" id="tog-LHR-60" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#ef4444">LGW</span>
    <button class="tog on" id="tog-LGW-30" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',30)">30</button>
    <button class="tog on" id="tog-LGW-45" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',45)">45</button>
    <button class="tog on" id="tog-LGW-60" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#22c55e">LTN</span>
    <button class="tog on" id="tog-LTN-30" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',30)">30</button>
    <button class="tog on" id="tog-LTN-45" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',45)">45</button>
    <button class="tog on" id="tog-LTN-60" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',60)">60</button>
  </div>
  <button class="aircraft-btn" onclick="openAircraft()">✈ Flight Info</button>
  <button class="compare-btn"  onclick="openCompare()">📊 Compare 24h</button>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">Click anywhere or pick an ITL3 region.<br>Toggle catchment zones above.<br>Use Compare for 24h analysis.</div>
    <div id="results"><div id="empty">Load the map then click anywhere.</div></div>
    <div id="loading-bar"> Calculating routes...</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<!-- Compare popup -->
<div id="compare-overlay">
  <div id="compare-panel">
    <div id="compare-header">
      <h2>📊 24h Journey Time Comparison</h2>
      <button onclick="closeCompare()">✕</button>
    </div>
    <div id="compare-controls">
      <span>Close panel and click up to 3 locations on the map to compare.</span>
      <button class="mode-btn" onclick="clearCompare()" style="margin-left:auto">Clear All</button>
    </div>
    <div id="compare-progress"></div>
    <div id="compare-body">
      <div style="color:#64748b;font-size:11px;text-align:center;padding:40px">
        Close this panel and click locations on the map.
      </div>
    </div>
  </div>
</div>

<!-- Aircraft popup -->
<div id="aircraft-overlay">
  <div id="aircraft-panel">
    <div id="aircraft-header">
      <h2>✈ Flight Information</h2>
      <button onclick="closeAircraft()">✕</button>
    </div>
    <div id="aircraft-body"></div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6',
          bbox:{{ lamin:51.44, lomin:-0.50, lamax:51.51, lomax:-0.42 }} }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444',
          bbox:{{ lamin:51.13, lomin:-0.22, lamax:51.18, lomax:-0.14 }} }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e',
          bbox:{{ lamin:51.86, lomin:-0.42, lamax:51.91, lomax:-0.32 }} }},
}};

const ROUTE_COLOURS = {{ DRIVING:'#ff8800', TRANSIT:'#00ccff' }};
const ISO_DATA      = {json.dumps(iso_geojson)};
const ITL3_DATA     = {json.dumps(itl3_json)};
const CENTROIDS     = {centroids_json};
const FLIGHT_DATA   = {flight_data_str};
const FETCH_TIME    = '{fetch_time}';
const TIMES         = [30, 45, 60];
const ISO_WEIGHTS   = [3, 2, 1.5];

let map, directionsService;
let startMarker=null, renderers=[], straightLines=[];
let isoPolygons={{}};
let mapLoaded=false, currentMode='to';
let compareMode=false, comparePoints=[], compareMarkers=[], compareResults=[];
const PIN_COLOURS=['#f59e0b','#a855f7','#ec4899'];

// Populate ITL3 dropdown
window.addEventListener('load', () => {{
  const sel = document.getElementById('itl3-select');
  CENTROIDS.sort((a,b)=>a.name.localeCompare(b.name)).forEach(c => {{
    const opt = document.createElement('option');
    opt.value = JSON.stringify({{lat:c.lat, lng:c.lng, name:c.name}});
    opt.textContent = c.name;
    sel.appendChild(opt);
  }});
}});

function selectITL3() {{
  const val = document.getElementById('itl3-select').value;
  if (!val || !map) return;
  const {{lat,lng}} = JSON.parse(val);
  handlePoint(lat, lng);
}}

function setMode(mode) {{
  currentMode = mode;
  document.getElementById('btn-to').className   = 'mode-btn'+(mode==='to'?' active':'');
  document.getElementById('btn-from').className = 'mode-btn'+(mode==='from'?' active':'');
  clearRoutes();
  if (startMarker) {{ startMarker.setMap(null); startMarker=null; }}
  document.getElementById('results').innerHTML  = '<div id="empty">Click anywhere to begin.</div>';
  document.getElementById('coords').textContent = 'Coordinates: —';
}}

function toggleIso(iata, mins) {{
  const btn  = document.getElementById(`tog-${{iata}}-${{mins}}`);
  const polys = isoPolygons[iata]?.[mins] || [];
  const isOn  = btn.classList.contains('on');
  polys.forEach(p => p.setMap(isOn ? null : map));
  btn.classList.toggle('on', !isOn);
  btn.style.opacity = isOn ? '0.35' : '1';
}}

function haversine(lat1,lng1,lat2,lng2) {{
  const R=6371, dLat=(lat2-lat1)*Math.PI/180, dLng=(lng2-lng1)*Math.PI/180;
  const a=Math.sin(dLat/2)**2+Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLng/2)**2;
  return (R*2*Math.atan2(Math.sqrt(a),Math.sqrt(1-a))).toFixed(1);
}}

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display='none';
  document.getElementById('map').style.display='block';
  const script=document.createElement('script');
  script.src=`https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async=true; script.defer=true;
  document.head.appendChild(script);
  mapLoaded=true;
}}

function coordsToPath(coords) {{ return coords.map(c=>({{lat:c[1],lng:c[0]}})); }}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center:{{lat:51.5,lng:-0.3}}, zoom:8, mapTypeId:'roadmap',
    styles:[
      {{elementType:'geometry',stylers:[{{color:'#1a1a2e'}}]}},
      {{elementType:'labels.text.fill',stylers:[{{color:'#94a3b8'}}]}},
      {{elementType:'labels.text.stroke',stylers:[{{color:'#1a1a2e'}}]}},
      {{featureType:'road',elementType:'geometry',stylers:[{{color:'#2d3748'}}]}},
      {{featureType:'road.highway',elementType:'geometry',stylers:[{{color:'#2d4a7a'}}]}},
      {{featureType:'water',elementType:'geometry',stylers:[{{color:'#0d1525'}}]}},
      {{featureType:'transit.line',elementType:'geometry',stylers:[{{color:'#4a5568'}}]}},
      {{featureType:'transit.station',elementType:'geometry',stylers:[{{color:'#374151'}}]}},
      {{featureType:'poi',stylers:[{{visibility:'off'}}]}},
    ]
  }});

  directionsService = new google.maps.DirectionsService();

  // ITL3 choropleth
  const pops=ITL3_DATA.features.map(f=>f.properties.population_2024||0);
  const minPop=Math.min(...pops), maxPop=Math.max(...pops);
  function popColour(pop) {{
    if (!pop) return '#374151';
    const t=(pop-minPop)/(maxPop-minPop);
    return `rgb(${{Math.round(255*Math.min(1,t*2))}},${{Math.round(255*Math.max(0,1-t*2))}},50)`;
  }}

  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const colour = popColour(f.properties.population_2024);
    const rings  = f.geometry.type==='MultiPolygon'
      ? f.geometry.coordinates.map(p=>p[0])
      : [f.geometry.coordinates[0]];
    rings.forEach(ring => {{
      new google.maps.Polygon({{
        paths:coordsToPath(ring), strokeColor:'#374151', strokeWeight:0.5,
        fillColor:colour, fillOpacity:0.5, clickable:false, map
      }});
    }});
  }});

  // ITL3 centroid markers
  CENTROIDS.forEach(c => {{
    const marker = new google.maps.Marker({{
      position:{{lat:c.lat,lng:c.lng}}, map, title:c.name,
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:5,fillColor:'#fff',fillOpacity:0.6,strokeColor:'#94a3b8',strokeWeight:1}}
    }});
    const iw = new google.maps.InfoWindow({{
      content:`<div style="font-size:11px"><b>${{c.name}}</b><br>Pop: ${{c.pop.toLocaleString()}}<br>GDHI: £${{c.gdhi.toLocaleString()}}M</div>`
    }});
    marker.addListener('click',()=>{{ iw.open(map,marker); handlePoint(c.lat,c.lng); }});
  }});

  // Isochrones
  const isoColours={{LHR:'#3b82f6',LGW:'#ef4444',LTN:'#22c55e'}};
  Object.entries(ISO_DATA).forEach(([iata,geojson]) => {{
    isoPolygons[iata]={{}};
    geojson.features.forEach((feature,i) => {{
      const mins=TIMES[i];
      const poly=new google.maps.Polygon({{
        paths:coordsToPath(feature.geometry.coordinates[0]),
        strokeColor:isoColours[iata], strokeWeight:ISO_WEIGHTS[i],
        strokeOpacity:0.9, fillOpacity:0, clickable:false, map
      }});
      if (!isoPolygons[iata][mins]) isoPolygons[iata][mins]=[];
      isoPolygons[iata][mins].push(poly);
    }});
  }});

  // Airport markers
  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    const marker=new google.maps.Marker({{
      position:{{lat:ap.lat,lng:ap.lng}}, map, title:ap.name,
      label:{{text:'✈',color:'white',fontSize:'11px'}},
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:13,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}}
    }});
    const iw=new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click',()=>iw.open(map,marker));
  }});

  // Click handler
  map.addListener('click', e => {{
    const lat=e.latLng.lat(), lng=e.latLng.lng();
    if (compareMode) {{ addComparePoint(lat,lng); }}
    else {{ handlePoint(lat,lng); }}
  }});

  document.getElementById('empty').textContent='Click anywhere or pick an ITL3 region.';
}}

function clearRoutes() {{
  renderers.forEach(r=>r.setMap(null)); renderers=[];
  straightLines.forEach(l=>l.setMap(null)); straightLines=[];
}}

function handlePoint(lat,lng) {{
  document.getElementById('coords').textContent=
    `${{currentMode==='to'?'From':'To'}}: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
  if (startMarker) startMarker.setMap(null);
  startMarker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:9,
      fillColor:currentMode==='to'?'#ff8800':'#ff00ff',
      fillOpacity:1,strokeColor:'white',strokeWeight:2}}
  }});
  clearRoutes();
  calculateAllRoutes(lat,lng);
}}

function calculateAllRoutes(lat,lng) {{
  const lb=document.getElementById('loading-bar');
  lb.style.display='block'; lb.style.color='#f59e0b'; lb.textContent='Calculating routes...';

  const straight={{}};
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{ straight[code]=haversine(lat,lng,ap.lat,ap.lng); }});

  const modeLabel=currentMode==='to'?'→ ✈':'✈ →';
  document.getElementById('results').innerHTML=Object.entries(AIRPORTS).map(([code,ap])=>`
    <div class="airport-block">
      <div class="airport-header"><div class="dot" style="background:${{ap.colour}}"></div>${{modeLabel}} ${{ap.name}} (${{code}})</div>
      <div class="straight-row"><span> Straight line</span><span style="color:#e2e8f0">${{straight[code]}} km</span></div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left"><div class="route-icon" style="background:#ff880033">🚗</div><div class="route-label">Drive</div></div>
        <div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left"><div class="route-icon" style="background:#00ccff33">🚆</div><div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div></div>
        <div class="spinner"></div>
      </div>
    </div>`).join('');

  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    const line=new google.maps.Polyline({{
      path:[{{lat,lng}},{{lat:ap.lat,lng:ap.lng}}],
      strokeColor:ap.colour,strokeWeight:1,strokeOpacity:0.5,
      icons:[{{icon:{{path:'M 0,-1 0,1',strokeOpacity:1,scale:3}},offset:'0',repeat:'12px'}}],map
    }});
    straightLines.push(line);
  }});

  let completed=0;
  const total=Object.keys(AIRPORTS).length*2;

  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    ['DRIVING','TRANSIT'].forEach(mode=>{{
      const renderer=new google.maps.DirectionsRenderer({{
        suppressMarkers:true,
        polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeWeight:mode==='DRIVING'?5:4,strokeOpacity:0.85}}
      }});
      renderers.push(renderer);

      const origin     =currentMode==='to'?{{lat,lng}}:{{lat:ap.lat,lng:ap.lng}};
      const destination=currentMode==='to'?{{lat:ap.lat,lng:ap.lng}}:{{lat,lng}};

      directionsService.route({{
        origin,destination,travelMode:google.maps.TravelMode[mode],
        ...(mode==='TRANSIT'?{{transitOptions:{{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}}}:{{}})
      }},(result,status)=>{{
        completed++;
        const el=document.getElementById(`r-${{code}}-${{mode}}`);
        if (status==='OK') {{
          renderer.setMap(map); renderer.setDirections(result);
          const leg=result.routes[0].legs[0];
          const mins=Math.round(leg.duration.value/60);
          const km=(leg.distance.value/1000).toFixed(1);
          if (el) {{
            el.querySelector('.spinner').outerHTML=`<div class="route-val done">${{mins}} min · ${{km}} km</div>`;
            if (mode==='TRANSIT') {{
              const steps=result.routes[0].legs[0].steps
                .filter(s=>s.travel_mode==='TRANSIT')
                .map(s=>s.transit?.line?.short_name||s.transit?.line?.name||'')
                .filter(Boolean).join(' → ');
              if (steps) {{ const sub=el.querySelector('.route-sub'); if(sub) sub.textContent=steps; }}
            }}
            el.onclick=()=>{{
              renderers.forEach(r=>r.setOptions({{polylineOptions:{{strokeOpacity:0.15,strokeWeight:2}}}}));
              renderer.setOptions({{polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeOpacity:1,strokeWeight:6}}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) el.querySelector('.spinner').outerHTML=`<div class="route-val" style="color:#64748b">not available</div>`;
        }}
        if (completed===total) {{
          lb.textContent='✓ Routes loaded — click a row to highlight'; lb.style.color='#22c55e';
        }}
      }});
    }});
  }});
}}

// ── COMPARISON ────────────────────────────────────────────
function openCompare() {{
  compareMode=true;
  document.getElementById('compare-overlay').classList.add('open');
  document.getElementById('instruction').innerHTML='<b style="color:#a855f7">Compare mode ON</b> — click up to 3 locations';
}}

function closeCompare() {{
  compareMode=false;
  document.getElementById('compare-overlay').classList.remove('open');
  document.getElementById('instruction').innerHTML='Click anywhere or pick an ITL3 region.<br>Toggle catchment zones above.';
}}

function clearCompare() {{
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Close panel and click locations on the map.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function addComparePoint(lat,lng) {{
  if (compareResults.length>=3) {{ alert('Maximum 3 locations. Click Clear All to reset.'); return; }}
  const idx=compareResults.length;
  const colour=PIN_COLOURS[idx];
  const name=`Location ${{idx+1}} (${{lat.toFixed(3)}}, ${{lng.toFixed(3)}})`;
  const marker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},
    label:{{text:`${{idx+1}}`,color:'white',fontSize:'10px'}}
  }});
  compareMarkers.push(marker);
  const entry={{lat,lng,name,colour,data:{{}}}};
  compareResults.push(entry);
  fetchCompareData(entry,idx);
}}

async function fetchCompareData(entry,idx) {{
  const prog=document.getElementById('compare-progress');
  prog.style.display='block';
  prog.textContent=`Calculating 24h routes for ${{entry.name}}...`;

  const hours=Array.from({{length:24}},(_,i)=>i);
  entry.data={{}};

  for (const hour of hours) {{
    const depTime=new Date();
    depTime.setHours(hour,0,0,0);
    if (depTime<new Date()) depTime.setDate(depTime.getDate()+1);
    entry.data[hour]={{}};

    for (const [code,ap] of Object.entries(AIRPORTS)) {{
      entry.data[hour][code]={{}};
      for (const mode of ['DRIVING','TRANSIT']) {{
        await new Promise(resolve=>{{
          directionsService.route({{
            origin:{{lat:entry.lat,lng:entry.lng}},
            destination:{{lat:ap.lat,lng:ap.lng}},
            travelMode:google.maps.TravelMode[mode],
            ...(mode==='TRANSIT'?{{transitOptions:{{
              departureTime:depTime,
              modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
              routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
            }}}}:{{}})
          }},(result,status)=>{{
            entry.data[hour][code][mode]=status==='OK'?Math.round(result.routes[0].legs[0].duration.value/60):null;
            resolve();
          }});
        }});
        await new Promise(r=>setTimeout(r,200));
      }}
    }}
    prog.textContent=`${{entry.name}} — hour ${{hour+1}}/24...`;
  }}

  prog.textContent=`✓ ${{entry.name}} complete!`;
  renderCompareTable();
}}

function renderCompareTable() {{
  if (compareResults.length===0) return;
  const hours=Array.from({{length:24}},(_,i)=>i);
  let thead='<tr><th>Time</th>';
  compareResults.forEach(entry=>{{
    Object.keys(AIRPORTS).forEach(code=>{{
      thead+=`<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px"></span>${{entry.name.split('(')[0].trim()}}→${{code}} 🚗</th>`;
      thead+=`<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px"></span>${{entry.name.split('(')[0].trim()}}→${{code}} 🚆</th>`;
    }});
  }});
  thead+='</tr>';
  let tbody='';
  hours.forEach(h=>{{
    tbody+=`<tr><td style="font-weight:bold;color:#94a3b8">${{String(h).padStart(2,'0')}}:00</td>`;
    compareResults.forEach(entry=>{{
      Object.keys(AIRPORTS).forEach(code=>{{
        const d=entry.data[h]?.[code];
        tbody+=`<td class="drive">${{d?.DRIVING!=null?d.DRIVING+' min':'—'}}</td>`;
        tbody+=`<td class="transit">${{d?.TRANSIT!=null?d.TRANSIT+' min':'—'}}</td>`;
      }});
    }});
    tbody+='</tr>';
  }});
  document.getElementById('compare-body').innerHTML=
    `<table id="compare-table"><thead>${{thead}}</thead><tbody>${{tbody}}</tbody></table>`;
}}

// ── FLIGHT INFO ───────────────────────────────────────────
function openAircraft() {{
  document.getElementById('aircraft-overlay').classList.add('open');
  const body=document.getElementById('aircraft-body');
  let html=`<p style="color:#64748b;font-size:10px;margin-bottom:12px">📅 Scheduled flights — fetched: ${{FETCH_TIME}}</p>`;

  for (const [code,ap] of Object.entries(AIRPORTS)) {{
    const arr=FLIGHT_DATA[code]?.arrivals||[];
    const dep=FLIGHT_DATA[code]?.departures||[];
    html+=`<div class="aircraft-airport">`;
    html+=`<h3 style="color:${{ap.colour}}">✈ ${{ap.name}} (${{code}}) — ${{arr.length}} arrivals · ${{dep.length}} departures</h3>`;
    html+='<b style="font-size:11px;color:#94a3b8">🛬 Arrivals</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>From</th><th>Airport</th><th>Status</th></tr>';
    arr.forEach(f=>{{
      const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';
      html+=`<tr><td style="font-weight:bold">${{f.flight}}</td><td>${{f.from}}</td><td style="font-size:10px;color:#94a3b8">${{f.from_name}}</td><td style="color:${{sc}}">${{f.status}}</td></tr>`;
    }});
    html+='</table>';
    html+='<b style="font-size:11px;color:#94a3b8;display:block;margin-top:8px">🛫 Departures</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>To</th><th>Airport</th><th>Status</th></tr>';
    dep.forEach(f=>{{
      const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';
      html+=`<tr><td style="font-weight:bold">${{f.flight}}</td><td>${{f.to}}</td><td style="font-size:10px;color:#94a3b8">${{f.to_name}}</td><td style="color:${{sc}}">${{f.status}}</td></tr>`;
    }});
    html+='</table></div>';
  }}
  body.innerHTML=html;
}}

function closeAircraft() {{
  document.getElementById('aircraft-overlay').classList.remove('open');
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_FINAL.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print(f'Saved: airport_journey_planner_FINAL.html')

In [ ]:
# Read current file
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Fix 1: openCompare should NOT close the panel
# Fix 2: Show locations being added immediately
# Fix 3: Render table progressively as data comes in

old_compare = "function openCompare() {{\n  compareMode=true;\n  document.getElementById('compare-overlay').classList.add('open');\n  document.getElementById('instruction').innerHTML='<b style=\"color:#a855f7\">Compare mode ON</b> — click up to 3 locations';\n}}"

new_compare = """function openCompare() {
  compareMode = true;
  document.getElementById('compare-overlay').classList.add('open');
  document.getElementById('instruction').innerHTML = '<b style="color:#a855f7">✓ Compare mode ON</b> — click locations on map behind this panel (drag to move it)';
  document.getElementById('compare-body').innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:20px">Click anywhere on the map to add a location.<br>You can drag this panel to see the map.</div>';
}"""

content = content.replace(old_compare, new_compare)

# Fix addComparePoint to show immediate feedback
old_add = "function addComparePoint(lat,lng) {{\n  if (compareResults.length>=3) {{ alert('Maximum 3 locations. Click Clear All to reset.'); return; }}\n  const idx=compareResults.length;\n  const colour=PIN_COLOURS[idx];\n  const name=`Location ${{idx+1}} (${{lat.toFixed(3)}}, ${{lng.toFixed(3)}})`;\n  const marker=new google.maps.Marker({{\n    position:{{lat,lng}}, map,\n    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},\n    label:{{text:`${{idx+1}}`,color:'white',fontSize:'10px'}}\n  }});\n  compareMarkers.push(marker);\n  const entry={{lat,lng,name,colour,data:{{}}}};\n  compareResults.push(entry);\n  fetchCompareData(entry,idx);\n}}"

new_add = """function addComparePoint(lat, lng) {
  if (compareResults.length >= 3) {
    alert('Maximum 3 locations. Click Clear All to reset.');
    return;
  }
  const idx    = compareResults.length;
  const colour = PIN_COLOURS[idx];
  const name   = `Location ${idx+1} (${lat.toFixed(3)}, ${lng.toFixed(3)})`;

  const marker = new google.maps.Marker({
    position: {lat, lng}, map,
    icon: {path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2},
    label: {text:`${idx+1}`,color:'white',fontSize:'10px'}
  });
  compareMarkers.push(marker);

  const entry = {lat, lng, name, colour, data: {}};
  compareResults.push(entry);

  // Show immediate feedback in table
  updateCompareTable();
  fetchCompareData(entry, idx);
}

function updateCompareTable() {
  if (compareResults.length === 0) return;
  const hours = Array.from({length:24}, (_,i) => i);

  let thead = '<tr><th>Time</th>';
  compareResults.forEach(entry => {
    Object.keys(AIRPORTS).forEach(code => {
      thead += `<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>${entry.name.split('(')[0].trim()}→${code} 🚗</th>`;
      thead += `<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>${entry.name.split('(')[0].trim()}→${code} 🚆</th>`;
    });
  });
  thead += '</tr>';

  let tbody = '';
  hours.forEach(h => {
    tbody += `<tr><td style="font-weight:bold;color:#94a3b8">${String(h).padStart(2,'0')}:00</td>`;
    compareResults.forEach(entry => {
      Object.keys(AIRPORTS).forEach(code => {
        const d = entry.data[h]?.[code];
        const drive   = d?.DRIVING  != null ? `${d.DRIVING} min`  : '<span style="color:#475569">...</span>';
        const transit = d?.TRANSIT  != null ? `${d.TRANSIT} min`  : '<span style="color:#475569">...</span>';
        tbody += `<td class="drive">${drive}</td>`;
        tbody += `<td class="transit">${transit}</td>`;
      });
    });
    tbody += '</tr>';
  });

  document.getElementById('compare-body').innerHTML =
    `<table id="compare-table"><thead>${thead}</thead><tbody>${tbody}</tbody></table>`;
}"""

content = content.replace(old_add, new_add)

# Fix fetchCompareData to call updateCompareTable after each hour
old_fetch_line = "prog.textContent=`⏳ ${{entry.name}} — hour ${{hour+1}}/24...`;"
new_fetch_line = "prog.textContent=`⏳ ${entry.name} — hour ${hour+1}/24...`; updateCompareTable();"

content = content.replace(old_fetch_line, new_fetch_line)

# Fix renderCompareTable to call updateCompareTable
old_render = "prog.textContent=`✓ ${{entry.name}} complete!`;\n  renderCompareTable();"
new_render = "prog.textContent=`✓ ${entry.name} complete!`; updateCompareTable();"

content = content.replace(old_render, new_render)

# Make panel draggable
old_panel = '<div id="compare-panel">'
new_panel = '<div id="compare-panel" style="cursor:move" id="compare-panel">'
content = content.replace(old_panel, new_panel, 1)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'File size: {len(content):,} chars ({len(content)//1024} KB)')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Fix closeCompare to keep compareMode ON
content = content.replace(
    "function closeCompare() {\n  compareMode=false;",
    "function closeCompare() {\n  // Keep compareMode ON so clicks still register\n  // compareMode stays true until user clicks Clear All"
)

# Fix clearCompare to turn compareMode OFF
content = content.replace(
    "function clearCompare() {\n  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];\n  compareResults=[];",
    "function clearCompare() {\n  compareMode=false;\n  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];\n  compareResults=[];"
)

# Add floating status indicator - insert after #toggle-bar div
old_toggle_end = '</div>\n\n<div id="main">'
new_toggle_end = '''</div>

<div id="compare-status" style="display:none;position:fixed;bottom:20px;left:50%;transform:translateX(-50%);
  background:#7c3aed;color:white;padding:8px 20px;border-radius:20px;font-size:12px;
  z-index:1000;box-shadow:0 2px 10px rgba(0,0,0,0.5);">
  📍 Compare mode ON — click locations on map · <span id="compare-count">0</span>/3 selected
  · <button onclick="openCompare()" style="background:white;color:#7c3aed;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;margin-left:4px">View Table</button>
  · <button onclick="clearCompare()" style="background:rgba(255,255,255,0.2);color:white;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px">Clear</button>
</div>

<div id="main">'''

content = content.replace(old_toggle_end, new_toggle_end)

# Update openCompare to show status bar
content = content.replace(
    "function openCompare() {\n  compareMode = true;\n  document.getElementById('compare-overlay').classList.add('open');",
    "function openCompare() {\n  compareMode = true;\n  document.getElementById('compare-status').style.display='block';\n  document.getElementById('compare-overlay').classList.add('open');"
)

# Update closeCompare to keep status bar visible
content = content.replace(
    "  document.getElementById('compare-overlay').classList.remove('open');\n  document.getElementById('instruction').innerHTML = 'Click anywhere or pick an ITL3 region.<br>Toggle catchment zones above.';",
    "  document.getElementById('compare-overlay').classList.remove('open');\n  // Status bar stays visible so user knows compare mode is active"
)

# Update clearCompare to hide status bar
content = content.replace(
    "  document.getElementById('compare-body').innerHTML=\n    '<div style=\"color:#64748b;font-size:11px;text-align:center;padding:40px\">Close panel and click locations on the map.</div>';\n  document.getElementById('compare-progress').style.display='none';",
    "  document.getElementById('compare-body').innerHTML='<div style=\"color:#64748b;font-size:11px;text-align:center;padding:40px\">Close panel and click locations on the map.</div>';\n  document.getElementById('compare-progress').style.display='none';\n  document.getElementById('compare-status').style.display='none';\n  document.getElementById('compare-count').textContent='0';"
)

# Update addComparePoint to update counter
content = content.replace(
    "  compareMarkers.push(marker);\n\n  const entry = {lat, lng, name, colour, data: {}};\n  compareResults.push(entry);\n\n  // Show immediate feedback in table\n  updateCompareTable();\n  fetchCompareData(entry, idx);",
    "  compareMarkers.push(marker);\n  const entry = {lat, lng, name, colour, data: {}};\n  compareResults.push(entry);\n  document.getElementById('compare-count').textContent = compareResults.length;\n  updateCompareTable();\n  fetchCompareData(entry, idx);\n  openCompare(); // Auto-open panel to show progress"
)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'  File size: {len(content):,} chars ({len(content)//1024} KB)')

comparison table issue - google api seems to only take historical data into consideration rather than live updates.

adding a real time traffic route mode


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Add real-time traffic to driving routes
content = content.replace(
    "travelMode:google.maps.TravelMode[mode],\n        ...(mode==='TRANSIT'?{{transitOptions:{{",
    """travelMode:google.maps.TravelMode[mode],
        ...(mode==='DRIVING'?{{drivingOptions:{{
          departureTime: new Date(),
          trafficModel: google.maps.TrafficModel.BEST_GUESS
        }}}}:{{}}),
        ...(mode==='TRANSIT'?{{transitOptions:{{"""
)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'Real-time traffic added to driving routes')
print(f'File size: {len(content):,} chars ({len(content)//1024} KB)')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

print(f'trafficModel in file: {"trafficModel" in content}')
print(f'drivingOptions in file: {"drivingOptions" in content}')
print(f'BEST_GUESS in file: {"BEST_GUESS" in content}')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find the driving route section
idx = content.find('travelMode:google.maps.TravelMode[mode]')
print(f'Found at position: {idx}')
print(f'Context around it:')
print(content[idx:idx+300])

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_text = """travelMode:google.maps.TravelMode[mode],
        ...(mode==='TRANSIT'?{transitOptions:{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}:{})"""

new_text = """travelMode:google.maps.TravelMode[mode],
        ...(mode==='DRIVING'?{drivingOptions:{
          departureTime: new Date(),
          trafficModel: google.maps.TrafficModel.BEST_GUESS
        }}:{}),
        ...(mode==='TRANSIT'?{transitOptions:{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}:{})"""

# Check how many occurrences
count = content.count(old_text)
print(f'Found {count} occurrences')

new_content = content.replace(old_text, new_text)

print(f'trafficModel in file: {"trafficModel" in new_content}')
print(f'drivingOptions in file: {"drivingOptions" in new_content}')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Done')

google api key for the comparison does not work. it works on historical data
switching to TomTom

In [ ]:
import requests

TOMTOM_KEY = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj'

# Test route from Oxford to Heathrow at 8am
url = f'https://api.tomtom.com/routing/1/calculateRoute/51.7520,-1.2577:51.4706,-0.4541/json'
params = {
    'key': TOMTOM_KEY,
    'traffic': 'true',
    'departAt': '2026-07-21T10:00:00',  # Monday 8am
    'travelMode': 'car',
    'routeType': 'fastest'
}

response = requests.get(url, params=params)
print(f'Status: {response.status_code}')
if response.status_code == 200:
    data = response.json()
    route = data['routes'][0]['summary']
    mins  = round(route['travelTimeInSeconds'] / 60)
    km    = round(route['lengthInMeters'] / 1000, 1)
    print(f'Oxford → Heathrow at 8am: {mins} min · {km} km')
else:
    print(f'Error: {response.text[:200]}')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Add TomTom key as a constant after FETCH_TIME
content = content.replace(
    "const FETCH_TIME    = '",
    "const TOMTOM_KEY    = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj';\nconst FETCH_TIME    = '"
)

# Replace fetchCompareData with TomTom for drive + Google for transit
old_fetch = """async function fetchCompareData(entry,idx) {
  const prog=document.getElementById('compare-progress');
  prog.style.display='block';
  prog.textContent=`⏳ Calculating 24h routes for ${entry.name}...`;

  const hours=Array.from({length:24},(_,i)=>i);
  entry.data={};

  for (const hour of hours) {
    const depTime=new Date();
    depTime.setHours(hour,0,0,0);
    if (depTime<new Date()) depTime.setDate(depTime.getDate()+1);
    entry.data[hour]={};

    for (const [code,ap] of Object.entries(AIRPORTS)) {
      entry.data[hour][code]={};
      for (const mode of ['DRIVING','TRANSIT']) {
        await new Promise(resolve=>{
          directionsService.route({
            origin:{lat:entry.lat,lng:entry.lng},
            destination:{lat:ap.lat,lng:ap.lng},
            travelMode:google.maps.TravelMode[mode],
            ...(mode==='TRANSIT'?{transitOptions:{
              departureTime:depTime,
              modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
              routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
            }}:{})
          },(result,status)=>{
            entry.data[hour][code][mode]=status==='OK'?Math.round(result.routes[0].legs[0].duration.value/60):null;
            resolve();
          });
        });
        await new Promise(r=>setTimeout(r,200));
      }
    }
    prog.textContent=`⏳ ${entry.name} — hour ${hour+1}/24...`; updateCompareTable();
  }

  prog.textContent=`✓ ${entry.name} complete!`; updateCompareTable();
}"""

new_fetch = """async function fetchCompareData(entry, idx) {
  const prog = document.getElementById('compare-progress');
  prog.style.display = 'block';
  prog.textContent   = `⏳ Calculating 24h routes for ${entry.name}...`;

  const hours = Array.from({length:24}, (_,i) => i);
  entry.data  = {};

  // Get next Monday's date for consistent traffic patterns
  const nextMonday = new Date();
  const day = nextMonday.getDay();
  nextMonday.setDate(nextMonday.getDate() + (day <= 1 ? 1 - day : 8 - day));

  for (const hour of hours) {
    entry.data[hour] = {};

    // Format departure time for TomTom
    const depStr = `${nextMonday.toISOString().split('T')[0]}T${String(hour).padStart(2,'0')}:00:00`;

    // Format departure time for Google Maps
    const depTime = new Date(nextMonday);
    depTime.setHours(hour, 0, 0, 0);

    for (const [code, ap] of Object.entries(AIRPORTS)) {
      entry.data[hour][code] = {};

      // ── DRIVE via TomTom ─────────────────────────────
      try {
        const url = `https://api.tomtom.com/routing/1/calculateRoute/${entry.lat},${entry.lng}:${ap.lat},${ap.lng}/json?key=${TOMTOM_KEY}&traffic=true&departAt=${depStr}&travelMode=car&routeType=fastest`;
        const res  = await fetch(url);
        const data = await res.json();
        const secs = data.routes?.[0]?.summary?.travelTimeInSeconds;
        entry.data[hour][code]['DRIVING'] = secs ? Math.round(secs/60) : null;
      } catch(e) {
        entry.data[hour][code]['DRIVING'] = null;
      }
      await new Promise(r => setTimeout(r, 100));

      // ── TRANSIT via Google Maps ───────────────────────
      await new Promise(resolve => {
        directionsService.route({
          origin:      {lat:entry.lat, lng:entry.lng},
          destination: {lat:ap.lat,   lng:ap.lng},
          travelMode:  google.maps.TravelMode.TRANSIT,
          transitOptions: {
            departureTime: depTime,
            modes: [google.maps.TransitMode.RAIL, google.maps.TransitMode.SUBWAY, google.maps.TransitMode.BUS],
            routingPreference: google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }
        }, (result, status) => {
          entry.data[hour][code]['TRANSIT'] = status === 'OK'
            ? Math.round(result.routes[0].legs[0].duration.value/60)
            : null;
          resolve();
        });
      });
      await new Promise(r => setTimeout(r, 200));
    }

    prog.textContent = `⏳ ${entry.name} — hour ${hour+1}/24 (TomTom drive + Google transit)...`;
    updateCompareTable();
  }

  prog.textContent = `✓ ${entry.name} complete!`;
  updateCompareTable();
}"""

content = content.replace(old_fetch, new_fetch)

print(f'TomTom key in file: {"TOMTOM_KEY" in content}')
print(f'fetchCompareData replaced: {"TomTom drive" in content}')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'Done — file size: {len(content)//1024} KB')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

print(f'TOMTOM_KEY in file: {"TOMTOM_KEY" in content}')
print(f'TomTom drive in file: {"TomTom drive" in content}')
print(f'api.tomtom.com in file: {"api.tomtom.com" in content}')

# Show fetchCompareData context
idx = content.find('async function fetchCompareData')
print(f'\nfetchCompareData found at: {idx}')
print(content[idx:idx+300])

reducing calculating times.

setting up rush hours + chosen times set by the user

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# 1. Replace compare-controls section with time selector
old_controls = """    <div id="compare-controls">
      <span>Close panel and click up to 3 locations on the map to compare.</span>
      <button class="mode-btn" onclick="clearCompare()" style="margin-left:auto">Clear All</button>
    </div>"""

new_controls = """    <div id="compare-controls">
      <span style="font-size:11px;color:#64748b;">Close panel, click up to 3 locations on map, then select time slots to compare.</span>
      <div style="display:flex;gap:6px;flex-wrap:wrap;margin-top:6px;width:100%;">
        <label style="font-size:11px;color:#888;">Time slots:</label>
        <label style="font-size:11px;color:#e2e8f0;display:flex;align-items:center;gap:4px;">
          <input type="checkbox" id="t-morning" checked> 🌅 Morning rush (08:15)
        </label>
        <label style="font-size:11px;color:#e2e8f0;display:flex;align-items:center;gap:4px;">
          <input type="checkbox" id="t-noon" checked> ☀️ Midday (12:30)
        </label>
        <label style="font-size:11px;color:#e2e8f0;display:flex;align-items:center;gap:4px;">
          <input type="checkbox" id="t-evening" checked> 🌆 Evening rush (17:00)
        </label>
        <label style="font-size:11px;color:#e2e8f0;display:flex;align-items:center;gap:4px;">
          <input type="checkbox" id="t-night"> 🌙 Night (22:00)
        </label>
        <label style="font-size:11px;color:#e2e8f0;display:flex;align-items:center;gap:4px;">
          <input type="checkbox" id="t-custom"> ⏰ Custom:
          <input type="time" id="t-custom-time" value="10:00" style="background:#1e2d45;border:1px solid #2a3a50;color:white;padding:2px 4px;font-size:11px;border-radius:3px;">
        </label>
      </div>
      <button class="mode-btn" onclick="clearCompare()" style="margin-left:auto;margin-top:6px">Clear All</button>
    </div>"""

content = content.replace(old_controls, new_controls)

# 2. Replace fetchCompareData with slot-based version
start = content.find('async function fetchCompareData(entry, idx)')
end   = content.find('\nfunction renderCompareTable', start)

new_func = """async function fetchCompareData(entry, idx) {
  const prog = document.getElementById('compare-progress');
  prog.style.display = 'block';

  // Get selected time slots
  const slots = [];
  if (document.getElementById('t-morning').checked) slots.push({label:'🌅 08:15', hour:8,  min:15});
  if (document.getElementById('t-noon').checked)    slots.push({label:'☀️ 12:30', hour:12, min:30});
  if (document.getElementById('t-evening').checked) slots.push({label:'🌆 17:00', hour:17, min:0});
  if (document.getElementById('t-night').checked)   slots.push({label:'🌙 22:00', hour:22, min:0});
  if (document.getElementById('t-custom').checked) {
    const val = document.getElementById('t-custom-time').value;
    const [h,m] = val.split(':').map(Number);
    slots.push({label:`⏰ ${val}`, hour:h, min:m});
  }

  if (slots.length === 0) {
    prog.textContent = '⚠ Please select at least one time slot.';
    return;
  }

  entry.data  = {};
  entry.slots = slots;

  // Use next Monday for consistent traffic
  const nextMonday = new Date();
  const day = nextMonday.getDay();
  nextMonday.setDate(nextMonday.getDate() + (day <= 1 ? 1 - day : 8 - day));

  for (const slot of slots) {
    entry.data[slot.label] = {};
    prog.textContent = `⏳ ${entry.name} — calculating ${slot.label}...`;

    const depStr  = `${nextMonday.toISOString().split('T')[0]}T${String(slot.hour).padStart(2,'0')}:${String(slot.min).padStart(2,'0')}:00`;
    const depTime = new Date(nextMonday);
    depTime.setHours(slot.hour, slot.min, 0, 0);

    for (const [code, ap] of Object.entries(AIRPORTS)) {
      entry.data[slot.label][code] = {};

      // Drive via TomTom
      try {
        const url = `https://api.tomtom.com/routing/1/calculateRoute/${entry.lat},${entry.lng}:${ap.lat},${ap.lng}/json?key=${TOMTOM_KEY}&traffic=true&departAt=${depStr}&travelMode=car&routeType=fastest`;
        const res  = await fetch(url);
        const data = await res.json();
        const secs = data.routes?.[0]?.summary?.travelTimeInSeconds;
        entry.data[slot.label][code]['DRIVING'] = secs ? Math.round(secs/60) : null;
      } catch(e) {
        entry.data[slot.label][code]['DRIVING'] = null;
      }
      await new Promise(r => setTimeout(r, 100));

      // Transit via Google Maps
      await new Promise(resolve => {
        directionsService.route({
          origin:      {lat:entry.lat, lng:entry.lng},
          destination: {lat:ap.lat,   lng:ap.lng},
          travelMode:  google.maps.TravelMode.TRANSIT,
          transitOptions: {
            departureTime: depTime,
            modes: [google.maps.TransitMode.RAIL, google.maps.TransitMode.SUBWAY, google.maps.TransitMode.BUS],
            routingPreference: google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }
        }, (result, status) => {
          entry.data[slot.label][code]['TRANSIT'] = status === 'OK'
            ? Math.round(result.routes[0].legs[0].duration.value/60)
            : null;
          resolve();
        });
      });
      await new Promise(r => setTimeout(r, 150));
    }
    updateCompareTable();
  }

  prog.textContent = `✓ ${entry.name} complete!`;
  updateCompareTable();
}
"""

content = content[:start] + new_func + content[end:]

# 3. Replace renderCompareTable / updateCompareTable with slot-based version
old_update = content.find('function updateCompareTable()')
old_update_end = content.find('\nasync function fetchCompareData', old_update)

new_update = """function updateCompareTable() {
  if (compareResults.length === 0) return;

  // Collect all unique slot labels across all entries
  const allSlots = [];
  compareResults.forEach(entry => {
    if (entry.slots) {
      entry.slots.forEach(s => {
        if (!allSlots.find(x => x.label === s.label)) allSlots.push(s);
      });
    }
  });

  if (allSlots.length === 0) {
    document.getElementById('compare-body').innerHTML =
      '<div style="color:#64748b;font-size:11px;text-align:center;padding:20px">Calculating routes...</div>';
    return;
  }

  let thead = '<tr><th>Time Slot</th>';
  compareResults.forEach(entry => {
    Object.keys(AIRPORTS).forEach(code => {
      thead += `<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>${entry.name.split('(')[0].trim()} → ${code} 🚗</th>`;
      thead += `<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>${entry.name.split('(')[0].trim()} → ${code} 🚆</th>`;
    });
  });
  thead += '</tr>';

  let tbody = '';
  allSlots.forEach(slot => {
    tbody += `<tr><td style="font-weight:bold;color:#94a3b8;white-space:nowrap">${slot.label}</td>`;
    compareResults.forEach(entry => {
      Object.keys(AIRPORTS).forEach(code => {
        const d       = entry.data[slot.label]?.[code];
        const drive   = d?.DRIVING  != null ? `${d.DRIVING} min`  : '<span style="color:#475569">...</span>';
        const transit = d?.TRANSIT  != null ? `${d.TRANSIT} min`  : '<span style="color:#475569">...</span>';
        tbody += `<td class="drive">${drive}</td>`;
        tbody += `<td class="transit">${transit}</td>`;
      });
    });
    tbody += '</tr>';
  });

  document.getElementById('compare-body').innerHTML =
    `<table id="compare-table"><thead>${thead}</thead><tbody>${tbody}</tbody></table>`;
}
"""

content = content[:old_update] + new_update + content[old_update_end:]

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'Updated with time slot selector')
print(f'File size: {len(content)//1024} KB')
print(f'TomTom in file: {"api.tomtom.com" in content}')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Check for unclosed script tags or mismatched braces
script_opens  = content.count('<script')
script_closes = content.count('</script>')
print(f'<script> tags:  {script_opens}')
print(f'</script> tags: {script_closes}')

# Find where updateCompareTable ends and what comes after
idx = content.find('function updateCompareTable()')
end = content.find('\n}', content.find('document.getElementById(\'compare-body\')', idx))
print(f'\nAfter updateCompareTable ends:')
print(content[end:end+200])

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find and fix the stray >
idx = content.find('function updateCompareTable()')
end = content.find('\n}', content.find("document.getElementById('compare-body')", idx))

print('Before fix:')
print(repr(content[end:end+20]))

# Remove the stray >
content = content[:end+2] + content[end+2:].lstrip('\n>').lstrip()

# Verify
print('After fix:')
idx2 = content.find('function updateCompareTable()')
end2 = content.find('\n}', content.find("document.getElementById('compare-body')", idx2))
print(repr(content[end2:end2+20]))

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print('Fixed')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find everything after the last </script>
last_script = content.rfind('</script>')
print(f'Last </script> at position: {last_script}')
print(f'Total file length: {len(content)}')
print(f'\nEverything after </script>:')
print(repr(content[last_script:last_script+200]))

# Also check what's just before </script>
print(f'\nJust before </script>:')
print(repr(content[last_script-200:last_script]))

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Extract the misplaced updateCompareTable function
misplaced_start = content.find('</htmlfunction updateCompareTable')
misplaced_func_start = content.find('function updateCompareTable', misplaced_start)
misplaced_func_end = content.find('\nasync function fetchCompareData', misplaced_func_start)

# Get the misplaced function text
misplaced_func = content[misplaced_func_start:misplaced_func_end]

# Fix the broken </html tag
content = content.replace('</htmlfunction updateCompareTable', '</html')

# Remove everything after </html that doesn't belong
end_html = content.find('</html>') + len('</html>')
content = content[:end_html]

# Now insert updateCompareTable before closeAircraft inside the script
insert_before = 'function closeAircraft()'
content = content.replace(
    insert_before,
    misplaced_func + '\n\n' + insert_before
)

# Verify
print(f'updateCompareTable in file: {"function updateCompareTable" in content}')
print(f'File ends with: {repr(content[-30:])}')
print(f'</script> count: {content.count("</script>")}')
print(f'File size: {len(content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print('Fixed')

In [ ]:
import os

# Check what HTML files exist in Drive
files = []
for f in os.listdir('/content/drive/MyDrive/'):
    if f.endswith('.html'):
        size = os.path.getsize(f'/content/drive/MyDrive/{f}')
        files.append((f, size))
        print(f'{f}: {size//1024} KB')

In [ ]:
print('iso_geojson' in dir())
print('itl3_json' in dir())
print('centroids_json' in dir())
print('slim_data' in dir())
print(f'itl3_json size: {len(json.dumps(itl3_json)):,} chars')

file got corrupted - re-doing the map again

In [ ]:
import json

flight_data_str  = json.dumps(slim_data, ensure_ascii=True)
TOMTOM_KEY       = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj'

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; overflow:hidden; }}
  #header {{ background:#1a1a2e; color:white; padding:8px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:13px; font-weight:bold; }}
  #api-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; }}
  #api-bar button:hover {{ background:#2563eb; }}
  #control-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:8px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #control-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .mode-btn {{ padding:4px 10px; font-size:11px; border-radius:3px; cursor:pointer; border:1px solid #2a3a50; background:#1e2d45; color:#94a3b8; }}
  .mode-btn.active {{ background:#3b82f6; color:white; border-color:#3b82f6; }}
  #itl3-select {{ background:#1e2d45; border:1px solid #2a3a50; color:#e2e8f0; padding:4px 8px; font-size:11px; border-radius:3px; max-width:190px; }}
  #toggle-bar {{ background:#080d1a; padding:5px 20px; display:flex; align-items:center; gap:16px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #toggle-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .airport-group {{ display:flex; align-items:center; gap:5px; }}
  .airport-group-label {{ font-size:11px; font-weight:bold; margin-right:2px; }}
  .tog {{ padding:2px 8px; font-size:10px; border-radius:3px; cursor:pointer; border:1px solid; opacity:0.35; transition:opacity 0.15s; }}
  .tog.on {{ opacity:1; }}
  button.compare-btn {{ background:#7c3aed; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; margin-left:auto; }}
  button.compare-btn:hover {{ background:#6d28d9; }}
  button.aircraft-btn {{ background:#0891b2; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; }}
  button.aircraft-btn:hover {{ background:#0e7490; }}
  #compare-status {{ display:none; position:fixed; bottom:20px; left:50%; transform:translateX(-50%);
    background:#7c3aed; color:white; padding:8px 20px; border-radius:20px; font-size:12px;
    z-index:1000; box-shadow:0 2px 10px rgba(0,0,0,0.5); white-space:nowrap; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}
  #sidebar {{ width:320px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:8px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:8px; }}
  .airport-block {{ margin-bottom:8px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:7px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; }}
  .dot {{ width:7px; height:7px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:6px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .straight-row {{ padding:4px 12px; background:#0d1525; border-bottom:1px solid #1e2d45; font-size:10px; color:#64748b; display:flex; justify-content:space-between; }}
  .route-left {{ display:flex; align-items:center; gap:7px; }}
  .route-icon {{ width:18px; height:18px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:10px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:10px; height:10px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:24px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:4px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:6px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:6px; }}
  .leg {{ display:flex; align-items:center; gap:3px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:14px; height:2px; border-radius:1px; }}
  #loading-bar {{ padding:5px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
  #compare-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #compare-overlay.open {{ display:flex; }}
  #compare-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:90vw; max-width:1100px; max-height:85vh; display:flex; flex-direction:column; overflow:hidden; }}
  #compare-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #compare-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #compare-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #compare-controls {{ padding:10px 20px; border-bottom:1px solid #1e2d45; display:flex; flex-direction:column; gap:8px; }}
  #compare-controls span {{ font-size:11px; color:#64748b; }}
  .time-slots {{ display:flex; gap:12px; flex-wrap:wrap; align-items:center; }}
  .time-slots label {{ font-size:11px; color:#e2e8f0; display:flex; align-items:center; gap:4px; cursor:pointer; }}
  #compare-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  #compare-table {{ width:100%; border-collapse:collapse; font-size:11px; }}
  #compare-table th {{ background:#1e2d45; color:#94a3b8; padding:8px 10px; text-align:left; border:1px solid #2a3a50; white-space:nowrap; position:sticky; top:0; }}
  #compare-table td {{ padding:6px 10px; border:1px solid #1e2d45; color:#e2e8f0; white-space:nowrap; }}
  #compare-table tr:nth-child(even) td {{ background:#0d1525; }}
  #compare-table td.drive {{ color:#ff8800; }}
  #compare-table td.transit {{ color:#00ccff; }}
  #compare-progress {{ padding:8px 20px; border-top:1px solid #1e2d45; font-size:10px; color:#f59e0b; display:none; }}
  #aircraft-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #aircraft-overlay.open {{ display:flex; }}
  #aircraft-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:700px; max-height:80vh; display:flex; flex-direction:column; overflow:hidden; }}
  #aircraft-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #aircraft-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #aircraft-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #aircraft-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  .aircraft-airport {{ margin-bottom:16px; }}
  .aircraft-airport h3 {{ font-size:12px; font-weight:bold; margin-bottom:6px; padding-bottom:4px; border-bottom:1px solid #1e2d45; }}
  .aircraft-table {{ width:100%; border-collapse:collapse; font-size:11px; margin-bottom:8px; }}
  .aircraft-table th {{ background:#1e2d45; color:#94a3b8; padding:6px 8px; text-align:left; border:1px solid #2a3a50; }}
  .aircraft-table td {{ padding:5px 8px; border:1px solid #1e2d45; color:#e2e8f0; }}
  .aircraft-table tr:nth-child(even) td {{ background:#0d1525; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="control-bar">
  <label>Direction:</label>
  <button class="mode-btn active" id="btn-to"   onclick="setMode('to')">📍→✈ To Airport</button>
  <button class="mode-btn"        id="btn-from" onclick="setMode('from')">✈→📍 From Airport</button>
  <label style="margin-left:8px">ITL3:</label>
  <select id="itl3-select" onchange="selectITL3()">
    <option value="">— pick a region —</option>
  </select>
</div>

<div id="toggle-bar">
  <label>Catchment:</label>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#3b82f6">LHR</span>
    <button class="tog on" id="tog-LHR-30" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',30)">30</button>
    <button class="tog on" id="tog-LHR-45" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',45)">45</button>
    <button class="tog on" id="tog-LHR-60" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#ef4444">LGW</span>
    <button class="tog on" id="tog-LGW-30" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',30)">30</button>
    <button class="tog on" id="tog-LGW-45" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',45)">45</button>
    <button class="tog on" id="tog-LGW-60" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#22c55e">LTN</span>
    <button class="tog on" id="tog-LTN-30" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',30)">30</button>
    <button class="tog on" id="tog-LTN-45" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',45)">45</button>
    <button class="tog on" id="tog-LTN-60" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',60)">60</button>
  </div>
  <button class="aircraft-btn" onclick="openAircraft()">✈ Flight Info</button>
  <button class="compare-btn"  onclick="openCompare()">📊 Compare</button>
</div>

<div id="compare-status">
  📍 Compare ON — <span id="compare-count">0</span>/3 locations selected
  · <button onclick="openCompare()" style="background:white;color:#7c3aed;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">View Table</button>
  · <button onclick="clearCompare()" style="background:rgba(255,255,255,0.2);color:white;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">Clear</button>
  · <button onclick="exitCompare()" style="background:#ef4444;color:white;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">✕ Exit</button>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">Click anywhere or pick an ITL3 region.<br>Toggle catchment zones above.<br>Use Compare for time slot analysis.</div>
    <div id="results"><div id="empty">Load the map then click anywhere.</div></div>
    <div id="loading-bar">⏳ Calculating routes...</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<!-- Compare popup -->
<div id="compare-overlay">
  <div id="compare-panel">
    <div id="compare-header">
      <h2>📊 Journey Time Comparison</h2>
      <button onclick="closeCompare()">✕</button>
    </div>
    <div id="compare-controls">
      <span>Close panel and click up to 3 locations on the map. Select time slots to compare.</span>
      <div class="time-slots">
        <label><input type="checkbox" id="t-morning" checked> 🌅 Morning rush (08:15)</label>
        <label><input type="checkbox" id="t-noon" checked> ☀️ Midday (12:30)</label>
        <label><input type="checkbox" id="t-evening" checked> 🌆 Evening rush (17:00)</label>
        <label><input type="checkbox" id="t-night"> 🌙 Night (22:00)</label>
        <label><input type="checkbox" id="t-custom"> ⏰ Custom:
          <input type="time" id="t-custom-time" value="10:00"
            style="background:#1e2d45;border:1px solid #2a3a50;color:white;padding:2px 4px;font-size:11px;border-radius:3px;">
        </label>
        <button class="mode-btn" onclick="clearCompare()">Clear All</button>
      </div>
    </div>
    <div id="compare-progress"></div>
    <div id="compare-body">
      <div style="color:#64748b;font-size:11px;text-align:center;padding:40px">
        Close this panel and click locations on the map.
      </div>
    </div>
  </div>
</div>

<!-- Aircraft popup -->
<div id="aircraft-overlay">
  <div id="aircraft-panel">
    <div id="aircraft-header">
      <h2>✈ Flight Information</h2>
      <button onclick="closeAircraft()">✕</button>
    </div>
    <div id="aircraft-body"></div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e' }},
}};

const ROUTE_COLOURS  = {{ DRIVING:'#ff8800', TRANSIT:'#00ccff' }};
const TOMTOM_KEY     = '{TOMTOM_KEY}';
const ISO_DATA       = {json.dumps(iso_geojson)};
const ITL3_DATA      = {json.dumps(itl3_json)};
const CENTROIDS      = {centroids_json};
const FLIGHT_DATA    = {flight_data_str};
const FETCH_TIME     = '{fetch_time}';
const TIMES          = [30, 45, 60];
const ISO_WEIGHTS    = [3, 2, 1.5];

let map, directionsService;
let startMarker=null, renderers=[], straightLines=[];
let isoPolygons={{}};
let mapLoaded=false, currentMode='to';
let compareMode=false, compareMarkers=[], compareResults=[];
const PIN_COLOURS=['#f59e0b','#a855f7','#ec4899'];

window.addEventListener('load', () => {{
  const sel = document.getElementById('itl3-select');
  CENTROIDS.sort((a,b)=>a.name.localeCompare(b.name)).forEach(c => {{
    const opt = document.createElement('option');
    opt.value = JSON.stringify({{lat:c.lat, lng:c.lng}});
    opt.textContent = `${{c.name}} — ${{c.station}}`;
    sel.appendChild(opt);
  }});
}});

function selectITL3() {{
  const val = document.getElementById('itl3-select').value;
  if (!val || !map) return;
  const {{lat,lng}} = JSON.parse(val);
  handlePoint(lat,lng);
}}

function setMode(mode) {{
  currentMode=mode;
  document.getElementById('btn-to').className   = 'mode-btn'+(mode==='to'?' active':'');
  document.getElementById('btn-from').className = 'mode-btn'+(mode==='from'?' active':'');
  clearRoutes();
  if (startMarker) {{ startMarker.setMap(null); startMarker=null; }}
  document.getElementById('results').innerHTML  = '<div id="empty">Click anywhere to begin.</div>';
  document.getElementById('coords').textContent = 'Coordinates: —';
}}

function toggleIso(iata, mins) {{
  const btn   = document.getElementById(`tog-${{iata}}-${{mins}}`);
  const polys = isoPolygons[iata]?.[mins] || [];
  const isOn  = btn.classList.contains('on');
  polys.forEach(p=>p.setMap(isOn?null:map));
  btn.classList.toggle('on',!isOn);
  btn.style.opacity = isOn?'0.35':'1';
}}

function haversine(lat1,lng1,lat2,lng2) {{
  const R=6371,dLat=(lat2-lat1)*Math.PI/180,dLng=(lng2-lng1)*Math.PI/180;
  const a=Math.sin(dLat/2)**2+Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLng/2)**2;
  return (R*2*Math.atan2(Math.sqrt(a),Math.sqrt(1-a))).toFixed(1);
}}

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display='none';
  document.getElementById('map').style.display='block';
  const script=document.createElement('script');
  script.src=`https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async=true; script.defer=true;
  document.head.appendChild(script);
  mapLoaded=true;
}}

function coordsToPath(c) {{ return c.map(x=>({{lat:x[1],lng:x[0]}})); }}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center:{{lat:51.5,lng:-0.3}}, zoom:8, mapTypeId:'roadmap',
    styles:[
      {{elementType:'geometry',stylers:[{{color:'#1a1a2e'}}]}},
      {{elementType:'labels.text.fill',stylers:[{{color:'#94a3b8'}}]}},
      {{elementType:'labels.text.stroke',stylers:[{{color:'#1a1a2e'}}]}},
      {{featureType:'road',elementType:'geometry',stylers:[{{color:'#2d3748'}}]}},
      {{featureType:'road.highway',elementType:'geometry',stylers:[{{color:'#2d4a7a'}}]}},
      {{featureType:'water',elementType:'geometry',stylers:[{{color:'#0d1525'}}]}},
      {{featureType:'transit.line',elementType:'geometry',stylers:[{{color:'#4a5568'}}]}},
      {{featureType:'transit.station',elementType:'geometry',stylers:[{{color:'#374151'}}]}},
      {{featureType:'poi',stylers:[{{visibility:'off'}}]}},
    ]
  }});
  directionsService = new google.maps.DirectionsService();

  // ITL3 choropleth
  const pops=ITL3_DATA.features.map(f=>f.properties.population_2024||0);
  const minPop=Math.min(...pops), maxPop=Math.max(...pops);
  function popColour(pop) {{
    if (!pop) return '#374151';
    const t=(pop-minPop)/(maxPop-minPop);
    return `rgb(${{Math.round(255*Math.min(1,t*2))}},${{Math.round(255*Math.max(0,1-t*2))}},50)`;
  }}
  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const colour=popColour(f.properties.population_2024);
    const rings=f.geometry.type==='MultiPolygon'
      ? f.geometry.coordinates.map(p=>p[0])
      : [f.geometry.coordinates[0]];
    rings.forEach(ring => {{
      new google.maps.Polygon({{
        paths:coordsToPath(ring), strokeColor:'#374151', strokeWeight:0.5,
        fillColor:colour, fillOpacity:0.5, clickable:false, map
      }});
    }});
  }});

  // ITL3 centroid markers
  CENTROIDS.forEach(c => {{
    const marker=new google.maps.Marker({{
      position:{{lat:c.lat,lng:c.lng}}, map, title:c.name,
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:5,fillColor:'#fff',fillOpacity:0.6,strokeColor:'#94a3b8',strokeWeight:1}}
    }});
    const iw=new google.maps.InfoWindow({{
      content:`<div style="font-size:11px"><b>${{c.name}}</b><br>Pop: ${{c.pop?.toLocaleString()}}<br>GDHI: £${{c.gdhi?.toLocaleString()}}M</div>`
    }});
    marker.addListener('click',()=>{{ iw.open(map,marker); handlePoint(c.lat,c.lng); }});
  }});

  // Isochrones
  const isoColours={{LHR:'#3b82f6',LGW:'#ef4444',LTN:'#22c55e'}};
  Object.entries(ISO_DATA).forEach(([iata,geojson]) => {{
    isoPolygons[iata]={{}};
    geojson.features.forEach((feature,i) => {{
      const mins=TIMES[i];
      const poly=new google.maps.Polygon({{
        paths:coordsToPath(feature.geometry.coordinates[0]),
        strokeColor:isoColours[iata], strokeWeight:ISO_WEIGHTS[i],
        strokeOpacity:0.9, fillOpacity:0, clickable:false, map
      }});
      if (!isoPolygons[iata][mins]) isoPolygons[iata][mins]=[];
      isoPolygons[iata][mins].push(poly);
    }});
  }});

  // Airport markers
  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    const marker=new google.maps.Marker({{
      position:{{lat:ap.lat,lng:ap.lng}}, map, title:ap.name,
      label:{{text:'✈',color:'white',fontSize:'11px'}},
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:13,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}}
    }});
    const iw=new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click',()=>iw.open(map,marker));
  }});

  map.addListener('click', e => {{
    const lat=e.latLng.lat(), lng=e.latLng.lng();
    if (compareMode) {{ addComparePoint(lat,lng); }}
    else {{ handlePoint(lat,lng); }}
  }});

  document.getElementById('empty').textContent='Click anywhere or pick an ITL3 region.';
}}

function clearRoutes() {{
  renderers.forEach(r=>r.setMap(null)); renderers=[];
  straightLines.forEach(l=>l.setMap(null)); straightLines=[];
}}

function handlePoint(lat,lng) {{
  document.getElementById('coords').textContent=
    `${{currentMode==='to'?'From':'To'}}: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
  if (startMarker) startMarker.setMap(null);
  startMarker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:9,
      fillColor:currentMode==='to'?'#ff8800':'#ff00ff',
      fillOpacity:1,strokeColor:'white',strokeWeight:2}}
  }});
  clearRoutes();
  calculateAllRoutes(lat,lng);
}}

function calculateAllRoutes(lat,lng) {{
  const lb=document.getElementById('loading-bar');
  lb.style.display='block'; lb.style.color='#f59e0b'; lb.textContent='⏳ Calculating routes...';
  const straight={{}};
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{ straight[code]=haversine(lat,lng,ap.lat,ap.lng); }});
  const modeLabel=currentMode==='to'?'→ ✈':'✈ →';
  document.getElementById('results').innerHTML=Object.entries(AIRPORTS).map(([code,ap])=>`
    <div class="airport-block">
      <div class="airport-header"><div class="dot" style="background:${{ap.colour}}"></div>${{modeLabel}} ${{ap.name}} (${{code}})</div>
      <div class="straight-row"><span> Straight line</span><span style="color:#e2e8f0">${{straight[code]}} km</span></div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left"><div class="route-icon" style="background:#ff880033">🚗</div><div class="route-label">Drive</div></div>
        <div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left"><div class="route-icon" style="background:#00ccff33">🚆</div><div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div></div>
        <div class="spinner"></div>
      </div>
    </div>`).join('');

  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    const line=new google.maps.Polyline({{
      path:[{{lat,lng}},{{lat:ap.lat,lng:ap.lng}}],
      strokeColor:ap.colour,strokeWeight:1,strokeOpacity:0.5,
      icons:[{{icon:{{path:'M 0,-1 0,1',strokeOpacity:1,scale:3}},offset:'0',repeat:'12px'}}],map
    }});
    straightLines.push(line);
  }});

  let completed=0;
  const total=Object.keys(AIRPORTS).length*2;
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    ['DRIVING','TRANSIT'].forEach(mode=>{{
      const renderer=new google.maps.DirectionsRenderer({{
        suppressMarkers:true,
        polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeWeight:mode==='DRIVING'?5:4,strokeOpacity:0.85}}
      }});
      renderers.push(renderer);
      const origin     =currentMode==='to'?{{lat,lng}}:{{lat:ap.lat,lng:ap.lng}};
      const destination=currentMode==='to'?{{lat:ap.lat,lng:ap.lng}}:{{lat,lng}};
      directionsService.route({{
        origin,destination,travelMode:google.maps.TravelMode[mode],
        ...(mode==='DRIVING'?{{drivingOptions:{{departureTime:new Date(),trafficModel:google.maps.TrafficModel.BEST_GUESS}}}}:{{}}),
        ...(mode==='TRANSIT'?{{transitOptions:{{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}}}:{{}})
      }},(result,status)=>{{
        completed++;
        const el=document.getElementById(`r-${{code}}-${{mode}}`);
        if (status==='OK') {{
          renderer.setMap(map); renderer.setDirections(result);
          const leg=result.routes[0].legs[0];
          const mins=Math.round(leg.duration_in_traffic?.value/60 || leg.duration.value/60);
          const km=(leg.distance.value/1000).toFixed(1);
          if (el) {{
            el.querySelector('.spinner').outerHTML=`<div class="route-val done">${{mins}} min · ${{km}} km</div>`;
            if (mode==='TRANSIT') {{
              const steps=result.routes[0].legs[0].steps
                .filter(s=>s.travel_mode==='TRANSIT')
                .map(s=>s.transit?.line?.short_name||s.transit?.line?.name||'')
                .filter(Boolean).join(' → ');
              if (steps) {{ const sub=el.querySelector('.route-sub'); if(sub) sub.textContent=steps; }}
            }}
            el.onclick=()=>{{
              renderers.forEach(r=>r.setOptions({{polylineOptions:{{strokeOpacity:0.15,strokeWeight:2}}}}));
              renderer.setOptions({{polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeOpacity:1,strokeWeight:6}}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) el.querySelector('.spinner').outerHTML=`<div class="route-val" style="color:#64748b">not available</div>`;
        }}
        if (completed===total) {{
          lb.textContent='✓ Routes loaded — click a row to highlight'; lb.style.color='#22c55e';
        }}
      }});
    }});
  }});
}}

// ── COMPARE ───────────────────────────────────────────────
function openCompare() {{
  compareMode=true;
  document.getElementById('compare-overlay').classList.add('open');
  document.getElementById('compare-status').style.display='block';
}}

function closeCompare() {{
  document.getElementById('compare-overlay').classList.remove('open');
  // compareMode stays ON
}}

function exitCompare() {{
  compareMode=false;
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-status').style.display='none';
  document.getElementById('compare-count').textContent='0';
  document.getElementById('compare-overlay').classList.remove('open');
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Close panel and click locations on the map.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function clearCompare() {{
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-count').textContent='0';
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Cleared. Click locations on the map to start again.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function addComparePoint(lat,lng) {{
  if (compareResults.length>=3) {{ alert('Maximum 3 locations. Click Clear All to reset.'); return; }}
  const idx=compareResults.length;
  const colour=PIN_COLOURS[idx];
  const name=`Loc ${{idx+1}} (${{lat.toFixed(2)}},${{lng.toFixed(2)}})`;
  const marker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},
    label:{{text:`${{idx+1}}`,color:'white',fontSize:'10px'}}
  }});
  compareMarkers.push(marker);
  const entry={{lat,lng,name,colour,slots:[],data:{{}}}};
  compareResults.push(entry);
  document.getElementById('compare-count').textContent=compareResults.length;
  openCompare();
  fetchCompareData(entry,idx);
}}

function getSelectedSlots() {{
  const slots=[];
  if (document.getElementById('t-morning').checked) slots.push({{label:'🌅 08:15',hour:8,min:15}});
  if (document.getElementById('t-noon').checked)    slots.push({{label:'☀️ 12:30',hour:12,min:30}});
  if (document.getElementById('t-evening').checked) slots.push({{label:'🌆 17:00',hour:17,min:0}});
  if (document.getElementById('t-night').checked)   slots.push({{label:'🌙 22:00',hour:22,min:0}});
  if (document.getElementById('t-custom').checked) {{
    const val=document.getElementById('t-custom-time').value;
    const [h,m]=val.split(':').map(Number);
    slots.push({{label:`⏰ ${{val}}`,hour:h,min:m}});
  }}
  return slots;
}}

async function fetchCompareData(entry,idx) {{
  const prog=document.getElementById('compare-progress');
  prog.style.display='block';
  const slots=getSelectedSlots();
  if (slots.length===0) {{ prog.textContent='⚠ Select at least one time slot.'; return; }}

  entry.slots=slots;

  const nextMonday=new Date();
  const day=nextMonday.getDay();
  nextMonday.setDate(nextMonday.getDate()+(day<=1?1-day:8-day));

  for (const slot of slots) {{
    entry.data[slot.label]={{}};
    prog.textContent=`${{entry.name}} — ${{slot.label}}...`;

    const depStr=`${{nextMonday.toISOString().split('T')[0]}}T${{String(slot.hour).padStart(2,'0')}}:${{String(slot.min).padStart(2,'0')}}:00`;
    const depTime=new Date(nextMonday);
    depTime.setHours(slot.hour,slot.min,0,0);

    for (const [code,ap] of Object.entries(AIRPORTS)) {{
      entry.data[slot.label][code]={{}};

      // Drive via TomTom
      try {{
        const url=`https://api.tomtom.com/routing/1/calculateRoute/${{entry.lat}},${{entry.lng}}:${{ap.lat}},${{ap.lng}}/json?key=${{TOMTOM_KEY}}&traffic=true&departAt=${{depStr}}&travelMode=car&routeType=fastest`;
        const res=await fetch(url);
        const data=await res.json();
        const secs=data.routes?.[0]?.summary?.travelTimeInSeconds;
        entry.data[slot.label][code]['DRIVING']=secs?Math.round(secs/60):null;
      }} catch(e) {{
        entry.data[slot.label][code]['DRIVING']=null;
      }}
      await new Promise(r=>setTimeout(r,100));

      // Transit via Google Maps
      await new Promise(resolve=>{{
        directionsService.route({{
          origin:{{lat:entry.lat,lng:entry.lng}},
          destination:{{lat:ap.lat,lng:ap.lng}},
          travelMode:google.maps.TravelMode.TRANSIT,
          transitOptions:{{
            departureTime:depTime,
            modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
            routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }}
        }},(result,status)=>{{
          entry.data[slot.label][code]['TRANSIT']=status==='OK'
            ?Math.round(result.routes[0].legs[0].duration.value/60):null;
          resolve();
        }});
      }});
      await new Promise(r=>setTimeout(r,150));
    }}
    updateCompareTable();
  }}

  prog.textContent=`✓ ${{entry.name}} complete!`;
  updateCompareTable();
}}

function updateCompareTable() {{
  if (compareResults.length===0) return;
  const allSlots=[];
  compareResults.forEach(entry=>{{
    (entry.slots||[]).forEach(s=>{{
      if (!allSlots.find(x=>x.label===s.label)) allSlots.push(s);
    }});
  }});
  if (allSlots.length===0) return;

  let thead='<tr><th>Time Slot</th>';
  compareResults.forEach(entry=>{{
    Object.keys(AIRPORTS).forEach(code=>{{
      thead+=`<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px"></span>${{entry.name}}→${{code}} 🚗</th>`;
      thead+=`<th><span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px"></span>${{entry.name}}→${{code}} 🚆</th>`;
    }});
  }});
  thead+='</tr>';

  let tbody='';
  allSlots.forEach(slot=>{{
    tbody+=`<tr><td style="font-weight:bold;color:#94a3b8;white-space:nowrap">${{slot.label}}</td>`;
    compareResults.forEach(entry=>{{
      Object.keys(AIRPORTS).forEach(code=>{{
        const d=entry.data[slot.label]?.[code];
        tbody+=`<td class="drive">${{d?.DRIVING!=null?d.DRIVING+' min':'<span style="color:#475569">...</span>'}}</td>`;
        tbody+=`<td class="transit">${{d?.TRANSIT!=null?d.TRANSIT+' min':'<span style="color:#475569">...</span>'}}</td>`;
      }});
    }});
    tbody+='</tr>';
  }});

  document.getElementById('compare-body').innerHTML=
    `<table id="compare-table"><thead>${{thead}}</thead><tbody>${{tbody}}</tbody></table>`;
}}

// ── FLIGHT INFO ───────────────────────────────────────────
function openAircraft() {{
  document.getElementById('aircraft-overlay').classList.add('open');
  const body=document.getElementById('aircraft-body');
  let html=`<p style="color:#64748b;font-size:10px;margin-bottom:12px">📅 Scheduled flights — fetched: ${{FETCH_TIME}}</p>`;
  for (const [code,ap] of Object.entries(AIRPORTS)) {{
    const arr=FLIGHT_DATA[code]?.arrivals||[];
    const dep=FLIGHT_DATA[code]?.departures||[];
    html+=`<div class="aircraft-airport">`;
    html+=`<h3 style="color:${{ap.colour}}">✈ ${{ap.name}} (${{code}}) — ${{arr.length}} arrivals · ${{dep.length}} departures</h3>`;
    html+='<b style="font-size:11px;color:#94a3b8">🛬 Arrivals</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>Airline</th><th>From</th><th>Aircraft</th><th>Terminal</th><th>Delay</th><th>Status</th></tr>';
arr.forEach(f=>{{
  const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';
  const delay=f.delay>0?`<span style="color:#ef4444">+${{f.delay}}m</span>`:'<span style="color:#22c55e">On time</span>';
  html+=`<tr><td style="font-weight:bold">${{f.flight}}</td><td style="font-size:10px">${{f.airline}}</td><td>${{f.from}}</td><td>${{f.aircraft}}</td><td>${{f.terminal}}</td><td>${{delay}}</td><td style="color:${{sc}}">${{f.status}}</td></tr>`;
}});
    html+='</table>';
    html+='<b style="font-size:11px;color:#94a3b8;display:block;margin-top:8px">🛫 Departures</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>Airline</th><th>To</th><th>Aircraft</th><th>Terminal</th><th>Delay</th><th>Status</th></tr>';
dep.forEach(f=>{{
  const sc=f.status==='active'?'#22c55e':f.status==='landed'?'#94a3b8':'#f59e0b';
  const delay=f.delay>0?`<span style="color:#ef4444">+${{f.delay}}m</span>`:'<span style="color:#22c55e">On time</span>';
  html+=`<tr><td style="font-weight:bold">${{f.flight}}</td><td style="font-size:10px">${{f.airline}}</td><td>${{f.to}}</td><td>${{f.aircraft}}</td><td>${{f.terminal}}</td><td>${{delay}}</td><td style="color:${{sc}}">${{f.status}}</td></tr>`;
}});
    html+='</table></div>';
  }}
  body.innerHTML=html;
}}

function closeAircraft() {{
  document.getElementById('aircraft-overlay').classList.remove('open');
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_FINAL.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print(f'Saved: airport_journey_planner_FINAL.html')
print(f'File size: {len(html_content)//1024} KB')
print(f'TomTom key: {"TOMTOM_KEY" in html_content}')
print(f'api.tomtom.com: {"api.tomtom.com" in html_content}')
print(f'Compare feature: {"updateCompareTable" in html_content}')
print(f'Flight info: {"FLIGHT_DATA" in html_content}')

adding more info for the flight status

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_func_start = content.find('function updateCompareTable()')
old_func_end   = content.find('\nasync function fetchCompareData', old_func_start)

new_func = """function updateCompareTable() {
  if (compareResults.length === 0) return;

  const allSlots = [];
  compareResults.forEach(entry => {
    (entry.slots || []).forEach(s => {
      if (!allSlots.find(x => x.label === s.label)) allSlots.push(s);
    });
  });
  if (allSlots.length === 0) return;

  let html = '';

  // One table per airport
  Object.entries(AIRPORTS).forEach(([code, ap]) => {
    html += `<div style="margin-bottom:24px">`;
    html += `<h3 style="color:${ap.colour};font-size:12px;margin-bottom:8px;padding-bottom:4px;border-bottom:1px solid #1e2d45;">
      ✈ ${ap.name} (${code})</h3>`;
    html += '<table id="compare-table" style="width:100%;border-collapse:collapse;font-size:11px;">';

    // Header: Time Slot | Loc1 🚗 | Loc1 🚆 | Loc2 🚗 | Loc2 🚆 ...
    html += '<tr><th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:left;border:1px solid #2a3a50;">Time Slot</th>';
    compareResults.forEach(entry => {
      html += `<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:left;border:1px solid #2a3a50;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>
        ${entry.name} 🚗</th>`;
      html += `<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:left;border:1px solid #2a3a50;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px"></span>
        ${entry.name} 🚆</th>`;
    });
    html += '</tr>';

    // Rows: one per time slot
    allSlots.forEach(slot => {
      html += `<tr><td style="font-weight:bold;color:#94a3b8;white-space:nowrap;padding:6px 10px;border:1px solid #1e2d45;">${slot.label}</td>`;
      compareResults.forEach(entry => {
        const d       = entry.data[slot.label]?.[code];
        const drive   = d?.DRIVING  != null ? `${d.DRIVING} min`  : '<span style="color:#475569">...</span>';
        const transit = d?.TRANSIT  != null ? `${d.TRANSIT} min`  : '<span style="color:#475569">...</span>';
        html += `<td style="color:#ff8800;padding:6px 10px;border:1px solid #1e2d45;">${drive}</td>`;
        html += `<td style="color:#00ccff;padding:6px 10px;border:1px solid #1e2d45;">${transit}</td>`;
      });
      html += '</tr>';
    });

    html += '</table></div>';
  });

  document.getElementById('compare-body').innerHTML = html;
}
"""

new_content = content[:old_func_start] + new_func + content[old_func_end:]

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print(f'Updated: 3 separate airport tables')
print(f'File size: {len(new_content)//1024} KB')

also choosing to have the random point for each itl3 region assigned to one of the main transportation hubs within there


In [ ]:
from shapely.geometry import Point
import geopandas as gpd

for _, row in itl3_catchment_filtered.iterrows():
    centroid = row.geometry.centroid
    # Find nearest rail station
    rail_projected = rail_dedup.to_crs('EPSG:27700')
    centroid_bng   = gpd.GeoSeries([centroid], crs='EPSG:4326').to_crs('EPSG:27700').iloc[0]
    distances      = rail_projected.geometry.distance(centroid_bng)
    nearest        = rail_dedup.loc[distances.idxmin(), 'stationName']
    print(f'{row["ITL321NM"]:45s} → {nearest}')

some need changing (be careful which ones)
update:

Camden → Kentish Town West (should be King's Cross)
Westminster → Bond Street (should be Victoria or Waterloo)
Hackney and Newham → Leytonstone High Road (should be Stratford)
Tower Hamlets → Shadwell (should be Liverpool Street)
Hertfordshire CC → Welwyn North (should be Watford Junction)
Brent → Wembley Stadium (should be Wembley Central)
Harrow and Hillingdon → South Ruislip (should be Uxbridge or Harrow)


In [ ]:
# Manual overrides for key regions
manual_overrides = {
    'Camden and City of London':              {'name': 'London Kings Cross',    'lat': 51.5308, 'lng': -0.1238},
    'Westminster':                             {'name': 'London Victoria',       'lat': 51.4952, 'lng': -0.1441},
    'Hackney and Newham':                     {'name': 'Stratford',             'lat': 51.5416, 'lng': -0.0042},
    'Tower Hamlets':                          {'name': 'London Liverpool Street','lat': 51.5179, 'lng': -0.0823},
    'Hertfordshire CC':                       {'name': 'Watford Junction',       'lat': 51.6565, 'lng': -0.3965},
    'Brent':                                  {'name': 'Wembley Central',        'lat': 51.5526, 'lng': -0.2963},
    'Harrow and Hillingdon':                  {'name': 'Harrow on the Hill',     'lat': 51.5793, 'lng': -0.3350},
    'Kensington & Chelsea and Hammersmith & Fulham': {'name': 'London Paddington', 'lat': 51.5154, 'lng': -0.1755},
    'Wandsworth':                             {'name': 'Clapham Junction',       'lat': 51.4642, 'lng': -0.1703},
    'Lewisham and Southwark':                 {'name': 'London Bridge',          'lat': 51.5052, 'lng': -0.0864},
    'Lambeth':                                {'name': 'Vauxhall',               'lat': 51.4861, 'lng': -0.1228},
    'Haringey and Islington':                 {'name': 'Finsbury Park',          'lat': 51.5644, 'lng': -0.1057},
    'Bromley':                                {'name': 'Bromley South',          'lat': 51.4012, 'lng': 0.0153},
    'Croydon':                                {'name': 'East Croydon',           'lat': 51.3756, 'lng': -0.0907},
    'Barnet':                                 {'name': 'Barnet',                 'lat': 51.6520, 'lng': -0.1997},
    'Ealing':                                 {'name': 'Ealing Broadway',        'lat': 51.5151, 'lng': -0.3018},
    'Hounslow and Richmond upon Thames':      {'name': 'Richmond',               'lat': 51.4633, 'lng': -0.3015},
    'Oxfordshire CC':                         {'name': 'Oxford',                 'lat': 51.7535, 'lng': -1.2700},
    'Cambridgeshire CC':                      {'name': 'Cambridge',              'lat': 52.1940, 'lng': 0.1374},
    'Milton Keynes':                          {'name': 'Milton Keynes Central',  'lat': 52.0337, 'lng': -0.7717},
    'Buckinghamshire':                        {'name': 'Aylesbury',              'lat': 51.8157, 'lng': -0.8120},
    'Berkshire':                              {'name': 'Reading',                'lat': 51.4585, 'lng': -0.9720},
    'West Surrey':                            {'name': 'Guildford',              'lat': 51.2369, 'lng': -0.5700},
    'East Surrey':                            {'name': 'Redhill',                'lat': 51.2404, 'lng': -0.1699},
    'West Sussex (North East)':               {'name': 'Crawley',                'lat': 51.1091, 'lng': -0.1872},
    'Brighton and Hove':                      {'name': 'Brighton',               'lat': 50.8293, 'lng': -0.1413},
    'Medway':                                 {'name': 'Rochester',              'lat': 51.3881, 'lng': 0.5037},
    'West Kent':                              {'name': 'Sevenoaks',              'lat': 51.2727, 'lng': 0.1877},
    'Luton':                                  {'name': 'Luton',                  'lat': 51.8839, 'lng': -0.4158},
    'Swindon':                                {'name': 'Swindon',                'lat': 51.5663, 'lng': -1.7849},
}

# Build final centroids with overrides
import geopandas as gpd
import pandas as pd

rail_projected = rail_dedup.to_crs('EPSG:27700')

centroids = []
for _, row in itl3_catchment_filtered.iterrows():
    name = row['ITL321NM']
    pop  = int(row['population_2024']) if pd.notna(row['population_2024']) else 0
    gdhi = round(float(row['GDHI_2023_million_gbp']), 1) if pd.notna(row['GDHI_2023_million_gbp']) else 0

    if name in manual_overrides:
        override = manual_overrides[name]
        centroids.append({
            'name':    name,
            'station': override['name'],
            'lat':     override['lat'],
            'lng':     override['lng'],
            'pop':     pop,
            'gdhi':    gdhi
        })
    else:
        # Auto — nearest rail station
        centroid     = row.geometry.centroid
        centroid_bng = gpd.GeoSeries([centroid], crs='EPSG:4326').to_crs('EPSG:27700').iloc[0]
        distances    = rail_projected.geometry.distance(centroid_bng)
        nearest_row  = rail_dedup.loc[distances.idxmin()]
        centroids.append({
            'name':    name,
            'station': nearest_row['stationName'],
            'lat':     nearest_row.geometry.y,
            'lng':     nearest_row.geometry.x,
            'pop':     pop,
            'gdhi':    gdhi
        })

import json
centroids_json = json.dumps(centroids)
print(f'{len(centroids)} centroids updated')
print('\nSample:')
for c in centroids[:5]:
    print(f'  {c["name"]:45s} → {c["station"]}')

adding the option that the user can actually add his flight info to find out more about his flight in the flight panel


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Add search bar to aircraft panel header area
old_aircraft_body = '<div id="aircraft-body"></div>'
new_aircraft_body = '''<div id="aircraft-search" style="padding:10px 20px;border-bottom:1px solid #1e2d45;display:flex;gap:8px;align-items:center;">
  <input type="text" id="flight-search-input" placeholder="Enter flight number e.g. BA123, EZY456..."
    style="flex:1;background:#1e2d45;border:1px solid #2a3a50;color:white;padding:6px 10px;font-size:11px;border-radius:3px;"
    onkeydown="if(event.key==='Enter') searchFlight()"/>
  <button onclick="searchFlight()"
    style="background:#3b82f6;color:white;border:none;padding:6px 14px;font-size:11px;border-radius:3px;cursor:pointer;">
     Search
  </button>
</div>
<div id="flight-search-result" style="padding:0 20px;"></div>
<div id="aircraft-body"></div>'''

content = content.replace(old_aircraft_body, new_aircraft_body)

# Add searchFlight function before closeAircraft
search_func = """function searchFlight() {
  const input  = document.getElementById('flight-search-input').value.trim().toUpperCase();
  const result = document.getElementById('flight-search-result');

  if (!input) { result.innerHTML = ''; return; }

  result.innerHTML = '<p style="color:#f59e0b;font-size:11px;padding:10px 0"> Searching...</p>';

  const KEY = 'affba01de2df68e89abe5d15b2b83a3c';
  const url  = `https://corsproxy.io/?https://api.aviationstack.com/v1/flights?access_key=${KEY}&flight_iata=${input}&limit=3`;

  fetch(url)
    .then(r => r.json())
    .then(data => {
      const flights = data.data || [];
      if (flights.length === 0) {
        result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">No flights found for "${input}". Try a different flight number.</p>`;
        return;
      }

      let html = `<div style="margin:10px 0;padding:10px;background:#0d1525;border-radius:6px;border:1px solid #1e2d45;">`;
      flights.forEach(f => {
        const status    = f.flight_status || 'unknown';
        const sc        = status==='active'?'#22c55e':status==='landed'?'#94a3b8':'#f59e0b';
        const depAirport= f.departure?.airport || '—';
        const arrAirport= f.arrival?.airport   || '—';
        const depIata   = f.departure?.iata    || '—';
        const arrIata   = f.arrival?.iata      || '—';
        const airline   = f.airline?.name      || '—';
        const aircraft  = f.aircraft?.iata     || '—';
        const depSched  = f.departure?.scheduled ? new Date(f.departure.scheduled).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const arrSched  = f.arrival?.scheduled   ? new Date(f.arrival.scheduled).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'})   : '—';
        const depDelay  = f.departure?.delay || 0;
        const arrDelay  = f.arrival?.delay   || 0;

        html += `
          <div style="margin-bottom:12px;padding-bottom:12px;border-bottom:1px solid #1e2d45;">
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
              <span style="font-size:14px;font-weight:bold;color:#e2e8f0">${f.flight?.iata || input}</span>
              <span style="color:${sc};font-size:11px;font-weight:bold">${status.toUpperCase()}</span>
            </div>
            <div style="font-size:11px;color:#94a3b8;margin-bottom:6px">${airline} · ${aircraft}</div>
            <div style="display:grid;grid-template-columns:1fr auto 1fr;gap:8px;align-items:center;margin-bottom:8px;">
              <div>
                <div style="font-size:16px;font-weight:bold;color:#e2e8f0">${depIata}</div>
                <div style="font-size:10px;color:#64748b">${depAirport}</div>
                <div style="font-size:12px;color:#e2e8f0;margin-top:2px">${depSched}</div>
                ${depDelay > 0 ? `<div style="font-size:10px;color:#ef4444">+${depDelay} min delay</div>` : '<div style="font-size:10px;color:#22c55e">On time</div>'}
              </div>
              <div style="text-align:center;color:#64748b;font-size:18px">✈</div>
              <div style="text-align:right;">
                <div style="font-size:16px;font-weight:bold;color:#e2e8f0">${arrIata}</div>
                <div style="font-size:10px;color:#64748b">${arrAirport}</div>
                <div style="font-size:12px;color:#e2e8f0;margin-top:2px">${arrSched}</div>
                ${arrDelay > 0 ? `<div style="font-size:10px;color:#ef4444">+${arrDelay} min delay</div>` : '<div style="font-size:10px;color:#22c55e">On time</div>'}
              </div>
            </div>
          </div>`;
      });
      html += '</div>';
      result.innerHTML = html;
    })
    .catch(e => {
      result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">Could not fetch flight data. Try again.</p>`;
    });
}

"""

content = content.replace(
    'function closeAircraft() {',
    search_func + 'function closeAircraft() {'
)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'Flight search added')
print(f'File size: {len(content)//1024} KB')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Replace corsproxy with allorigins
content = content.replace(
    "const url  = `https://corsproxy.io/?https://api.aviationstack.com/v1/flights?access_key=${KEY}&flight_iata=${input}&limit=3`;",
    "const url  = `https://api.allorigins.win/get?url=${encodeURIComponent(`https://api.aviationstack.com/v1/flights?access_key=${KEY}&flight_iata=${input}&limit=3`)}`);"
)

# Fix the fetch to handle allorigins wrapper
content = content.replace(
    "    .then(r => r.json())\n    .then(data => {\n      const flights = data.data || [];",
    "    .then(r => r.json())\n    .then(wrapper => {\n      const data = JSON.parse(wrapper.contents);\n      const flights = data.data || [];"
)

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print(f'Updated to allorigins proxy')
print(f'allorigins in file: {"allorigins" in content}')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find the searchFlight function
idx = content.find('function searchFlight()')
print(content[idx:idx+400])

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find and replace the entire broken URL + fetch section
old_search = """  const KEY = 'affba01de2df68e89abe5d15b2b83a3c';
  const url  = `h"""

# Find where it ends
idx = content.find(old_search)
# Find the next .then( after this
end = content.find('.then(r => r.json())', idx)
broken_section = content[idx:end]
print('Broken section:')
print(repr(broken_section))

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_section = "  const KEY = 'affba01de2df68e89abe5d15b2b83a3c';\n  const url  = `https://api.allorigins.win/get?url=${encodeURIComponent(`https://api.aviationstack.com/v1/flights?access_key=${KEY}&flight_iata=${input}&limit=3`)}`);\n\n  fetch(url)\n    .then(r => r.json())\n    .then(wrapper => {\n      const data = JSON.parse(wrapper.contents);\n      const flights = data.data || [];"

new_section = """  const KEY = 'affba01de2df68e89abe5d15b2b83a3c';
  const encodedUrl = encodeURIComponent(`https://api.aviationstack.com/v1/flights?access_key=${KEY}&flight_iata=${input}&limit=3`);
  const url = `https://api.allorigins.win/get?url=${encodedUrl}`;

  fetch(url)
    .then(r => r.json())
    .then(wrapper => {
      const data = JSON.parse(wrapper.contents);
      const flights = data.data || [];"""

content = content.replace(old_section, new_section)

# Verify
print(f'Fixed URL in file: {"encodedUrl" in content}')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print('Fixed, i hope')

In [ ]:
import requests

KEY = 'affba01de2df68e89abe5d15b2b83a3c'

# Test allorigins proxy
url = f'https://api.allorigins.win/get?url={requests.utils.quote(f"https://api.aviationstack.com/v1/flights?access_key={KEY}&flight_iata=TG910&limit=3")}'

response = requests.get(url)
print(f'Status: {response.status_code}')
print(response.text[:300])

adding rapid api - aero data box for the flights

In [ ]:
import requests

RAPIDAPI_KEY = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae'

# Test flight search - BA112 (BA London to New York)
url = 'https://aerodatabox.p.rapidapi.com/flights/number/BA112'
headers = {
    'x-rapidapi-key':  RAPIDAPI_KEY,
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com'
}

response = requests.get(url, headers=headers)
print(f'Status: {response.status_code}')
print(response.text[:400])

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Find and replace the entire searchFlight function
start = content.find('function searchFlight()')
end   = content.find('\nfunction closeAircraft()', start)

new_search = """function searchFlight() {
  const input  = document.getElementById('flight-search-input').value.trim().toUpperCase().replace(/\\s/g,'');
  const result = document.getElementById('flight-search-result');

  if (!input) { result.innerHTML = ''; return; }

  result.innerHTML = '<p style="color:#f59e0b;font-size:11px;padding:10px 0"> Searching...</p>';

  const RAPIDAPI_KEY = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae';
  const url     = `https://aerodatabox.p.rapidapi.com/flights/number/${input}`;
  const headers = {
    'x-rapidapi-key':  RAPIDAPI_KEY,
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com'
  };

  fetch(url, {headers})
    .then(r => r.json())
    .then(data => {
      const flights = Array.isArray(data) ? data : [data];
      if (!flights.length || flights[0].message) {
        result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">No flights found for "${input}". Try BA112, EZY8531, TG910 etc.</p>`;
        return;
      }

      let html = '<div style="margin:10px 0;">';
      flights.slice(0,3).forEach(f => {
        const dep        = f.departure || {};
        const arr        = f.arrival   || {};
        const depAirport = dep.airport?.iata  || '—';
        const arrAirport = arr.airport?.iata  || '—';
        const depName    = dep.airport?.name  || '—';
        const arrName    = arr.airport?.name  || '—';
        const depTime    = dep.scheduledTime?.local
          ? new Date(dep.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const arrTime    = arr.scheduledTime?.local
          ? new Date(arr.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const status     = f.status || 'scheduled';
        const sc         = status==='Active'?'#22c55e':status==='Landed'?'#94a3b8':'#f59e0b';
        const airline    = f.airline?.name    || '—';
        const aircraft   = f.aircraft?.model  || f.aircraft?.reg || '—';
        const depDelay   = dep.delay || 0;
        const arrDelay   = arr.delay || 0;
        const distKm     = f.greatCircleDistance?.km ? Math.round(f.greatCircleDistance.km)+' km' : '—';

        html += `
        <div style="padding:12px;background:#0d1525;border-radius:8px;border:1px solid #1e2d45;margin-bottom:8px;">
          <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <span style="font-size:14px;font-weight:bold;color:#e2e8f0">${input}</span>
            <span style="color:${sc};font-size:11px;font-weight:bold;background:${sc}22;padding:2px 8px;border-radius:10px;">${status}</span>
          </div>
          <div style="font-size:11px;color:#64748b;margin-bottom:10px;">${airline} · ${aircraft} · ${distKm}</div>
          <div style="display:grid;grid-template-columns:1fr auto 1fr;gap:8px;align-items:center;">
            <div>
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${depAirport}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${depName}</div>
              <div style="font-size:13px;color:#e2e8f0">${depTime}</div>
              ${depDelay>0
                ? `<div style="font-size:10px;color:#ef4444;margin-top:2px">+${depDelay} min delay</div>`
                : `<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>`}
            </div>
            <div style="text-align:center;">
              <div style="color:#64748b;font-size:20px;">✈</div>
              <div style="font-size:9px;color:#475569;margin-top:2px">${distKm}</div>
            </div>
            <div style="text-align:right;">
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${arrAirport}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${arrName}</div>
              <div style="font-size:13px;color:#e2e8f0">${arrTime}</div>
              ${arrDelay>0
                ? `<div style="font-size:10px;color:#ef4444;margin-top:2px">+${arrDelay} min delay</div>`
                : `<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>`}
            </div>
          </div>
        </div>`;
      });
      html += '</div>';
      result.innerHTML = html;
    })
    .catch(e => {
      result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">Error: ${e.message}</p>`;
    });
}

"""

new_content = content[:start] + new_search + content[end:]

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print(f'Flight search updated with AeroDataBox')
print(f'File size: {len(new_content)//1024} KB')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_end = """      html += '</div>';
      result.innerHTML = html;
    })
    .catch(e => {
      result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">Error: ${e.message}</p>`;
    });
}"""

new_end = """      html += '</div>';
      result.innerHTML = html;

      // ── Animate plane on map ──────────────────────────
      if (window.flightPathLayers) window.flightPathLayers.forEach(l=>l.setMap(null));
      if (window.planeAnimInterval) clearInterval(window.planeAnimInterval);
      window.flightPathLayers = [];

      const f0      = flights[0];
      const depLat  = f0.departure?.airport?.location?.lat;
      const depLon  = f0.departure?.airport?.location?.lon;
      const arrLat  = f0.arrival?.airport?.location?.lat;
      const arrLon  = f0.arrival?.airport?.location?.lon;
      const depIata = f0.departure?.airport?.iata || '';
      const arrIata = f0.arrival?.airport?.iata   || '';
      const depTimeStr = f0.departure?.scheduledTime?.utc;
      const arrTimeStr = f0.arrival?.scheduledTime?.utc;

      if (depLat && arrLat && map) {
        // Airport markers
        [{lat:depLat,lng:depLon,iata:depIata,colour:'#f59e0b'},{lat:arrLat,lng:arrLon,iata:arrIata,colour:'#22c55e'}]
        .forEach(ap => {
          const m = new google.maps.Marker({
            position:{lat:ap.lat,lng:ap.lng}, map,
            icon:{path:google.maps.SymbolPath.CIRCLE,scale:12,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2},
            label:{text:ap.iata,color:'white',fontSize:'9px',fontWeight:'bold'}
          });
          window.flightPathLayers.push(m);
        });

        // Great circle path
        function toRad(d){return d*Math.PI/180;}
        function toDeg(r){return r*180/Math.PI;}
        function gcPoints(la1,lo1,la2,lo2,n){
          const pts=[];
          for(let i=0;i<=n;i++){
            const f=i/n;
            const d=Math.acos(Math.sin(toRad(la1))*Math.sin(toRad(la2))+Math.cos(toRad(la1))*Math.cos(toRad(la2))*Math.cos(toRad(lo2-lo1)));
            const A=Math.sin((1-f)*d)/Math.sin(d)||1-f;
            const B=Math.sin(f*d)/Math.sin(d)||f;
            const x=A*Math.cos(toRad(la1))*Math.cos(toRad(lo1))+B*Math.cos(toRad(la2))*Math.cos(toRad(lo2));
            const y=A*Math.cos(toRad(la1))*Math.sin(toRad(lo1))+B*Math.cos(toRad(la2))*Math.sin(toRad(lo2));
            const z=A*Math.sin(toRad(la1))+B*Math.sin(toRad(la2));
            pts.push({lat:toDeg(Math.atan2(z,Math.sqrt(x*x+y*y))),lng:toDeg(Math.atan2(y,x))});
          }
          return pts;
        }

        const pts = gcPoints(depLat,depLon,arrLat,arrLon,100);

        const pathLine = new google.maps.Polyline({
          path:pts, geodesic:false,
          strokeColor:'#f59e0b', strokeOpacity:0.6, strokeWeight:2, map
        });
        window.flightPathLayers.push(pathLine);

        // Progress based on time
        const now=new Date();
        const dep=depTimeStr?new Date(depTimeStr.replace('Z',' UTC')):null;
        const arr=arrTimeStr?new Date(arrTimeStr.replace('Z',' UTC')):null;
        let prog=0, inFlight=false;
        if(dep&&arr){
          const total=arr-dep, elapsed=now-dep;
          if(elapsed<0) prog=0;
          else if(elapsed>total) prog=1;
          else { prog=elapsed/total; inFlight=true; }
        }

        function getPt(p,f){
          const i=Math.min(Math.floor(f*(p.length-1)),p.length-2);
          const r=f*(p.length-1)-i;
          return {lat:p[i].lat+r*(p[i+1].lat-p[i].lat),lng:p[i].lng+r*(p[i+1].lng-p[i].lng)};
        }
        function bearing(p1,p2){
          const dL=toRad(p2.lng-p1.lng);
          const y=Math.sin(dL)*Math.cos(toRad(p2.lat));
          const x=Math.cos(toRad(p1.lat))*Math.sin(toRad(p2.lat))-Math.sin(toRad(p1.lat))*Math.cos(toRad(p2.lat))*Math.cos(dL);
          return(toDeg(Math.atan2(y,x))+360)%360;
        }

        const planeIcon = {path:'M 0,-3 L 1.5,1 L 0,0 L -1.5,1 Z',scale:7,fillColor:'#ffffff',fillOpacity:1,strokeColor:'#f59e0b',strokeWeight:1.5,rotation:0,anchor:new google.maps.Point(0,0)};
        const plane = new google.maps.Marker({position:getPt(pts,prog),map,icon:planeIcon,zIndex:999});
        window.flightPathLayers.push(plane);

        let p=prog;
        const spd=inFlight?0.0001:0.0005;
        window.planeAnimInterval=setInterval(()=>{
          p+=spd; if(p>=1) p=0;
          const pos=getPt(pts,p), pos2=getPt(pts,Math.min(p+0.001,1));
          plane.setPosition(pos);
          plane.setIcon({...planeIcon,rotation:bearing(pos,pos2)});
        },50);

        const bounds=new google.maps.LatLngBounds();
        pts.forEach(pt=>bounds.extend(pt));
        map.fitBounds(bounds,{top:80,bottom:80,left:80,right:80});

        const statusTxt = prog<=0?'🕐 Not yet departed':prog>=1?'🛬 Landed':`✈ In flight — ${Math.round(prog*100)}% complete`;
        result.innerHTML+=`<div style="margin-top:8px;font-size:10px;color:#94a3b8;display:flex;gap:10px;align-items:center;">
          <span>${statusTxt}</span>
          <button onclick="clearFlightPath()" style="background:#475569;color:white;border:none;padding:3px 10px;font-size:10px;border-radius:3px;cursor:pointer;">✕ Clear</button>
        </div>`;
      }
    })
    .catch(e => {
      result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">Error: ${e.message}</p>`;
    });
}

function clearFlightPath() {
  if (window.flightPathLayers) { window.flightPathLayers.forEach(l=>l.setMap(null)); window.flightPathLayers=[]; }
  if (window.planeAnimInterval) { clearInterval(window.planeAnimInterval); window.planeAnimInterval=null; }
  document.getElementById('flight-search-result').innerHTML='';
  document.getElementById('flight-search-input').value='';
}"""

new_content = content.replace(old_end, new_end)

print(f'flightPathLayers in file: {"flightPathLayers" in new_content}')
print(f'planeAnimInterval in file: {"planeAnimInterval" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Done')

In [ ]:
import requests

RAPIDAPI_KEY = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae'

# Test FIDS for Heathrow arrivals
url = 'https://aerodatabox.p.rapidapi.com/flights/airports/iata/LHR'
headers = {
    'x-rapidapi-key':  RAPIDAPI_KEY,
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com'
}
params = {
    'withLeg':        'true',
    'direction':      'Arrival',
    'withCancelled':  'false',
    'withCodeshared': 'false',
    'withLocation':   'false'
}

response = requests.get(url, headers=headers, params=params)
print(f'Status: {response.status_code}')
print(response.text[:500])

works, updating the flight info panel to an hourly rate


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Add RapidAPI key constant after TOMTOM_KEY
content = content.replace(
    "const TOMTOM_KEY     = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj';",
    "const TOMTOM_KEY     = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj';\nconst RAPIDAPI_KEY   = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae';"
)

# Replace the entire openAircraft function with live FIDS version
start = content.find('function openAircraft()')
end   = content.find('\nfunction closeAircraft()', start)

new_aircraft = """function openAircraft() {
  document.getElementById('aircraft-overlay').classList.add('open');
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">⏳ Loading live flight data...</div>';

  const codes   = Object.keys(AIRPORTS);
  const results = {};
  let   done    = 0;

  codes.forEach(code => {
    // Fetch arrivals and departures in parallel for each airport
    const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
    const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
    const hdrs   = { 'x-rapidapi-key': RAPIDAPI_KEY, 'x-rapidapi-host': 'aerodatabox.p.rapidapi.com' };

    results[code] = { arrivals: [], departures: [] };

    Promise.all([
      fetch(arrUrl, {headers: hdrs}).then(r => r.json()),
      fetch(depUrl, {headers: hdrs}).then(r => r.json())
    ]).then(([arrData, depData]) => {
      results[code].arrivals   = (arrData.arrivals   || []).slice(0, 8);
      results[code].departures = (depData.departures || []).slice(0, 8);
      done++;
      if (done === codes.length) renderFIDS(results);
    }).catch(() => {
      done++;
      if (done === codes.length) renderFIDS(results);
    });
  });
}

function renderFIDS(results) {
  const body = document.getElementById('aircraft-body');
  const now  = new Date().toLocaleTimeString('en-GB', {hour:'2-digit', minute:'2-digit'});
  let html   = `<p style="color:#64748b;font-size:10px;margin-bottom:12px">🔴 Live data — updated at ${now}</p>`;

  Object.entries(AIRPORTS).forEach(([code, ap]) => {
    const arr = results[code]?.arrivals   || [];
    const dep = results[code]?.departures || [];

    html += `<div class="aircraft-airport">
      <h3 style="color:${ap.colour}">✈ ${ap.name} (${code})</h3>`;

    // Arrivals table
    html += '<b style="font-size:11px;color:#94a3b8">🛬 Arrivals</b>';
    html += '<table class="aircraft-table"><tr><th>Flight</th><th>From</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';

    if (arr.length === 0) {
      html += '<tr><td colspan="6" style="color:#64748b;text-align:center">No arrivals found</td></tr>';
    } else {
      arr.forEach(f => {
        const flight   = f.number || '—';
        const from     = f.departure?.airport?.iata || '—';
        const sched    = f.arrival?.scheduledTime?.local
          ? new Date(f.arrival.scheduledTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const expected = f.arrival?.revisedTime?.local
          ? new Date(f.arrival.revisedTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'})
          : f.arrival?.runwayTime?.local
          ? new Date(f.arrival.runwayTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const terminal = f.arrival?.terminal || '—';
        const status   = f.arrival?.runwayTime ? 'Landed' : f.arrival?.revisedTime ? 'On approach' : 'Scheduled';
        const sc       = status==='Landed'?'#94a3b8':status==='On approach'?'#22c55e':'#f59e0b';
        html += `<tr>
          <td style="font-weight:bold">${flight}</td>
          <td>${from}</td>
          <td>${sched}</td>
          <td>${expected}</td>
          <td>${terminal}</td>
          <td style="color:${sc}">${status}</td>
        </tr>`;
      });
    }
    html += '</table>';

    // Departures table
    html += '<b style="font-size:11px;color:#94a3b8;display:block;margin-top:10px">🛫 Departures</b>';
    html += '<table class="aircraft-table"><tr><th>Flight</th><th>To</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';

    if (dep.length === 0) {
      html += '<tr><td colspan="6" style="color:#64748b;text-align:center">No departures found</td></tr>';
    } else {
      dep.forEach(f => {
        const flight   = f.number || '—';
        const to       = f.arrival?.airport?.iata || '—';
        const sched    = f.departure?.scheduledTime?.local
          ? new Date(f.departure.scheduledTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const expected = f.departure?.revisedTime?.local
          ? new Date(f.departure.revisedTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'})
          : f.departure?.runwayTime?.local
          ? new Date(f.departure.runwayTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const terminal = f.departure?.terminal || '—';
        const status   = f.departure?.runwayTime ? 'Departed' : f.departure?.revisedTime ? 'Boarding' : 'Scheduled';
        const sc       = status==='Departed'?'#94a3b8':status==='Boarding'?'#22c55e':'#f59e0b';
        html += `<tr>
          <td style="font-weight:bold">${flight}</td>
          <td>${to}</td>
          <td>${sched}</td>
          <td>${expected}</td>
          <td>${terminal}</td>
          <td style="color:${sc}">${status}</td>
        </tr>`;
      });
    }
    html += '</table></div>';
  });

  document.getElementById('aircraft-body').innerHTML = html;
}"""

new_content = content[:start] + new_aircraft + content[end:]

print(f'FIDS in file: {"renderFIDS" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Done, live FIDS data - not static data anymore')

comparing table layout is shown in a continous table, trying to make them separate so it would be easier to compare


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_update = content.find('function updateCompareTable()')
old_update_end = content.find('\nasync function fetchCompareData', old_update)

new_update = """function updateCompareTable() {
  if (compareResults.length === 0) return;

  const allSlots = [];
  compareResults.forEach(entry => {
    (entry.slots || []).forEach(s => {
      if (!allSlots.find(x => x.label === s.label)) allSlots.push(s);
    });
  });
  if (allSlots.length === 0) return;

  let html = '';

  // One table per airport, stacked vertically
  Object.entries(AIRPORTS).forEach(([code, ap]) => {
    html += `<div style="margin-bottom:28px;">`;
    html += `<h3 style="color:${ap.colour};font-size:12px;margin-bottom:8px;
      padding:6px 10px;background:#0d1525;border-radius:6px;
      border-left:3px solid ${ap.colour};">
      ✈ ${ap.name} (${code})</h3>`;
    html += '<div style="overflow-x:auto;">';
    html += '<table style="width:100%;border-collapse:collapse;font-size:11px;">';

    // Header row
    html += '<tr><th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:left;border:1px solid #2a3a50;white-space:nowrap;">Time Slot</th>';
    compareResults.forEach(entry => {
      html += `<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:center;border:1px solid #2a3a50;white-space:nowrap;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px;"></span>
        ${entry.name}<br>
        <span style="font-size:9px;color:#64748b;">🚗 Drive</span>
      </th>`;
      html += `<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:center;border:1px solid #2a3a50;white-space:nowrap;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${entry.colour};margin-right:4px;"></span>
        ${entry.name}<br>
        <span style="font-size:9px;color:#64748b;">🚆 Transit</span>
      </th>`;
    });
    html += '</tr>';

    // One row per time slot
    allSlots.forEach((slot, si) => {
      const rowBg = si % 2 === 0 ? 'background:#111827' : 'background:#0d1525';
      html += `<tr><td style="${rowBg};font-weight:bold;color:#94a3b8;padding:8px 10px;border:1px solid #1e2d45;white-space:nowrap;">${slot.label}</td>`;
      compareResults.forEach(entry => {
        const d       = entry.data[slot.label]?.[code];
        const drive   = d?.DRIVING  != null
          ? `<span style="color:#ff8800;font-weight:bold">${d.DRIVING} min</span>`
          : '<span style="color:#475569">...</span>';
        const transit = d?.TRANSIT  != null
          ? `<span style="color:#00ccff;font-weight:bold">${d.TRANSIT} min</span>`
          : '<span style="color:#475569">...</span>';
        html += `<td style="${rowBg};padding:8px 10px;border:1px solid #1e2d45;text-align:center;">${drive}</td>`;
        html += `<td style="${rowBg};padding:8px 10px;border:1px solid #1e2d45;text-align:center;">${transit}</td>`;
      });
      html += '</tr>';
    });

    html += '</table></div></div>';
  });

  document.getElementById('compare-body').innerHTML = html;
}
"""

new_content = content[:old_update] + new_update + content[old_update_end:]

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print(f'Updated: 3 separate vertical airport tables')
print(f'File size: {len(new_content)//1024} KB')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

script_opens  = content.count('<script')
script_closes = content.count('</script>')
print(f'<script> tags:  {script_opens}')
print(f'</script> tags: {script_closes}')
print(f'File size: {len(content)//1024} KB')

# Check end of file
print(f'\nFile ends with:')
print(repr(content[-100:]))

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Fix the end of file
content = content.rstrip()
if content.endswith('>') and not content.endswith('</html>'):
    content = content[:-1].rstrip()

# Add missing closing tags
if '</script>' not in content:
    content += '\n</script>\n</body>\n</html>'

print(f'</script> count: {content.count("</script>")}')
print(f'File ends with: {repr(content[-50:])}')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(content)

print('Fixed')

comparison table works as intended
but
flight info panel stopped working all together

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

print(f'openAircraft in file: {"function openAircraft" in content}')
print(f'renderFIDS in file: {"function renderFIDS" in content}')
print(f'closeAircraft in file: {"function closeAircraft" in content}')
print(f'</script> count: {content.count("</script>")}')
print(f'File size: {len(content)//1024} KB')

# Show end of file
print(f'\nLast 200 chars:')
print(repr(content[-200:]))

functions got cut off when i tried to update the comparison tables,
re-adding them


In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Insert missing functions before </script>
missing_functions = """
function openAircraft() {
  document.getElementById('aircraft-overlay').classList.add('open');
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">⏳ Loading live flight data...</div>';

  const codes   = Object.keys(AIRPORTS);
  const results = {};
  let   done    = 0;

  codes.forEach(code => {
    const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
    const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
    const hdrs   = { 'x-rapidapi-key': RAPIDAPI_KEY, 'x-rapidapi-host': 'aerodatabox.p.rapidapi.com' };

    results[code] = { arrivals: [], departures: [] };

    Promise.all([
      fetch(arrUrl, {headers: hdrs}).then(r => r.json()),
      fetch(depUrl, {headers: hdrs}).then(r => r.json())
    ]).then(([arrData, depData]) => {
      results[code].arrivals   = (arrData.arrivals   || []).slice(0, 8);
      results[code].departures = (depData.departures || []).slice(0, 8);
      done++;
      if (done === codes.length) renderFIDS(results);
    }).catch(() => {
      done++;
      if (done === codes.length) renderFIDS(results);
    });
  });
}

function renderFIDS(results) {
  const body = document.getElementById('aircraft-body');
  const now  = new Date().toLocaleTimeString('en-GB', {hour:'2-digit', minute:'2-digit'});
  let html   = `<p style="color:#64748b;font-size:10px;margin-bottom:12px">🔴 Live data — updated at ${now}</p>`;

  Object.entries(AIRPORTS).forEach(([code, ap]) => {
    const arr = results[code]?.arrivals   || [];
    const dep = results[code]?.departures || [];

    html += `<div class="aircraft-airport"><h3 style="color:${ap.colour}">✈ ${ap.name} (${code})</h3>`;
    html += '<b style="font-size:11px;color:#94a3b8">🛬 Arrivals</b>';
    html += '<table class="aircraft-table"><tr><th>Flight</th><th>From</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';

    if (arr.length === 0) {
      html += '<tr><td colspan="6" style="color:#64748b;text-align:center">No arrivals found</td></tr>';
    } else {
      arr.forEach(f => {
        const flight   = f.number || '—';
        const from     = f.departure?.airport?.iata || '—';
        const sched    = f.arrival?.scheduledTime?.local ? new Date(f.arrival.scheduledTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const expected = f.arrival?.revisedTime?.local  ? new Date(f.arrival.revisedTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'})
                       : f.arrival?.runwayTime?.local   ? new Date(f.arrival.runwayTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const terminal = f.arrival?.terminal || '—';
        const status   = f.arrival?.runwayTime ? 'Landed' : f.arrival?.revisedTime ? 'On approach' : 'Scheduled';
        const sc       = status==='Landed'?'#94a3b8':status==='On approach'?'#22c55e':'#f59e0b';
        html += `<tr><td style="font-weight:bold">${flight}</td><td>${from}</td><td>${sched}</td><td>${expected}</td><td>${terminal}</td><td style="color:${sc}">${status}</td></tr>`;
      });
    }
    html += '</table>';

    html += '<b style="font-size:11px;color:#94a3b8;display:block;margin-top:10px">🛫 Departures</b>';
    html += '<table class="aircraft-table"><tr><th>Flight</th><th>To</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';

    if (dep.length === 0) {
      html += '<tr><td colspan="6" style="color:#64748b;text-align:center">No departures found</td></tr>';
    } else {
      dep.forEach(f => {
        const flight   = f.number || '—';
        const to       = f.arrival?.airport?.iata || '—';
        const sched    = f.departure?.scheduledTime?.local ? new Date(f.departure.scheduledTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const expected = f.departure?.revisedTime?.local   ? new Date(f.departure.revisedTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'})
                       : f.departure?.runwayTime?.local    ? new Date(f.departure.runwayTime.local).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const terminal = f.departure?.terminal || '—';
        const status   = f.departure?.runwayTime ? 'Departed' : f.departure?.revisedTime ? 'Boarding' : 'Scheduled';
        const sc       = status==='Departed'?'#94a3b8':status==='Boarding'?'#22c55e':'#f59e0b';
        html += `<tr><td style="font-weight:bold">${flight}</td><td>${to}</td><td>${sched}</td><td>${expected}</td><td>${terminal}</td><td style="color:${sc}">${status}</td></tr>`;
      });
    }
    html += '</table></div>';
  });

  document.getElementById('aircraft-body').innerHTML = html;
}

function closeAircraft() {
  document.getElementById('aircraft-overlay').classList.remove('open');
}

function searchFlight() {
  const input  = document.getElementById('flight-search-input').value.trim().toUpperCase().replace(/\\s/g,'');
  const result = document.getElementById('flight-search-result');
  if (!input) { result.innerHTML = ''; return; }
  result.innerHTML = '<p style="color:#f59e0b;font-size:11px;padding:10px 0">⏳ Searching...</p>';

  const RAPIDAPI_KEY_SEARCH = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae';
  const url     = `https://aerodatabox.p.rapidapi.com/flights/number/${input}`;
  const headers = { 'x-rapidapi-key': RAPIDAPI_KEY_SEARCH, 'x-rapidapi-host': 'aerodatabox.p.rapidapi.com' };

  fetch(url, {headers})
    .then(r => r.json())
    .then(data => {
      const flights = Array.isArray(data) ? data : [data];
      if (!flights.length || flights[0].message) {
        result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">No flights found for "${input}".</p>`;
        return;
      }

      let html = '<div style="margin:10px 0;">';
      flights.slice(0,3).forEach(f => {
        const dep        = f.departure || {};
        const arr        = f.arrival   || {};
        const depIata    = dep.airport?.iata || '—';
        const arrIata    = arr.airport?.iata || '—';
        const depName    = dep.airport?.name || '—';
        const arrName    = arr.airport?.name || '—';
        const depTime    = dep.scheduledTime?.local ? new Date(dep.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const arrTime    = arr.scheduledTime?.local ? new Date(arr.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{hour:'2-digit',minute:'2-digit'}) : '—';
        const status     = f.status || 'scheduled';
        const sc         = status==='Active'?'#22c55e':status==='Landed'?'#94a3b8':'#f59e0b';
        const airline    = f.airline?.name   || '—';
        const aircraft   = f.aircraft?.model || f.aircraft?.reg || '—';
        const depDelay   = dep.delay || 0;
        const arrDelay   = arr.delay || 0;
        const distKm     = f.greatCircleDistance?.km ? Math.round(f.greatCircleDistance.km)+' km' : '—';

        html += `<div style="padding:12px;background:#0d1525;border-radius:8px;border:1px solid #1e2d45;margin-bottom:8px;">
          <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <span style="font-size:14px;font-weight:bold;color:#e2e8f0">${input}</span>
            <span style="color:${sc};font-size:11px;font-weight:bold;background:${sc}22;padding:2px 8px;border-radius:10px;">${status}</span>
          </div>
          <div style="font-size:11px;color:#64748b;margin-bottom:10px;">${airline} · ${aircraft} · ${distKm}</div>
          <div style="display:grid;grid-template-columns:1fr auto 1fr;gap:8px;align-items:center;">
            <div>
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${depIata}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${depName}</div>
              <div style="font-size:13px;color:#e2e8f0">${depTime}</div>
              ${depDelay>0?`<div style="font-size:10px;color:#ef4444;margin-top:2px">+${depDelay} min delay</div>`:`<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>`}
            </div>
            <div style="text-align:center;">
              <div style="color:#64748b;font-size:20px;">✈</div>
              <div style="font-size:9px;color:#475569;margin-top:2px">${distKm}</div>
            </div>
            <div style="text-align:right;">
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${arrIata}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${arrName}</div>
              <div style="font-size:13px;color:#e2e8f0">${arrTime}</div>
              ${arrDelay>0?`<div style="font-size:10px;color:#ef4444;margin-top:2px">+${arrDelay} min delay</div>`:`<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>`}
            </div>
          </div>
        </div>`;
      });
      html += '</div>';
      result.innerHTML = html;

      // Animate plane on map
      if (window.flightPathLayers) window.flightPathLayers.forEach(l=>l.setMap(null));
      if (window.planeAnimInterval) clearInterval(window.planeAnimInterval);
      window.flightPathLayers = [];

      const f0      = flights[0];
      const depLat  = f0.departure?.airport?.location?.lat;
      const depLon  = f0.departure?.airport?.location?.lon;
      const arrLat  = f0.arrival?.airport?.location?.lat;
      const arrLon  = f0.arrival?.airport?.location?.lon;
      const depIata2 = f0.departure?.airport?.iata || '';
      const arrIata2 = f0.arrival?.airport?.iata   || '';
      const depTimeStr = f0.departure?.scheduledTime?.utc;
      const arrTimeStr = f0.arrival?.scheduledTime?.utc;

      if (depLat && arrLat && map) {
        [{lat:depLat,lng:depLon,iata:depIata2,colour:'#f59e0b'},{lat:arrLat,lng:arrLon,iata:arrIata2,colour:'#22c55e'}]
        .forEach(ap => {
          const m = new google.maps.Marker({
            position:{lat:ap.lat,lng:ap.lng}, map,
            icon:{path:google.maps.SymbolPath.CIRCLE,scale:12,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2},
            label:{text:ap.iata,color:'white',fontSize:'9px',fontWeight:'bold'}
          });
          window.flightPathLayers.push(m);
        });

        function toRad(d){return d*Math.PI/180;}
        function toDeg(r){return r*180/Math.PI;}
        function gcPoints(la1,lo1,la2,lo2,n){
          const pts=[];
          for(let i=0;i<=n;i++){
            const f=i/n;
            const d=Math.acos(Math.min(1,Math.sin(toRad(la1))*Math.sin(toRad(la2))+Math.cos(toRad(la1))*Math.cos(toRad(la2))*Math.cos(toRad(lo2-lo1))));
            const A=d===0?1-f:Math.sin((1-f)*d)/Math.sin(d);
            const B=d===0?f:Math.sin(f*d)/Math.sin(d);
            const x=A*Math.cos(toRad(la1))*Math.cos(toRad(lo1))+B*Math.cos(toRad(la2))*Math.cos(toRad(lo2));
            const y=A*Math.cos(toRad(la1))*Math.sin(toRad(lo1))+B*Math.cos(toRad(la2))*Math.sin(toRad(lo2));
            const z=A*Math.sin(toRad(la1))+B*Math.sin(toRad(la2));
            pts.push({lat:toDeg(Math.atan2(z,Math.sqrt(x*x+y*y))),lng:toDeg(Math.atan2(y,x))});
          }
          return pts;
        }

        const pts = gcPoints(depLat,depLon,arrLat,arrLon,100);
        const pathLine = new google.maps.Polyline({path:pts,geodesic:false,strokeColor:'#f59e0b',strokeOpacity:0.6,strokeWeight:2,map});
        window.flightPathLayers.push(pathLine);

        const now2=new Date();
        const dep2=depTimeStr?new Date(depTimeStr.replace('Z',' UTC')):null;
        const arr2=arrTimeStr?new Date(arrTimeStr.replace('Z',' UTC')):null;
        let prog=0, inFlight=false;
        if(dep2&&arr2){
          const total=arr2-dep2, elapsed=now2-dep2;
          if(elapsed<0) prog=0;
          else if(elapsed>total) prog=1;
          else { prog=elapsed/total; inFlight=true; }
        }

        function getPt(p,f){const i=Math.min(Math.floor(f*(p.length-1)),p.length-2);const r=f*(p.length-1)-i;return {lat:p[i].lat+r*(p[i+1].lat-p[i].lat),lng:p[i].lng+r*(p[i+1].lng-p[i].lng)};}
        function bearing(p1,p2){const dL=toRad(p2.lng-p1.lng);const y=Math.sin(dL)*Math.cos(toRad(p2.lat));const x=Math.cos(toRad(p1.lat))*Math.sin(toRad(p2.lat))-Math.sin(toRad(p1.lat))*Math.cos(toRad(p2.lat))*Math.cos(dL);return(toDeg(Math.atan2(y,x))+360)%360;}

        const planeIcon={path:'M 0,-3 L 1.5,1 L 0,0 L -1.5,1 Z',scale:7,fillColor:'#ffffff',fillOpacity:1,strokeColor:'#f59e0b',strokeWeight:1.5,rotation:0,anchor:new google.maps.Point(0,0)};
        const plane=new google.maps.Marker({position:getPt(pts,prog),map,icon:planeIcon,zIndex:999});
        window.flightPathLayers.push(plane);

        let p=prog;
        const spd=inFlight?0.0001:0.0005;
        window.planeAnimInterval=setInterval(()=>{
          p+=spd; if(p>=1) p=0;
          const pos=getPt(pts,p),pos2=getPt(pts,Math.min(p+0.001,1));
          plane.setPosition(pos);
          plane.setIcon({...planeIcon,rotation:bearing(pos,pos2)});
        },50);

        const bounds=new google.maps.LatLngBounds();
        pts.forEach(pt=>bounds.extend(pt));
        map.fitBounds(bounds,{top:80,bottom:80,left:80,right:80});

        const statusTxt=prog<=0?'🕐 Not yet departed':prog>=1?'🛬 Landed':`✈ In flight — ${Math.round(prog*100)}% complete`;
        result.innerHTML+=`<div style="margin-top:8px;font-size:10px;color:#94a3b8;display:flex;gap:10px;align-items:center;">
          <span>${statusTxt}</span>
          <button onclick="clearFlightPath()" style="background:#475569;color:white;border:none;padding:3px 10px;font-size:10px;border-radius:3px;cursor:pointer;">✕ Clear</button>
        </div>`;
      }
    })
    .catch(e => {
      result.innerHTML = `<p style="color:#ef4444;font-size:11px;padding:10px 0">Error: ${e.message}</p>`;
    });
}

function clearFlightPath() {
  if (window.flightPathLayers) { window.flightPathLayers.forEach(l=>l.setMap(null)); window.flightPathLayers=[]; }
  if (window.planeAnimInterval) { clearInterval(window.planeAnimInterval); window.planeAnimInterval=null; }
  document.getElementById('flight-search-result').innerHTML='';
  document.getElementById('flight-search-input').value='';
}
"""

# Insert before </script>
new_content = content.replace('</script>\n</body>\n</html>', missing_functions + '\n</script>\n</body>\n</html>')

print(f'openAircraft in file: {"function openAircraft" in new_content}')
print(f'renderFIDS in file: {"function renderFIDS" in new_content}')
print(f'closeAircraft in file: {"function closeAircraft" in new_content}')
print(f'searchFlight in file: {"function searchFlight" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Fixed')

flight info is not all showing, might be the free tier limits
trying to do sequentially

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_open = """function openAircraft() {
  document.getElementById('aircraft-overlay').classList.add('open');
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">⏳ Loading live flight data...</div>';

  const codes   = Object.keys(AIRPORTS);
  const results = {};
  let   done    = 0;

  codes.forEach(code => {
    const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
    const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
    const hdrs   = { 'x-rapidapi-key': RAPIDAPI_KEY, 'x-rapidapi-host': 'aerodatabox.p.rapidapi.com' };

    results[code] = { arrivals: [], departures: [] };

    Promise.all([
      fetch(arrUrl, {headers: hdrs}).then(r => r.json()),
      fetch(depUrl, {headers: hdrs}).then(r => r.json())
    ]).then(([arrData, depData]) => {
      results[code].arrivals   = (arrData.arrivals   || []).slice(0, 8);
      results[code].departures = (depData.departures || []).slice(0, 8);
      done++;
      if (done === codes.length) renderFIDS(results);
    }).catch(() => {
      done++;
      if (done === codes.length) renderFIDS(results);
    });
  });
}"""

new_open = """async function openAircraft() {
  document.getElementById('aircraft-overlay').classList.add('open');
  const body = document.getElementById('aircraft-body');
  body.innerHTML = '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">⏳ Loading live flight data...</div>';

  const codes   = Object.keys(AIRPORTS);
  const results = {};
  const hdrs    = { 'x-rapidapi-key': RAPIDAPI_KEY, 'x-rapidapi-host': 'aerodatabox.p.rapidapi.com' };
  const delay   = ms => new Promise(r => setTimeout(r, ms));

  for (const code of codes) {
    results[code] = { arrivals: [], departures: [] };

    try {
      const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes = await fetch(arrUrl, {headers: hdrs});
      const arrData = await arrRes.json();
      results[code].arrivals = (arrData.arrivals || []).slice(0, 8);
    } catch(e) { results[code].arrivals = []; }

    await delay(500);

    try {
      const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes = await fetch(depUrl, {headers: hdrs});
      const depData = await depRes.json();
      results[code].departures = (depData.departures || []).slice(0, 8);
    } catch(e) { results[code].departures = []; }

    await delay(500);

    // Show partial results as they load
    renderFIDS(results);
  }

  renderFIDS(results);
}"""

new_content = content.replace(old_open, new_open)

print(f'async openAircraft in file: {"async function openAircraft" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Fixed: sequential fetching with 500ms delay')

In [ ]:
import requests
from datetime import datetime, timedelta

RAPIDAPI_KEY = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae'
hdrs = {
    'x-rapidapi-key':  RAPIDAPI_KEY,
    'x-rapidapi-host': 'aerodatabox.p.rapidapi.com'
}

# Try with time window - next 3 hours
now     = datetime.utcnow()
future  = now + timedelta(hours=3)
from_t  = now.strftime('%Y-%m-%dT%H:%M')
to_t    = future.strftime('%Y-%m-%dT%H:%M')

url = f'https://aerodatabox.p.rapidapi.com/flights/airports/iata/LHR/{from_t}/{to_t}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false'

res = requests.get(url, headers=hdrs)
print(f'Status: {res.status_code}')
print(res.text[:500])

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Replace the fetch URLs in openAircraft to use time window
old_urls = """    try {
      const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes = await fetch(arrUrl, {headers: hdrs});
      const arrData = await arrRes.json();
      results[code].arrivals = (arrData.arrivals || []).slice(0, 8);
    } catch(e) { results[code].arrivals = []; }

    await delay(500);

    try {
      const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes = await fetch(depUrl, {headers: hdrs});
      const depData = await depRes.json();
      results[code].departures = (depData.departures || []).slice(0, 8);
    } catch(e) { results[code].departures = []; }"""

new_urls = """    try {
      // Time window: last 1 hour to next 3 hours
      const now   = new Date();
      const from  = new Date(now.getTime() - 60*60*1000);
      const to    = new Date(now.getTime() + 3*60*60*1000);
      const fmt   = d => d.toISOString().slice(0,16);
      const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}/${fmt(from)}/${fmt(to)}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes  = await fetch(arrUrl, {headers: hdrs});
      const arrData = await arrRes.json();
      results[code].arrivals = (arrData.arrivals || []).slice(0, 8);
    } catch(e) { results[code].arrivals = []; }

    await delay(500);

    try {
      const now   = new Date();
      const from  = new Date(now.getTime() - 60*60*1000);
      const to    = new Date(now.getTime() + 3*60*60*1000);
      const fmt   = d => d.toISOString().slice(0,16);
      const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}/${fmt(from)}/${fmt(to)}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes  = await fetch(depUrl, {headers: hdrs});
      const depData = await depRes.json();
      results[code].departures = (depData.departures || []).slice(0, 8);
    } catch(e) { results[code].departures = []; }"""

new_content = content.replace(old_urls, new_urls)

print(f'Time window in file: {"getTime() - 60" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Updated: shows flights from last 1hr to next 3hrs')

In [ ]:
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

# Replace the fetch URLs in openAircraft to use time window
old_urls = """    try {
      const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes = await fetch(arrUrl, {headers: hdrs});
      const arrData = await arrRes.json();
      results[code].arrivals = (arrData.arrivals || []).slice(0, 8);
    } catch(e) { results[code].arrivals = []; }

    await delay(500);

    try {
      const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes = await fetch(depUrl, {headers: hdrs});
      const depData = await depRes.json();
      results[code].departures = (depData.departures || []).slice(0, 8);
    } catch(e) { results[code].departures = []; }"""

new_urls = """    try {
      // Time window: last 1 hour to next 3 hours
      const now   = new Date();
      const from  = new Date(now.getTime() - 60*60*1000);
      const to    = new Date(now.getTime() + 3*60*60*1000);
      const fmt   = d => d.toISOString().slice(0,16);
      const arrUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}/${fmt(from)}/${fmt(to)}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes  = await fetch(arrUrl, {headers: hdrs});
      const arrData = await arrRes.json();
      results[code].arrivals = (arrData.arrivals || []).slice(0, 8);
    } catch(e) { results[code].arrivals = []; }

    await delay(500);

    try {
      const now   = new Date();
      const from  = new Date(now.getTime() - 60*60*1000);
      const to    = new Date(now.getTime() + 3*60*60*1000);
      const fmt   = d => d.toISOString().slice(0,16);
      const depUrl = `https://aerodatabox.p.rapidapi.com/flights/airports/iata/${code}/${fmt(from)}/${fmt(to)}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes  = await fetch(depUrl, {headers: hdrs});
      const depData = await depRes.json();
      results[code].departures = (depData.departures || []).slice(0, 8);
    } catch(e) { results[code].departures = []; }"""

new_content = content.replace(old_urls, new_urls)

print(f'Time window in file: {"getTime() - 60" in new_content}')
print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Updated: shows flights from last 1hr to next 3hrs')

In [ ]:
# increasing delay times for the flight info panel from 500 to 1500
with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'r') as f:
    content = f.read()

old_delay = "    await delay(500);\n\n    try {\n      const now   = new Date();\n      const from  = new Date(now.getTime() - 60*60*1000);\n      const to    = new Date(now.getTime() + 3*60*60*1000);\n      const fmt   = d => d.toISOString().slice(0,16);\n      const depUrl"

new_delay = "    await delay(1500);\n\n    try {\n      const now   = new Date();\n      const from  = new Date(now.getTime() - 60*60*1000);\n      const to    = new Date(now.getTime() + 3*60*60*1000);\n      const fmt   = d => d.toISOString().slice(0,16);\n      const depUrl"

new_content = content.replace(old_delay, new_delay)

# Also increase delay between airports
new_content = new_content.replace(
    '    // Show partial results as they load\n    renderFIDS(results);\n  }',
    '    await delay(1500);\n    // Show partial results as they load\n    renderFIDS(results);\n  }'
)

print(f'File size: {len(new_content)//1024} KB')

with open('/content/drive/MyDrive/airport_journey_planner_FINAL.html', 'w') as f:
    f.write(new_content)

print('Delays increased to 1500ms')

In [ ]:
import json

print(f'iso_geojson: {len(json.dumps(iso_geojson)):,} chars')
print(f'itl3_json:   {len(json.dumps(itl3_json)):,} chars')
print(f'centroids:   {len(centroids_json):,} chars')
print(f'slim_data:   {len(json.dumps(slim_data)):,} chars')

# Check station field
sample = json.loads(centroids_json)[0]
print(f'\nSample centroid: {sample}')

In [ ]:
import json

flight_data_str = json.dumps(slim_data, ensure_ascii=True)
TOMTOM_KEY      = '5h5VNTeESb9hHq8iRJL0TKPgIUnxK5Oj'
RAPIDAPI_KEY    = '62ec75e40fmshf9a380b22d839a1p19eccdjsn1ce61fdce4ae'

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>London Airport Catchment — Journey Planner</title>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:Arial,sans-serif; height:100vh; display:flex; flex-direction:column; overflow:hidden; }}
  #header {{ background:#1a1a2e; color:white; padding:8px 20px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; }}
  #header h1 {{ font-size:13px; font-weight:bold; }}
  #api-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:10px; flex-shrink:0; border-bottom:1px solid #2a3a50; }}
  #api-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  #api-bar input {{ flex:1; background:#1e2d45; border:1px solid #2a3a50; color:white; padding:4px 8px; font-size:11px; border-radius:3px; }}
  #api-bar button {{ background:#3b82f6; color:white; border:none; padding:5px 14px; font-size:11px; border-radius:3px; cursor:pointer; }}
  #api-bar button:hover {{ background:#2563eb; }}
  #control-bar {{ background:#0d1525; padding:5px 20px; display:flex; align-items:center; gap:8px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #control-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .mode-btn {{ padding:4px 10px; font-size:11px; border-radius:3px; cursor:pointer; border:1px solid #2a3a50; background:#1e2d45; color:#94a3b8; }}
  .mode-btn.active {{ background:#3b82f6; color:white; border-color:#3b82f6; }}
  #itl3-select {{ background:#1e2d45; border:1px solid #2a3a50; color:#e2e8f0; padding:4px 8px; font-size:11px; border-radius:3px; max-width:220px; }}
  #toggle-bar {{ background:#080d1a; padding:5px 20px; display:flex; align-items:center; gap:16px; flex-shrink:0; border-bottom:1px solid #2a3a50; flex-wrap:wrap; }}
  #toggle-bar label {{ font-size:11px; color:#888; white-space:nowrap; }}
  .airport-group {{ display:flex; align-items:center; gap:5px; }}
  .airport-group-label {{ font-size:11px; font-weight:bold; margin-right:2px; }}
  .tog {{ padding:2px 8px; font-size:10px; border-radius:3px; cursor:pointer; border:1px solid; opacity:0.35; transition:opacity 0.15s; }}
  .tog.on {{ opacity:1; }}
  button.compare-btn {{ background:#7c3aed; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; margin-left:auto; }}
  button.compare-btn:hover {{ background:#6d28d9; }}
  button.aircraft-btn {{ background:#0891b2; color:white; border:none; padding:4px 12px; font-size:11px; border-radius:3px; cursor:pointer; }}
  button.aircraft-btn:hover {{ background:#0e7490; }}
  #compare-status {{ display:none; position:fixed; bottom:20px; left:50%; transform:translateX(-50%);
    background:#7c3aed; color:white; padding:8px 20px; border-radius:20px; font-size:12px;
    z-index:1000; box-shadow:0 2px 10px rgba(0,0,0,0.5); white-space:nowrap; }}
  #main {{ display:flex; flex:1; overflow:hidden; }}
  #map {{ flex:1; display:none; }}
  #map-placeholder {{ flex:1; background:#0a0e1a; display:flex; align-items:center; justify-content:center; color:#475569; font-size:13px; flex-direction:column; gap:8px; }}
  #sidebar {{ width:320px; background:#111827; color:#e2e8f0; display:flex; flex-direction:column; overflow:hidden; border-left:1px solid #1e2d45; flex-shrink:0; }}
  #sidebar-title {{ padding:10px 16px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; text-transform:uppercase; letter-spacing:0.08em; color:#94a3b8; }}
  #instruction {{ padding:8px 16px; border-bottom:1px solid #1e2d45; font-size:11px; color:#64748b; line-height:1.5; }}
  #results {{ flex:1; overflow-y:auto; padding:8px; }}
  .airport-block {{ margin-bottom:8px; border:1px solid #1e2d45; border-radius:6px; overflow:hidden; }}
  .airport-header {{ padding:7px 12px; display:flex; align-items:center; gap:8px; border-bottom:1px solid #1e2d45; font-size:11px; font-weight:bold; }}
  .dot {{ width:7px; height:7px; border-radius:50%; flex-shrink:0; }}
  .route-row {{ padding:6px 12px; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; font-size:11px; cursor:pointer; transition:background 0.1s; }}
  .route-row:last-child {{ border-bottom:none; }}
  .route-row:hover {{ background:rgba(255,255,255,0.04); }}
  .straight-row {{ padding:4px 12px; background:#0d1525; border-bottom:1px solid #1e2d45; font-size:10px; color:#64748b; display:flex; justify-content:space-between; }}
  .route-left {{ display:flex; align-items:center; gap:7px; }}
  .route-icon {{ width:18px; height:18px; border-radius:3px; display:flex; align-items:center; justify-content:center; font-size:10px; flex-shrink:0; }}
  .route-label {{ font-size:11px; line-height:1.3; }}
  .route-sub {{ font-size:9px; color:#64748b; }}
  .route-val {{ color:#94a3b8; font-size:11px; white-space:nowrap; }}
  .route-val.done {{ color:#e2e8f0; font-weight:bold; }}
  .spinner {{ width:10px; height:10px; border:2px solid #1e2d45; border-top-color:#64748b; border-radius:50%; animation:spin 0.7s linear infinite; flex-shrink:0; }}
  @keyframes spin {{ to {{ transform:rotate(360deg); }} }}
  #empty {{ padding:24px 16px; text-align:center; font-size:11px; color:#64748b; line-height:1.7; }}
  #coords {{ padding:4px 16px; border-top:1px solid #1e2d45; font-size:10px; color:#475569; }}
  #legend {{ padding:6px 16px; border-top:1px solid #1e2d45; display:flex; flex-wrap:wrap; gap:6px; }}
  .leg {{ display:flex; align-items:center; gap:3px; font-size:10px; color:#64748b; }}
  .leg-line {{ width:14px; height:2px; border-radius:1px; }}
  #loading-bar {{ padding:5px 16px; font-size:10px; color:#f59e0b; display:none; border-top:1px solid #1e2d45; }}
  #compare-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #compare-overlay.open {{ display:flex; }}
  #compare-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:90vw; max-width:1100px; max-height:85vh; display:flex; flex-direction:column; overflow:hidden; }}
  #compare-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #compare-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #compare-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #compare-controls {{ padding:10px 20px; border-bottom:1px solid #1e2d45; display:flex; flex-direction:column; gap:8px; }}
  #compare-controls span {{ font-size:11px; color:#64748b; }}
  .time-slots {{ display:flex; gap:12px; flex-wrap:wrap; align-items:center; }}
  .time-slots label {{ font-size:11px; color:#e2e8f0; display:flex; align-items:center; gap:4px; cursor:pointer; }}
  #compare-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  #compare-progress {{ padding:8px 20px; border-top:1px solid #1e2d45; font-size:10px; color:#f59e0b; display:none; }}
  #aircraft-overlay {{ display:none; position:fixed; inset:0; background:rgba(0,0,0,0.7); z-index:2000; align-items:center; justify-content:center; }}
  #aircraft-overlay.open {{ display:flex; }}
  #aircraft-panel {{ background:#111827; border:1px solid #1e2d45; border-radius:10px; width:700px; max-height:80vh; display:flex; flex-direction:column; overflow:hidden; }}
  #aircraft-header {{ padding:12px 20px; background:#1a1a2e; display:flex; align-items:center; justify-content:space-between; border-bottom:1px solid #1e2d45; }}
  #aircraft-header h2 {{ font-size:13px; font-weight:bold; color:#e2e8f0; }}
  #aircraft-header button {{ background:none; border:none; color:#94a3b8; font-size:18px; cursor:pointer; }}
  #aircraft-search {{ padding:10px 20px; border-bottom:1px solid #1e2d45; display:flex; gap:8px; align-items:center; }}
  #aircraft-body {{ flex:1; overflow:auto; padding:16px 20px; }}
  #flight-search-result {{ padding:0 20px; }}
  .aircraft-airport {{ margin-bottom:16px; }}
  .aircraft-airport h3 {{ font-size:12px; font-weight:bold; margin-bottom:6px; padding-bottom:4px; border-bottom:1px solid #1e2d45; }}
  .aircraft-table {{ width:100%; border-collapse:collapse; font-size:11px; margin-bottom:8px; }}
  .aircraft-table th {{ background:#1e2d45; color:#94a3b8; padding:6px 8px; text-align:left; border:1px solid #2a3a50; }}
  .aircraft-table td {{ padding:5px 8px; border:1px solid #1e2d45; color:#e2e8f0; }}
  .aircraft-table tr:nth-child(even) td {{ background:#0d1525; }}
</style>
</head>
<body>

<div id="header">
  <h1>✈ London Airport Catchment — Interactive Journey Planner</h1>
  <span style="font-size:11px;color:#64748b;">University of Westminster · SAC Project</span>
</div>

<div id="api-bar">
  <label>GOOGLE MAPS KEY:</label>
  <input type="password" id="apiKey" placeholder="Paste your Google Maps API key…"/>
  <button onclick="loadMap()">Load Map</button>
</div>

<div id="control-bar">
  <label>Direction:</label>
  <button class="mode-btn active" id="btn-to"   onclick="setMode('to')">📍→✈ To Airport</button>
  <button class="mode-btn"        id="btn-from" onclick="setMode('from')">✈→📍 From Airport</button>
  <label style="margin-left:8px">ITL3:</label>
  <select id="itl3-select" onchange="selectITL3()">
    <option value="">— pick a region —</option>
  </select>
</div>

<div id="toggle-bar">
  <label>Catchment:</label>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#3b82f6">LHR</span>
    <button class="tog on" id="tog-LHR-30" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',30)">30</button>
    <button class="tog on" id="tog-LHR-45" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',45)">45</button>
    <button class="tog on" id="tog-LHR-60" style="color:#3b82f6;border-color:#3b82f6;background:#3b82f622" onclick="toggleIso('LHR',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#ef4444">LGW</span>
    <button class="tog on" id="tog-LGW-30" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',30)">30</button>
    <button class="tog on" id="tog-LGW-45" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',45)">45</button>
    <button class="tog on" id="tog-LGW-60" style="color:#ef4444;border-color:#ef4444;background:#ef444422" onclick="toggleIso('LGW',60)">60</button>
  </div>
  <div class="airport-group">
    <span class="airport-group-label" style="color:#22c55e">LTN</span>
    <button class="tog on" id="tog-LTN-30" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',30)">30</button>
    <button class="tog on" id="tog-LTN-45" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',45)">45</button>
    <button class="tog on" id="tog-LTN-60" style="color:#22c55e;border-color:#22c55e;background:#22c55e22" onclick="toggleIso('LTN',60)">60</button>
  </div>
  <button class="aircraft-btn" onclick="openAircraft()">✈ Flight Info</button>
  <button class="compare-btn"  onclick="openCompare()">📊 Compare</button>
</div>

<div id="compare-status">
  📍 Compare ON — <span id="compare-count">0</span>/3 selected
  · <button onclick="openCompare()" style="background:white;color:#7c3aed;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">View Table</button>
  · <button onclick="clearCompare()" style="background:rgba(255,255,255,0.2);color:white;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">Clear</button>
  · <button onclick="exitCompare()" style="background:#ef4444;color:white;border:none;padding:2px 8px;border-radius:10px;cursor:pointer;font-size:11px;">✕ Exit</button>
</div>

<div id="main">
  <div id="map-placeholder">
    <div style="font-size:32px">🗺</div>
    <div>Enter your Google Maps API key above and click Load Map</div>
  </div>
  <div id="map"></div>
  <div id="sidebar">
    <div id="sidebar-title">Journey Routes</div>
    <div id="instruction">Click anywhere or pick an ITL3 region.<br>Toggle catchment zones above.<br>Use Compare for time slot analysis.</div>
    <div id="results"><div id="empty">Load the map then click anywhere.</div></div>
    <div id="loading-bar">⏳ Calculating routes...</div>
    <div id="legend">
      <div class="leg"><div class="leg-line" style="background:#ff8800"></div>Drive</div>
      <div class="leg"><div class="leg-line" style="background:#00ccff"></div>Transit</div>
    </div>
    <div id="coords">Coordinates: —</div>
  </div>
</div>

<!-- Compare popup -->
<div id="compare-overlay">
  <div id="compare-panel">
    <div id="compare-header">
      <h2>📊 Journey Time Comparison</h2>
      <button onclick="closeCompare()">✕</button>
    </div>
    <div id="compare-controls">
      <span>Close panel and click up to 3 locations on the map. Select time slots to compare.</span>
      <div class="time-slots">
        <label><input type="checkbox" id="t-morning" checked> 🌅 Morning rush (08:15)</label>
        <label><input type="checkbox" id="t-noon" checked> ☀️ Midday (12:30)</label>
        <label><input type="checkbox" id="t-evening" checked> 🌆 Evening rush (17:00)</label>
        <label><input type="checkbox" id="t-night"> 🌙 Night (22:00)</label>
        <label><input type="checkbox" id="t-custom"> ⏰ Custom:
          <input type="time" id="t-custom-time" value="10:00"
            style="background:#1e2d45;border:1px solid #2a3a50;color:white;padding:2px 4px;font-size:11px;border-radius:3px;">
        </label>
        <button class="mode-btn" onclick="clearCompare()">Clear All</button>
      </div>
    </div>
    <div id="compare-progress"></div>
    <div id="compare-body">
      <div style="color:#64748b;font-size:11px;text-align:center;padding:40px">
        Close this panel and click locations on the map.
      </div>
    </div>
  </div>
</div>

<!-- Aircraft popup -->
<div id="aircraft-overlay">
  <div id="aircraft-panel">
    <div id="aircraft-header">
      <h2>✈ Flight Information</h2>
      <button onclick="closeAircraft()">✕</button>
    </div>
    <div id="aircraft-search">
      <input type="text" id="flight-search-input" placeholder="Enter flight number e.g. BA112, EZY456..."
        style="flex:1;background:#1e2d45;border:1px solid #2a3a50;color:white;padding:6px 10px;font-size:11px;border-radius:3px;"
        onkeydown="if(event.key==='Enter') searchFlight()"/>
      <button onclick="searchFlight()"
        style="background:#3b82f6;color:white;border:none;padding:6px 14px;font-size:11px;border-radius:3px;cursor:pointer;">
        🔍 Search
      </button>
    </div>
    <div id="flight-search-result"></div>
    <div id="aircraft-body"></div>
  </div>
</div>

<script>
const AIRPORTS = {{
  LHR: {{ name:'London Heathrow', lat:51.4706, lng:-0.4541, colour:'#3b82f6' }},
  LGW: {{ name:'London Gatwick',  lat:51.1564, lng:-0.1622, colour:'#ef4444' }},
  LTN: {{ name:'London Luton',    lat:51.8796, lng:-0.3713, colour:'#22c55e' }},
}};

const ROUTE_COLOURS = {{ DRIVING:'#ff8800', TRANSIT:'#00ccff' }};
const TOMTOM_KEY    = '{TOMTOM_KEY}';
const RAPIDAPI_KEY  = '{RAPIDAPI_KEY}';
const ISO_DATA      = {json.dumps(iso_geojson)};
const ITL3_DATA     = {json.dumps(itl3_json)};
const CENTROIDS     = {centroids_json};
const TIMES         = [30, 45, 60];
const ISO_WEIGHTS   = [3, 2, 1.5];

let map, directionsService;
let startMarker=null, renderers=[], straightLines=[];
let isoPolygons={{}};
let mapLoaded=false, currentMode='to';
let compareMode=false, compareMarkers=[], compareResults=[];
const PIN_COLOURS=['#f59e0b','#a855f7','#ec4899'];

window.addEventListener('load', () => {{
  const sel = document.getElementById('itl3-select');
  CENTROIDS.sort((a,b)=>a.name.localeCompare(b.name)).forEach(c => {{
    const opt = document.createElement('option');
    opt.value = JSON.stringify({{lat:c.lat, lng:c.lng}});
    opt.textContent = c.station ? c.name + ' — ' + c.station : c.name;
    sel.appendChild(opt);
  }});
}});

function selectITL3() {{
  const val = document.getElementById('itl3-select').value;
  if (!val || !map) return;
  const {{lat,lng}} = JSON.parse(val);
  handlePoint(lat,lng);
}}

function setMode(mode) {{
  currentMode=mode;
  document.getElementById('btn-to').className   = 'mode-btn'+(mode==='to'?' active':'');
  document.getElementById('btn-from').className = 'mode-btn'+(mode==='from'?' active':'');
  clearRoutes();
  if (startMarker) {{ startMarker.setMap(null); startMarker=null; }}
  document.getElementById('results').innerHTML  = '<div id="empty">Click anywhere to begin.</div>';
  document.getElementById('coords').textContent = 'Coordinates: —';
}}

function toggleIso(iata, mins) {{
  const btn   = document.getElementById(`tog-${{iata}}-${{mins}}`);
  const polys = isoPolygons[iata]?.[mins] || [];
  const isOn  = btn.classList.contains('on');
  polys.forEach(p=>p.setMap(isOn?null:map));
  btn.classList.toggle('on',!isOn);
  btn.style.opacity = isOn?'0.35':'1';
}}

function haversine(lat1,lng1,lat2,lng2) {{
  const R=6371,dLat=(lat2-lat1)*Math.PI/180,dLng=(lng2-lng1)*Math.PI/180;
  const a=Math.sin(dLat/2)**2+Math.cos(lat1*Math.PI/180)*Math.cos(lat2*Math.PI/180)*Math.sin(dLng/2)**2;
  return (R*2*Math.atan2(Math.sqrt(a),Math.sqrt(1-a))).toFixed(1);
}}

function loadMap() {{
  const key = document.getElementById('apiKey').value.trim();
  if (!key) {{ alert('Please enter your API key first.'); return; }}
  if (mapLoaded) {{ alert('Map already loaded!'); return; }}
  document.getElementById('map-placeholder').style.display='none';
  document.getElementById('map').style.display='block';
  const script=document.createElement('script');
  script.src=`https://maps.googleapis.com/maps/api/js?key=${{key}}&callback=initMap`;
  script.async=true; script.defer=true;
  document.head.appendChild(script);
  mapLoaded=true;
}}

function coordsToPath(c) {{ return c.map(x=>({{lat:x[1],lng:x[0]}})); }}

function initMap() {{
  map = new google.maps.Map(document.getElementById('map'), {{
    center:{{lat:51.5,lng:-0.3}}, zoom:8, mapTypeId:'roadmap',
    styles:[
      {{elementType:'geometry',stylers:[{{color:'#1a1a2e'}}]}},
      {{elementType:'labels.text.fill',stylers:[{{color:'#94a3b8'}}]}},
      {{elementType:'labels.text.stroke',stylers:[{{color:'#1a1a2e'}}]}},
      {{featureType:'road',elementType:'geometry',stylers:[{{color:'#2d3748'}}]}},
      {{featureType:'road.highway',elementType:'geometry',stylers:[{{color:'#2d4a7a'}}]}},
      {{featureType:'water',elementType:'geometry',stylers:[{{color:'#0d1525'}}]}},
      {{featureType:'transit.line',elementType:'geometry',stylers:[{{color:'#4a5568'}}]}},
      {{featureType:'transit.station',elementType:'geometry',stylers:[{{color:'#374151'}}]}},
      {{featureType:'poi',stylers:[{{visibility:'off'}}]}},
    ]
  }});
  directionsService = new google.maps.DirectionsService();

  // ITL3 choropleth
  const pops=ITL3_DATA.features.map(f=>f.properties.population_2024||0);
  const minPop=Math.min(...pops), maxPop=Math.max(...pops);
  function popColour(pop) {{
    if (!pop) return '#374151';
    const t=(pop-minPop)/(maxPop-minPop);
    return `rgb(${{Math.round(255*Math.min(1,t*2))}},${{Math.round(255*Math.max(0,1-t*2))}},50)`;
  }}
  ITL3_DATA.features.forEach(f => {{
    if (!f.geometry?.coordinates) return;
    const colour=popColour(f.properties.population_2024);
    const rings=f.geometry.type==='MultiPolygon'
      ? f.geometry.coordinates.map(p=>p[0])
      : [f.geometry.coordinates[0]];
    rings.forEach(ring => {{
      new google.maps.Polygon({{
        paths:coordsToPath(ring), strokeColor:'#374151', strokeWeight:0.5,
        fillColor:colour, fillOpacity:0.5, clickable:false, map
      }});
    }});
  }});

  // ITL3 centroid markers (station-based)
  CENTROIDS.forEach(c => {{
    const marker=new google.maps.Marker({{
      position:{{lat:c.lat,lng:c.lng}}, map, title:c.name,
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:5,fillColor:'#fff',fillOpacity:0.6,strokeColor:'#94a3b8',strokeWeight:1}}
    }});
    const iw=new google.maps.InfoWindow({{
      content:`<div style="font-size:11px"><b>${{c.name}}</b><br>🚉 ${{c.station || ''}}<br>Pop: ${{c.pop?.toLocaleString()}}<br>GDHI: £${{c.gdhi?.toLocaleString()}}M</div>`
    }});
    marker.addListener('click',()=>{{ iw.open(map,marker); handlePoint(c.lat,c.lng); }});
  }});

  // Isochrones
  const isoColours={{LHR:'#3b82f6',LGW:'#ef4444',LTN:'#22c55e'}};
  Object.entries(ISO_DATA).forEach(([iata,geojson]) => {{
    isoPolygons[iata]={{}};
    geojson.features.forEach((feature,i) => {{
      const mins=TIMES[i];
      const poly=new google.maps.Polygon({{
        paths:coordsToPath(feature.geometry.coordinates[0]),
        strokeColor:isoColours[iata], strokeWeight:ISO_WEIGHTS[i],
        strokeOpacity:0.9, fillOpacity:0, clickable:false, map
      }});
      if (!isoPolygons[iata][mins]) isoPolygons[iata][mins]=[];
      isoPolygons[iata][mins].push(poly);
    }});
  }});

  // Airport markers
  Object.entries(AIRPORTS).forEach(([code,ap]) => {{
    const marker=new google.maps.Marker({{
      position:{{lat:ap.lat,lng:ap.lng}}, map, title:ap.name,
      label:{{text:'✈',color:'white',fontSize:'11px'}},
      icon:{{path:google.maps.SymbolPath.CIRCLE,scale:13,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}}
    }});
    const iw=new google.maps.InfoWindow({{content:`<b>✈ ${{ap.name}}</b><br>${{code}}`}});
    marker.addListener('click',()=>iw.open(map,marker));
  }});

  map.addListener('click', e => {{
    const lat=e.latLng.lat(), lng=e.latLng.lng();
    if (compareMode) {{ addComparePoint(lat,lng); }}
    else {{ handlePoint(lat,lng); }}
  }});

  document.getElementById('empty').textContent='Click anywhere or pick an ITL3 region.';
}}

function clearRoutes() {{
  renderers.forEach(r=>r.setMap(null)); renderers=[];
  straightLines.forEach(l=>l.setMap(null)); straightLines=[];
}}

function handlePoint(lat,lng) {{
  document.getElementById('coords').textContent=`${{currentMode==='to'?'From':'To'}}: ${{lat.toFixed(5)}}, ${{lng.toFixed(5)}}`;
  if (startMarker) startMarker.setMap(null);
  startMarker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:9,
      fillColor:currentMode==='to'?'#ff8800':'#ff00ff',
      fillOpacity:1,strokeColor:'white',strokeWeight:2}}
  }});
  clearRoutes();
  calculateAllRoutes(lat,lng);
}}

function calculateAllRoutes(lat,lng) {{
  const lb=document.getElementById('loading-bar');
  lb.style.display='block'; lb.style.color='#f59e0b'; lb.textContent='⏳ Calculating routes...';
  const straight={{}};
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{ straight[code]=haversine(lat,lng,ap.lat,ap.lng); }});
  const modeLabel=currentMode==='to'?'→ ✈':'✈ →';
  document.getElementById('results').innerHTML=Object.entries(AIRPORTS).map(([code,ap])=>`
    <div class="airport-block">
      <div class="airport-header"><div class="dot" style="background:${{ap.colour}}"></div>${{modeLabel}} ${{ap.name}} (${{code}})</div>
      <div class="straight-row"><span>📏 Straight line</span><span style="color:#e2e8f0">${{straight[code]}} km</span></div>
      <div class="route-row" id="r-${{code}}-DRIVING">
        <div class="route-left"><div class="route-icon" style="background:#ff880033">🚗</div><div class="route-label">Drive</div></div>
        <div class="spinner"></div>
      </div>
      <div class="route-row" id="r-${{code}}-TRANSIT">
        <div class="route-left"><div class="route-icon" style="background:#00ccff33">🚆</div><div class="route-label">Transit<div class="route-sub">Train · Tube · Bus</div></div></div>
        <div class="spinner"></div>
      </div>
    </div>`).join('');

  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    const line=new google.maps.Polyline({{
      path:[{{lat,lng}},{{lat:ap.lat,lng:ap.lng}}],
      strokeColor:ap.colour,strokeWeight:1,strokeOpacity:0.5,
      icons:[{{icon:{{path:'M 0,-1 0,1',strokeOpacity:1,scale:3}},offset:'0',repeat:'12px'}}],map
    }});
    straightLines.push(line);
  }});

  let completed=0;
  const total=Object.keys(AIRPORTS).length*2;
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    ['DRIVING','TRANSIT'].forEach(mode=>{{
      const renderer=new google.maps.DirectionsRenderer({{
        suppressMarkers:true,
        polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeWeight:mode==='DRIVING'?5:4,strokeOpacity:0.85}}
      }});
      renderers.push(renderer);
      const origin     =currentMode==='to'?{{lat,lng}}:{{lat:ap.lat,lng:ap.lng}};
      const destination=currentMode==='to'?{{lat:ap.lat,lng:ap.lng}}:{{lat,lng}};
      directionsService.route({{
        origin,destination,travelMode:google.maps.TravelMode[mode],
        ...(mode==='DRIVING'?{{drivingOptions:{{departureTime:new Date(),trafficModel:google.maps.TrafficModel.BEST_GUESS}}}}:{{}}),
        ...(mode==='TRANSIT'?{{transitOptions:{{
          departureTime:new Date(),
          modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
          routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
        }}}}:{{}})
      }},(result,status)=>{{
        completed++;
        const el=document.getElementById(`r-${{code}}-${{mode}}`);
        if (status==='OK') {{
          renderer.setMap(map); renderer.setDirections(result);
          const leg=result.routes[0].legs[0];
          const mins=Math.round((leg.duration_in_traffic?.value||leg.duration.value)/60);
          const km=(leg.distance.value/1000).toFixed(1);
          if (el) {{
            el.querySelector('.spinner').outerHTML=`<div class="route-val done">${{mins}} min · ${{km}} km</div>`;
            if (mode==='TRANSIT') {{
              const steps=result.routes[0].legs[0].steps
                .filter(s=>s.travel_mode==='TRANSIT')
                .map(s=>s.transit?.line?.short_name||s.transit?.line?.name||'')
                .filter(Boolean).join(' → ');
              if (steps) {{ const sub=el.querySelector('.route-sub'); if(sub) sub.textContent=steps; }}
            }}
            el.onclick=()=>{{
              renderers.forEach(r=>r.setOptions({{polylineOptions:{{strokeOpacity:0.15,strokeWeight:2}}}}));
              renderer.setOptions({{polylineOptions:{{strokeColor:ROUTE_COLOURS[mode],strokeOpacity:1,strokeWeight:6}}}});
              map.fitBounds(result.routes[0].bounds);
            }};
          }}
        }} else {{
          if (el) el.querySelector('.spinner').outerHTML=`<div class="route-val" style="color:#64748b">not available</div>`;
        }}
        if (completed===total) {{
          lb.textContent='✓ Routes loaded — click a row to highlight'; lb.style.color='#22c55e';
        }}
      }});
    }});
  }});
}}

// ── COMPARE ───────────────────────────────────────────────
function openCompare() {{
  compareMode=true;
  document.getElementById('compare-overlay').classList.add('open');
  document.getElementById('compare-status').style.display='block';
}}

function closeCompare() {{
  document.getElementById('compare-overlay').classList.remove('open');
}}

function exitCompare() {{
  compareMode=false;
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-status').style.display='none';
  document.getElementById('compare-count').textContent='0';
  document.getElementById('compare-overlay').classList.remove('open');
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Close panel and click locations on the map.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function clearCompare() {{
  compareMarkers.forEach(m=>m.setMap(null)); compareMarkers=[];
  compareResults=[];
  document.getElementById('compare-count').textContent='0';
  document.getElementById('compare-body').innerHTML=
    '<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">Cleared. Click locations on the map.</div>';
  document.getElementById('compare-progress').style.display='none';
}}

function addComparePoint(lat,lng) {{
  if (compareResults.length>=3) {{ alert('Maximum 3 locations. Click Clear All to reset.'); return; }}
  const idx=compareResults.length;
  const colour=PIN_COLOURS[idx];
  const name=`Loc ${{idx+1}} (${{lat.toFixed(2)}},${{lng.toFixed(2)}})`;
  const marker=new google.maps.Marker({{
    position:{{lat,lng}}, map,
    icon:{{path:google.maps.SymbolPath.CIRCLE,scale:10,fillColor:colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},
    label:{{text:`${{idx+1}}`,color:'white',fontSize:'10px'}}
  }});
  compareMarkers.push(marker);
  const entry={{lat,lng,name,colour,slots:[],data:{{}}}};
  compareResults.push(entry);
  document.getElementById('compare-count').textContent=compareResults.length;
  openCompare();
  fetchCompareData(entry,idx);
}}

function getSelectedSlots() {{
  const slots=[];
  if (document.getElementById('t-morning').checked) slots.push({{label:'🌅 08:15',hour:8,min:15}});
  if (document.getElementById('t-noon').checked)    slots.push({{label:'☀️ 12:30',hour:12,min:30}});
  if (document.getElementById('t-evening').checked) slots.push({{label:'🌆 17:00',hour:17,min:0}});
  if (document.getElementById('t-night').checked)   slots.push({{label:'🌙 22:00',hour:22,min:0}});
  if (document.getElementById('t-custom').checked) {{
    const val=document.getElementById('t-custom-time').value;
    const [h,m]=val.split(':').map(Number);
    slots.push({{label:`⏰ ${{val}}`,hour:h,min:m}});
  }}
  return slots;
}}

async function fetchCompareData(entry,idx) {{
  const prog=document.getElementById('compare-progress');
  prog.style.display='block';
  const slots=getSelectedSlots();
  if (slots.length===0) {{ prog.textContent='⚠ Select at least one time slot.'; return; }}
  entry.slots=slots;
  const nextMonday=new Date();
  const day=nextMonday.getDay();
  nextMonday.setDate(nextMonday.getDate()+(day<=1?1-day:8-day));

  for (const slot of slots) {{
    entry.data[slot.label]={{}};
    prog.textContent=`⏳ ${{entry.name}} — ${{slot.label}}...`;
    const depStr=`${{nextMonday.toISOString().split('T')[0]}}T${{String(slot.hour).padStart(2,'0')}}:${{String(slot.min).padStart(2,'0')}}:00`;
    const depTime=new Date(nextMonday);
    depTime.setHours(slot.hour,slot.min,0,0);

    for (const [code,ap] of Object.entries(AIRPORTS)) {{
      entry.data[slot.label][code]={{}};
      try {{
        const url=`https://api.tomtom.com/routing/1/calculateRoute/${{entry.lat}},${{entry.lng}}:${{ap.lat}},${{ap.lng}}/json?key=${{TOMTOM_KEY}}&traffic=true&departAt=${{depStr}}&travelMode=car&routeType=fastest`;
        const res=await fetch(url);
        const data=await res.json();
        const secs=data.routes?.[0]?.summary?.travelTimeInSeconds;
        entry.data[slot.label][code]['DRIVING']=secs?Math.round(secs/60):null;
      }} catch(e) {{ entry.data[slot.label][code]['DRIVING']=null; }}
      await new Promise(r=>setTimeout(r,100));
      await new Promise(resolve=>{{
        directionsService.route({{
          origin:{{lat:entry.lat,lng:entry.lng}},
          destination:{{lat:ap.lat,lng:ap.lng}},
          travelMode:google.maps.TravelMode.TRANSIT,
          transitOptions:{{
            departureTime:depTime,
            modes:[google.maps.TransitMode.RAIL,google.maps.TransitMode.SUBWAY,google.maps.TransitMode.BUS],
            routingPreference:google.maps.TransitRoutePreference.FEWER_TRANSFERS
          }}
        }},(result,status)=>{{
          entry.data[slot.label][code]['TRANSIT']=status==='OK'
            ?Math.round(result.routes[0].legs[0].duration.value/60):null;
          resolve();
        }});
      }});
      await new Promise(r=>setTimeout(r,150));
    }}
    updateCompareTable();
  }}
  prog.textContent=`✓ ${{entry.name}} complete!`;
  updateCompareTable();
}}

function updateCompareTable() {{
  if (compareResults.length===0) return;
  const allSlots=[];
  compareResults.forEach(entry=>{{
    (entry.slots||[]).forEach(s=>{{
      if (!allSlots.find(x=>x.label===s.label)) allSlots.push(s);
    }});
  }});
  if (allSlots.length===0) return;

  let html='';
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    html+=`<div style="margin-bottom:28px;">`;
    html+=`<h3 style="color:${{ap.colour}};font-size:12px;margin-bottom:8px;padding:6px 10px;background:#0d1525;border-radius:6px;border-left:3px solid ${{ap.colour}};">✈ ${{ap.name}} (${{code}})</h3>`;
    html+='<div style="overflow-x:auto;"><table style="width:100%;border-collapse:collapse;font-size:11px;">';
    html+='<tr><th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:left;border:1px solid #2a3a50;white-space:nowrap;">Time Slot</th>';
    compareResults.forEach(entry=>{{
      html+=`<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:center;border:1px solid #2a3a50;white-space:nowrap;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px;"></span>
        ${{entry.name}}<br><span style="font-size:9px;color:#64748b;">🚗 Drive</span></th>`;
      html+=`<th style="background:#1e2d45;color:#94a3b8;padding:8px 10px;text-align:center;border:1px solid #2a3a50;white-space:nowrap;">
        <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${{entry.colour}};margin-right:4px;"></span>
        ${{entry.name}}<br><span style="font-size:9px;color:#64748b;">🚆 Transit</span></th>`;
    }});
    html+='</tr>';
    allSlots.forEach((slot,si)=>{{
      const rowBg=si%2===0?'background:#111827':'background:#0d1525';
      html+=`<tr><td style="${{rowBg}};font-weight:bold;color:#94a3b8;padding:8px 10px;border:1px solid #1e2d45;white-space:nowrap;">${{slot.label}}</td>`;
      compareResults.forEach(entry=>{{
        const d=entry.data[slot.label]?.[code];
        const drive=d?.DRIVING!=null?`<span style="color:#ff8800;font-weight:bold">${{d.DRIVING}} min</span>`:'<span style="color:#475569">...</span>';
        const transit=d?.TRANSIT!=null?`<span style="color:#00ccff;font-weight:bold">${{d.TRANSIT}} min</span>`:'<span style="color:#475569">...</span>';
        html+=`<td style="${{rowBg}};padding:8px 10px;border:1px solid #1e2d45;text-align:center;">${{drive}}</td>`;
        html+=`<td style="${{rowBg}};padding:8px 10px;border:1px solid #1e2d45;text-align:center;">${{transit}}</td>`;
      }});
      html+='</tr>';
    }});
    html+='</table></div></div>';
  }});
  document.getElementById('compare-body').innerHTML=html;
}}

// ── FLIGHT INFO ───────────────────────────────────────────
async function openAircraft() {{
  document.getElementById('aircraft-overlay').classList.add('open');
  const body=document.getElementById('aircraft-body');
  body.innerHTML='<div style="color:#64748b;font-size:11px;text-align:center;padding:40px">⏳ Loading live flight data...</div>';
  const codes=Object.keys(AIRPORTS);
  const results={{}};
  const hdrs={{'x-rapidapi-key':RAPIDAPI_KEY,'x-rapidapi-host':'aerodatabox.p.rapidapi.com'}};
  const delay=ms=>new Promise(r=>setTimeout(r,ms));

  for (const code of codes) {{
    results[code]={{arrivals:[],departures:[]}};
    try {{
      const now=new Date(), from=new Date(now.getTime()-60*60*1000), to=new Date(now.getTime()+3*60*60*1000);
      const fmt=d=>d.toISOString().slice(0,16);
      const arrUrl=`https://aerodatabox.p.rapidapi.com/flights/airports/iata/${{code}}/${{fmt(from)}}/${{fmt(to)}}?withLeg=true&direction=Arrival&withCancelled=false&withCodeshared=false&withLocation=false`;
      const arrRes=await fetch(arrUrl,{{headers:hdrs}});
      const arrData=await arrRes.json();
      results[code].arrivals=(arrData.arrivals||[]).slice(0,8);
    }} catch(e) {{ results[code].arrivals=[]; }}
    await delay(1500);
    try {{
      const now=new Date(), from=new Date(now.getTime()-60*60*1000), to=new Date(now.getTime()+3*60*60*1000);
      const fmt=d=>d.toISOString().slice(0,16);
      const depUrl=`https://aerodatabox.p.rapidapi.com/flights/airports/iata/${{code}}/${{fmt(from)}}/${{fmt(to)}}?withLeg=true&direction=Departure&withCancelled=false&withCodeshared=false&withLocation=false`;
      const depRes=await fetch(depUrl,{{headers:hdrs}});
      const depData=await depRes.json();
      results[code].departures=(depData.departures||[]).slice(0,8);
    }} catch(e) {{ results[code].departures=[]; }}
    await delay(1500);
    renderFIDS(results);
  }}
}}

function renderFIDS(results) {{
  const body=document.getElementById('aircraft-body');
  const now=new Date().toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}});
  let html=`<p style="color:#64748b;font-size:10px;margin-bottom:12px">🔴 Live data — updated at ${{now}}</p>`;
  Object.entries(AIRPORTS).forEach(([code,ap])=>{{
    const arr=results[code]?.arrivals||[];
    const dep=results[code]?.departures||[];
    html+=`<div class="aircraft-airport"><h3 style="color:${{ap.colour}}">✈ ${{ap.name}} (${{code}})</h3>`;
    html+='<b style="font-size:11px;color:#94a3b8">🛬 Arrivals</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>From</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';
    if (arr.length===0) {{
      html+='<tr><td colspan="6" style="color:#64748b;text-align:center">No arrivals found</td></tr>';
    }} else {{
      arr.forEach(f=>{{
        const flight=f.number||'—';
        const from=f.departure?.airport?.iata||'—';
        const sched=f.arrival?.scheduledTime?.local?new Date(f.arrival.scheduledTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const expected=f.arrival?.revisedTime?.local?new Date(f.arrival.revisedTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):f.arrival?.runwayTime?.local?new Date(f.arrival.runwayTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const terminal=f.arrival?.terminal||'—';
        const status=f.arrival?.runwayTime?'Landed':f.arrival?.revisedTime?'On approach':'Scheduled';
        const sc=status==='Landed'?'#94a3b8':status==='On approach'?'#22c55e':'#f59e0b';
        html+=`<tr><td style="font-weight:bold">${{flight}}</td><td>${{from}}</td><td>${{sched}}</td><td>${{expected}}</td><td>${{terminal}}</td><td style="color:${{sc}}">${{status}}</td></tr>`;
      }});
    }}
    html+='</table>';
    html+='<b style="font-size:11px;color:#94a3b8;display:block;margin-top:10px">🛫 Departures</b>';
    html+='<table class="aircraft-table"><tr><th>Flight</th><th>To</th><th>Scheduled</th><th>Expected</th><th>Terminal</th><th>Status</th></tr>';
    if (dep.length===0) {{
      html+='<tr><td colspan="6" style="color:#64748b;text-align:center">No departures found</td></tr>';
    }} else {{
      dep.forEach(f=>{{
        const flight=f.number||'—';
        const to=f.arrival?.airport?.iata||'—';
        const sched=f.departure?.scheduledTime?.local?new Date(f.departure.scheduledTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const expected=f.departure?.revisedTime?.local?new Date(f.departure.revisedTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):f.departure?.runwayTime?.local?new Date(f.departure.runwayTime.local).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const terminal=f.departure?.terminal||'—';
        const status=f.departure?.runwayTime?'Departed':f.departure?.revisedTime?'Boarding':'Scheduled';
        const sc=status==='Departed'?'#94a3b8':status==='Boarding'?'#22c55e':'#f59e0b';
        html+=`<tr><td style="font-weight:bold">${{flight}}</td><td>${{to}}</td><td>${{sched}}</td><td>${{expected}}</td><td>${{terminal}}</td><td style="color:${{sc}}">${{status}}</td></tr>`;
      }});
    }}
    html+='</table></div>';
  }});
  body.innerHTML=html;
}}

function closeAircraft() {{
  document.getElementById('aircraft-overlay').classList.remove('open');
}}

function searchFlight() {{
  const input=document.getElementById('flight-search-input').value.trim().toUpperCase().replace(/\\s/g,'');
  const result=document.getElementById('flight-search-result');
  if (!input) {{ result.innerHTML=''; return; }}
  result.innerHTML='<p style="color:#f59e0b;font-size:11px;padding:10px 0">⏳ Searching...</p>';
  const url=`https://aerodatabox.p.rapidapi.com/flights/number/${{input}}`;
  const headers={{'x-rapidapi-key':RAPIDAPI_KEY,'x-rapidapi-host':'aerodatabox.p.rapidapi.com'}};
  fetch(url,{{headers}})
    .then(r=>r.json())
    .then(data=>{{
      const flights=Array.isArray(data)?data:[data];
      if (!flights.length||flights[0].message) {{
        result.innerHTML=`<p style="color:#ef4444;font-size:11px;padding:10px 0">No flights found for "${{input}}".</p>`;
        return;
      }}
      let html='<div style="margin:10px 0;">';
      flights.slice(0,3).forEach(f=>{{
        const dep=f.departure||{{}}, arr=f.arrival||{{}};
        const depIata=dep.airport?.iata||'—', arrIata=arr.airport?.iata||'—';
        const depName=dep.airport?.name||'—', arrName=arr.airport?.name||'—';
        const depTime=dep.scheduledTime?.local?new Date(dep.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const arrTime=arr.scheduledTime?.local?new Date(arr.scheduledTime.local.replace('Z','')).toLocaleTimeString('en-GB',{{hour:'2-digit',minute:'2-digit'}}):'—';
        const status=f.status||'scheduled';
        const sc=status==='Active'?'#22c55e':status==='Landed'?'#94a3b8':'#f59e0b';
        const airline=f.airline?.name||'—', aircraft=f.aircraft?.model||f.aircraft?.reg||'—';
        const depDelay=dep.delay||0, arrDelay=arr.delay||0;
        const distKm=f.greatCircleDistance?.km?Math.round(f.greatCircleDistance.km)+' km':'—';
        html+=`<div style="padding:12px;background:#0d1525;border-radius:8px;border:1px solid #1e2d45;margin-bottom:8px;">
          <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <span style="font-size:14px;font-weight:bold;color:#e2e8f0">${{input}}</span>
            <span style="color:${{sc}};font-size:11px;font-weight:bold;background:${{sc}}22;padding:2px 8px;border-radius:10px;">${{status}}</span>
          </div>
          <div style="font-size:11px;color:#64748b;margin-bottom:10px;">${{airline}} · ${{aircraft}} · ${{distKm}}</div>
          <div style="display:grid;grid-template-columns:1fr auto 1fr;gap:8px;align-items:center;">
            <div>
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${{depIata}}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${{depName}}</div>
              <div style="font-size:13px;color:#e2e8f0">${{depTime}}</div>
              ${{depDelay>0?`<div style="font-size:10px;color:#ef4444;margin-top:2px">+${{depDelay}} min delay</div>`:'<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>'}}
            </div>
            <div style="text-align:center;"><div style="color:#64748b;font-size:20px;">✈</div><div style="font-size:9px;color:#475569;margin-top:2px">${{distKm}}</div></div>
            <div style="text-align:right;">
              <div style="font-size:20px;font-weight:bold;color:#e2e8f0">${{arrIata}}</div>
              <div style="font-size:10px;color:#64748b;margin-bottom:4px">${{arrName}}</div>
              <div style="font-size:13px;color:#e2e8f0">${{arrTime}}</div>
              ${{arrDelay>0?`<div style="font-size:10px;color:#ef4444;margin-top:2px">+${{arrDelay}} min delay</div>`:'<div style="font-size:10px;color:#22c55e;margin-top:2px">On time</div>'}}
            </div>
          </div>
        </div>`;
      }});
      html+='</div>';
      result.innerHTML=html;

      // Animate plane
      if (window.flightPathLayers) window.flightPathLayers.forEach(l=>l.setMap(null));
      if (window.planeAnimInterval) clearInterval(window.planeAnimInterval);
      window.flightPathLayers=[];
      const f0=flights[0];
      const depLat=f0.departure?.airport?.location?.lat, depLon=f0.departure?.airport?.location?.lon;
      const arrLat=f0.arrival?.airport?.location?.lat,   arrLon=f0.arrival?.airport?.location?.lon;
      const depIata2=f0.departure?.airport?.iata||'', arrIata2=f0.arrival?.airport?.iata||'';
      const depTimeStr=f0.departure?.scheduledTime?.utc, arrTimeStr=f0.arrival?.scheduledTime?.utc;

      if (depLat&&arrLat&&map) {{
        [{{lat:depLat,lng:depLon,iata:depIata2,colour:'#f59e0b'}},{{lat:arrLat,lng:arrLon,iata:arrIata2,colour:'#22c55e'}}]
        .forEach(ap=>{{
          const m=new google.maps.Marker({{
            position:{{lat:ap.lat,lng:ap.lng}},map,
            icon:{{path:google.maps.SymbolPath.CIRCLE,scale:12,fillColor:ap.colour,fillOpacity:1,strokeColor:'white',strokeWeight:2}},
            label:{{text:ap.iata,color:'white',fontSize:'9px',fontWeight:'bold'}}
          }});
          window.flightPathLayers.push(m);
        }});

        function toRad(d){{return d*Math.PI/180;}}
        function toDeg(r){{return r*180/Math.PI;}}
        function gcPoints(la1,lo1,la2,lo2,n){{
          const pts=[];
          for(let i=0;i<=n;i++){{
            const f=i/n;
            const d=Math.acos(Math.min(1,Math.sin(toRad(la1))*Math.sin(toRad(la2))+Math.cos(toRad(la1))*Math.cos(toRad(la2))*Math.cos(toRad(lo2-lo1))));
            const A=d===0?1-f:Math.sin((1-f)*d)/Math.sin(d);
            const B=d===0?f:Math.sin(f*d)/Math.sin(d);
            const x=A*Math.cos(toRad(la1))*Math.cos(toRad(lo1))+B*Math.cos(toRad(la2))*Math.cos(toRad(lo2));
            const y=A*Math.cos(toRad(la1))*Math.sin(toRad(lo1))+B*Math.cos(toRad(la2))*Math.sin(toRad(lo2));
            const z=A*Math.sin(toRad(la1))+B*Math.sin(toRad(la2));
            pts.push({{lat:toDeg(Math.atan2(z,Math.sqrt(x*x+y*y))),lng:toDeg(Math.atan2(y,x))}});
          }}
          return pts;
        }}

        const pts=gcPoints(depLat,depLon,arrLat,arrLon,100);
        const pathLine=new google.maps.Polyline({{path:pts,geodesic:false,strokeColor:'#f59e0b',strokeOpacity:0.6,strokeWeight:2,map}});
        window.flightPathLayers.push(pathLine);

        const now2=new Date();
        const dep2=depTimeStr?new Date(depTimeStr.replace('Z',' UTC')):null;
        const arr2=arrTimeStr?new Date(arrTimeStr.replace('Z',' UTC')):null;
        let prog=0,inFlight=false;
        if(dep2&&arr2){{
          const total=arr2-dep2,elapsed=now2-dep2;
          if(elapsed<0) prog=0;
          else if(elapsed>total) prog=1;
          else{{prog=elapsed/total;inFlight=true;}}
        }}

        function getPt(p,f){{const i=Math.min(Math.floor(f*(p.length-1)),p.length-2);const r=f*(p.length-1)-i;return{{lat:p[i].lat+r*(p[i+1].lat-p[i].lat),lng:p[i].lng+r*(p[i+1].lng-p[i].lng)}};}}
        function bearing(p1,p2){{const dL=toRad(p2.lng-p1.lng);const y=Math.sin(dL)*Math.cos(toRad(p2.lat));const x=Math.cos(toRad(p1.lat))*Math.sin(toRad(p2.lat))-Math.sin(toRad(p1.lat))*Math.cos(toRad(p2.lat))*Math.cos(dL);return(toDeg(Math.atan2(y,x))+360)%360;}}

        const planeIcon={{path:'M 0,-3 L 1.5,1 L 0,0 L -1.5,1 Z',scale:7,fillColor:'#ffffff',fillOpacity:1,strokeColor:'#f59e0b',strokeWeight:1.5,rotation:0,anchor:new google.maps.Point(0,0)}};
        const plane=new google.maps.Marker({{position:getPt(pts,prog),map,icon:planeIcon,zIndex:999}});
        window.flightPathLayers.push(plane);

        let p=prog;
        const spd=inFlight?0.0001:0.0005;
        window.planeAnimInterval=setInterval(()=>{{
          p+=spd;if(p>=1)p=0;
          const pos=getPt(pts,p),pos2=getPt(pts,Math.min(p+0.001,1));
          plane.setPosition(pos);
          plane.setIcon({{...planeIcon,rotation:bearing(pos,pos2)}});
        }},50);

        const bounds=new google.maps.LatLngBounds();
        pts.forEach(pt=>bounds.extend(pt));
        map.fitBounds(bounds,{{top:80,bottom:80,left:80,right:80}});

        const statusTxt=prog<=0?'🕐 Not yet departed':prog>=1?'🛬 Landed':`✈ In flight — ${{Math.round(prog*100)}}% complete`;
        result.innerHTML+=`<div style="margin-top:8px;font-size:10px;color:#94a3b8;display:flex;gap:10px;align-items:center;">
          <span>${{statusTxt}}</span>
          <button onclick="clearFlightPath()" style="background:#475569;color:white;border:none;padding:3px 10px;font-size:10px;border-radius:3px;cursor:pointer;">✕ Clear</button>
        </div>`;
      }}
    }})
    .catch(e=>{{
      result.innerHTML=`<p style="color:#ef4444;font-size:11px;padding:10px 0">Error: ${{e.message}}</p>`;
    }});
}}

function clearFlightPath() {{
  if (window.flightPathLayers){{window.flightPathLayers.forEach(l=>l.setMap(null));window.flightPathLayers=[];}}
  if (window.planeAnimInterval){{clearInterval(window.planeAnimInterval);window.planeAnimInterval=null;}}
  document.getElementById('flight-search-result').innerHTML='';
  document.getElementById('flight-search-input').value='';
}}
</script>
</body>
</html>"""

output_path = '/content/drive/MyDrive/airport_journey_planner_FINAL.html'
with open(output_path, 'w') as f:
    f.write(html_content)

print(f' Saved: airport_journey_planner_FINAL.html')
print(f' File size: {len(html_content)//1024} KB')
print(f' Station-based centroids: {"station" in html_content}')
print(f' ITL3 data: {"ITL3_DATA" in html_content}')
print(f' TomTom: {"TOMTOM_KEY" in html_content}')
print(f' AeroDataBox: {"aerodatabox" in html_content}')
print(f' Plane animation: {"planeAnimInterval" in html_content}')